# CS-781 Phase 6 (Abstractive Legal Summarization) -- E2_lora

LoRA r=16, alpha 32 on q/k/v/o + FFN of the Phase-1 causal LM.

**Accelerator: GPU T4 x2. Internet: ON.** Expected runtime: <= 125 min (time budget 2.0 h; every job plans its epochs / data fraction / dev decoding from measured throughput and stops cleanly at its share of the budget). Two models train at the same time, one per GPU (no DDP).

Run it with **Save Version -> Save & Run All (Commit)** so the output persists.

### Inputs to attach (Add Input)
1. **The four Phase-1 datasets** (`01-train-gpt2style`, `02-train-llamastyle`,
   `03-train-mistral-sliding`, `04-train-bloom-alibi`: `model.pt` + `config.json`). Models are
   recognised by the `arch` field of `config.json`, so dataset names do not matter.
2. **The `bpe_model` dataset** (`bpe_model.json`; found anywhere under /kaggle/input).
3. **Data:** Add-ons -> Secrets -> `HF` (tick it for THIS notebook) **and** Settings -> Internet
   **On** (Phase_6_Summ/Train/train.json + Test/test.json are downloaded; Internet is also
   needed for `pip install bert-score` + the roberta-large BERTScore model, used for dev scoring
   only). *Alternative data source:* a Kaggle dataset holding `Phase_6_Summ/Train/train.json`
   and `Phase_6_Summ/Test/test.json`.
4. *Optional:* the output of an earlier session of THIS notebook (finished models are skipped,
   an unfinished one resumes from `last.pt`).

### What a healthy run prints in the first minutes
* `all 24 modules verified`, `bert-score: ok`
* `[phase1] M1 gpt2style -> ...` (all four found), `[bpe] using ...`
* `[data] train: N records; keys id='id' document='document' summary='summary'`,
  `[data] held-out dev split: ...`, `[cache] ...`, the `[audit]` line and `[repr] ... -> dancer|headtail`
* per job: `[check] ... pass`, `[preflight] train ... tok/s, generation ... ms/step`,
  `[plan] CHOSEN: epochs=... train_fraction=...`, then `[eval] epoch 0.50: dev loss ...`
* at the end of a job: `DONE dev BERTScore-F1 0.xxxx, ROUGE-1 ..., ROUGE-L ... submission validated`


In [ ]:
import os, sys, json, hashlib, subprocess, glob, time, traceback
CODE = "/kaggle/working/phase6"; OUT = "/kaggle/working/phase6_out"
os.makedirs(CODE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(sys.version.split()[0], time.strftime('%Y-%m-%d %H:%M:%S'))
try:
    import torch
    print('torch', torch.__version__, '| GPUs:', torch.cuda.device_count(),
          [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as e:
    print('torch not importable:', e)
print('inputs:', sorted(os.listdir('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'none')
EXPECTED_MD5 = {
 "bpe_tokenizer.py": "906d8a37d55bf0081dd5e8d7944c0b4e",
 "phase1_models.py": "1ae9785176b01f67a305e1504b0f7ae6",
 "ner_common.py": "020d134c97ddb4ef8fc97810501a107c",
 "ner_models.py": "6e3e619bf13a1248284f5d75184923e0",
 "ner_peft.py": "d418dc954f786d538ba288e4f34ca454",
 "cjpe_common.py": "7dfcb0910a5eae9a9c7fdb0278c02669",
 "cjpe_data.py": "b98947470aa7a63df75625512a0a45ff",
 "cjpe_tok.py": "8502c252f6cd039160ad2ac45d2fa55b",
 "cjpe_models.py": "275de149af23054f9dabc22804669279",
 "cjpe_peft.py": "1a2675371e02330c7de371609715686a",
 "cjpe_eval.py": "4786294a9c58dcb56e871ba01eff8ba9",
 "cjpe_kd.py": "96e8e80de8ddc909254ddc178a45b9c7",
 "lsi_ml.py": "693934d09f3df14d13586f1e1f03a2e1",
 "cjpe_train.py": "e8c11b7d4edcbf2a0c1d274f43e41248",
 "gen_ml.py": "3ec202e4867d9884be25d4206d267ee6",
 "summ_model.py": "a7073f4aafb82447c8044a7303cd1c38",
 "summ_data.py": "461891e6e1fe17fee1be532f3edadd01",
 "summ_train.py": "e9ca48b42f75db32a0f6eec877ab33a1",
 "cjpe_predict.py": "ad3f7efab7c537387ed98b3a6215b79e",
 "validate_submission.py": "cfb2cc0d259e3eab0ff2c64b7186980e",
 "select_best.py": "09cc07884ffa4ad8147f22b77a18d867",
 "aggregate.py": "612b171e984e42a4a3313b0bdab143bd",
 "run_experiment.py": "73de0395276e4c9bc5771edc56e4d637",
 "selftest.py": "6302b3a94127a9f018d2c84bb739aa8e"
}
# BERTScore (dev scoring only; never raises)
try:
    import bert_score  # noqa
    print('bert-score: ok')
except Exception:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bert-score'],
                       capture_output=True, text=True)
    print('bert-score:', 'ok' if r.returncode == 0 else 'NOT available -> ROUGE only ' + r.stderr[-300:])


In [ ]:
%%writefile /kaggle/working/phase6/bpe_tokenizer.py
"""
bpe_tokenizer.py -- from-scratch byte-level BPE (pure Python, stdlib + optional numpy/torch).

No `tokenizers`, `transformers`, `sentencepiece`, `tiktoken` or any other tokenizer library.
Only: re, heapq, json, collections, random, time, gc.

Token-id layout (matches the original notebook's special-token ids):
    0 <|endoftext|>   1 <|pad|>   2 <|unk|>   3..258 = raw bytes 0..255   259.. = merges
Internally training works with "internal ids" (byte b -> b, merge k -> 256+k);
the public API adds NUM_SPECIAL (=3) to every id.
"""
import gc
import heapq
import json
import random
import re
import time
from collections import Counter
from pathlib import Path

SPECIALS = ["<|endoftext|>", "<|pad|>", "<|unk|>"]
NUM_SPECIAL = len(SPECIALS)

# GPT-2 style pre-tokenisation, expressed with the stdlib `re` module
# (\p{L} -> [^\W\d_],  \p{N} -> \d,  punctuation/symbols -> [^\s\w] or '_').
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)"      # English contractions
    r"| ?[^\W\d_]+"               # optional space + letters
    r"| ?\d+"                     # optional space + digits
    r"| ?(?:[^\s\w]|_)+"          # optional space + punctuation / symbols
    r"|\s+(?!\S)"                 # whitespace run not followed by a non-space
    r"|\s+"                       # remaining whitespace
)

MAX_PRETOKEN_CHARS = 512          # pathological runs (e.g. 50k dashes) are cut into pieces


# =============================================================================
# 1. TRAINING
# =============================================================================
def count_pretokens(texts, max_unique=6_000_000, max_word_chars=64, log_every=20000,
                    log=print):
    """Stream documents -> Counter of pre-tokens, with a hard memory ceiling.

    When the Counter grows past `max_unique`, singletons (then rarer and rarer words) are
    pruned, so RAM stays bounded no matter how large the corpus is.
    """
    counter = Counter()
    findall = PRETOKEN_RE.findall
    n_docs = n_chars = 0
    t0 = time.time()
    for text in texts:
        counter.update(findall(text))
        n_docs += 1
        n_chars += len(text)
        if len(counter) > max_unique:
            thr = 2
            while len(counter) > int(max_unique * 0.7):
                counter = Counter({k: v for k, v in counter.items() if v >= thr})
                thr += 1
            gc.collect()
            log(f"   [count] pruned to {len(counter):,} unique pre-tokens (min count {thr - 1})")
        if n_docs % log_every == 0:
            log(f"   [count] {n_docs:,} docs | {n_chars/1e6:,.0f}M chars | "
                f"{len(counter):,} unique | {time.time()-t0:.0f}s")
    # drop absurdly long pre-tokens (junk); they carry no useful merge statistics
    if max_word_chars:
        for k in [k for k in counter if len(k) > max_word_chars]:
            del counter[k]
    return counter


def train_bpe(word_counts, vocab_size, max_words=2_000_000, min_word_count=2,
              min_pair_count=2, n_verify=5000, seed=0, log_every=1000, log=print):
    """Learn BPE merges from a Counter of pre-tokens.

    Classic incremental algorithm: exact pair counts + inverted index (pair -> words that
    contain it) + lazy max-heap, so each merge only touches the words that contain the pair.

    Returns (merges, verify) where
        merges : list[(a, b)] in rank order, in INTERNAL ids (byte b -> b, merge k -> 256+k)
        verify : list[(pretoken_str, final_internal_ids)] used to cross-check the encoder
    """
    num_merges = vocab_size - NUM_SPECIAL - 256
    assert num_merges > 0, "vocab_size too small"

    # ---- 1. pick the training vocabulary of words ---------------------------------
    items = [(w, c) for w, c in word_counts.items() if c >= min_word_count]
    if len(items) > max_words:
        items.sort(key=lambda kv: kv[1], reverse=True)
        items = items[:max_words]
    words, freqs, strs = [], [], []
    for w, c in items:
        ids = list(w.encode("utf-8", "replace"))
        if len(ids) > 1:
            words.append(ids); freqs.append(c); strs.append(w)
    del items
    log(f"   [bpe] training on {len(words):,} unique words "
        f"({sum(freqs)/1e6:,.1f}M word occurrences)")

    # ---- 2. initial pair statistics ------------------------------------------------
    pc = {}        # pair -> total count
    where = {}     # pair -> set(word index)
    for idx, ids in enumerate(words):
        f = freqs[idx]
        for p in zip(ids, ids[1:]):
            if p in pc:
                pc[p] += f
                where[p].add(idx)
            else:
                pc[p] = f
                where[p] = {idx}
    heap = [(-c, p) for p, c in pc.items()]
    heapq.heapify(heap)

    merges = []
    t0 = time.time()
    next_id = 256
    while next_id < 256 + num_merges:
        # ---- pick the most frequent pair (lazy heap: skip stale entries) -----------
        best = None
        while heap:
            negc, p = heapq.heappop(heap)
            if pc.get(p, 0) == -negc:
                best = (-negc, p)
                break
        if best is None:
            log("   [bpe] no more pairs to merge -- stopping early")
            break
        count, (a, b) = best
        if count < min_pair_count:
            log(f"   [bpe] best pair count {count} < {min_pair_count} -- stopping early")
            break

        merges.append((a, b))
        new = next_id
        next_id += 1

        # ---- apply the merge to every word that contains it ------------------------
        idxs = where.pop((a, b), ())
        changed = {}                      # pair -> count before this merge
        for idx in idxs:
            ids = words[idx]
            n = len(ids)
            out = []
            i = 0
            hit = False
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new); i += 2; hit = True
                else:
                    out.append(ids[i]); i += 1
            if not hit:
                continue
            f = freqs[idx]
            old_pairs = list(zip(ids, ids[1:]))
            new_pairs = list(zip(out, out[1:]))
            for q in old_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] -= f
            for q in new_pairs:
                if q not in changed:
                    changed[q] = pc.get(q, 0)
                pc[q] = pc.get(q, 0) + f
            new_set = set(new_pairs)
            for q in set(old_pairs) - new_set:
                s = where.get(q)
                if s is not None:
                    s.discard(idx)
            for q in new_set:
                s = where.get(q)
                if s is None:
                    where[q] = {idx}
                else:
                    s.add(idx)
            words[idx] = out

        # ---- publish the changed pair counts ---------------------------------------
        for q, orig in changed.items():
            c = pc.get(q, 0)
            if c <= 0:
                pc.pop(q, None)
                where.pop(q, None)
            elif c != orig:
                heapq.heappush(heap, (-c, q))
        # keep the heap from growing without bound
        if len(heap) > max(4_000_000, 8 * len(pc)):
            heap = [(-c, q) for q, c in pc.items()]
            heapq.heapify(heap)

        k = len(merges)
        if k % log_every == 0 or k == num_merges:
            log(f"   [bpe] merge {k:,}/{num_merges:,} | last pair count {count:,} | "
                f"{len(pc):,} live pairs | {time.time()-t0:.0f}s")

    # ---- 3. sample of final segmentations (to verify the encoder) ----------------
    rng = random.Random(seed)
    pick = rng.sample(range(len(words)), min(n_verify, len(words)))
    verify = [(strs[i], list(words[i])) for i in pick]
    del words, freqs, strs, pc, where, heap
    gc.collect()
    return merges, verify


# =============================================================================
# 2. TOKENIZER (encode / decode / save / load)
# =============================================================================
class BPETokenizer:
    """Byte-level BPE tokenizer with a small HuggingFace-like surface:
    tok(text, add_special_tokens=False)["input_ids"], tok.encode, tok.decode,
    len(tok), eos_token_id / pad_token_id, padding_side / truncation_side.
    """

    def __init__(self, merges, cache_max=2_000_000):
        self.merges = [tuple(m) for m in merges]
        self.vocab_size = NUM_SPECIAL + 256 + len(self.merges)
        self._merge_map = {m: 256 + r for r, m in enumerate(self.merges)}   # pair -> internal id
        # id -> bytes (internal ids)
        self._bytes = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self._bytes.append(self._bytes[a] + self._bytes[b])
        self._special_bytes = [s.encode("utf-8") for s in SPECIALS]

        self._cache = {}
        self._cache_max = cache_max

        self.eos_token, self.bos_token = SPECIALS[0], SPECIALS[0]
        self.pad_token, self.unk_token = SPECIALS[1], SPECIALS[2]
        self.eos_token_id = self.bos_token_id = 0
        self.pad_token_id = 1
        self.unk_token_id = 2
        self.padding_side = "left"        # decoder-only batched generation
        self.truncation_side = "left"     # an over-long prompt loses its beginning

    # ------------------------------------------------------------------ basics
    def __len__(self):
        return self.vocab_size

    def get_vocab(self):
        v = {s: i for i, s in enumerate(SPECIALS)}
        for i, bs in enumerate(self._bytes):
            v[bs.decode("latin-1") + f"#{i}"] = i + NUM_SPECIAL   # unique readable key
        return v

    # ------------------------------------------------------------------ encoding
    def _bpe_word(self, b):
        """BPE-merge one pre-token given as bytes -> tuple of PUBLIC ids."""
        ids = list(b)
        mm = self._merge_map
        while len(ids) > 1:
            best = None
            bp = None
            for p in zip(ids, ids[1:]):
                m = mm.get(p)
                if m is not None and (best is None or m < best):
                    best, bp = m, p
            if best is None:
                break
            a, c = bp
            out = []
            i, n = 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == c:
                    out.append(best); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        return tuple(i + NUM_SPECIAL for i in ids)

    def _encode_pretoken(self, w):
        if len(w) > MAX_PRETOKEN_CHARS:
            res = ()
            for s in range(0, len(w), MAX_PRETOKEN_CHARS):
                res += self._bpe_word(w[s:s + MAX_PRETOKEN_CHARS].encode("utf-8", "replace"))
            return res
        return self._bpe_word(w.encode("utf-8", "replace"))

    def encode(self, text, add_special_tokens=False):
        cache = self._cache
        out = []
        ext = out.extend
        for w in PRETOKEN_RE.findall(text):
            t = cache.get(w)
            if t is None:
                t = self._encode_pretoken(w)
                if len(cache) >= self._cache_max:
                    cache.clear()                 # bounded RAM
                cache[w] = t
            ext(t)
        return out

    def __call__(self, text, add_special_tokens=False, padding=False, truncation=False,
                 max_length=None, return_tensors=None, **_ignored):
        single = isinstance(text, str)
        batch = [text] if single else list(text)
        enc = [self.encode(t) for t in batch]
        if truncation and max_length is not None:
            if self.truncation_side == "left":
                enc = [e[-max_length:] if len(e) > max_length else e for e in enc]
            else:
                enc = [e[:max_length] for e in enc]
        masks = [[1] * len(e) for e in enc]
        if padding:
            target = max_length if (padding == "max_length" and max_length) else max(len(e) for e in enc)
            for k, e in enumerate(enc):
                gap = target - len(e)
                if gap > 0:
                    if self.padding_side == "left":
                        enc[k] = [self.pad_token_id] * gap + e
                        masks[k] = [0] * gap + masks[k]
                    else:
                        enc[k] = e + [self.pad_token_id] * gap
                        masks[k] = masks[k] + [0] * gap
        if return_tensors in ("pt", "np"):
            if return_tensors == "pt":
                import torch
                mk = lambda x: torch.tensor(x, dtype=torch.long)
            else:
                import numpy as np
                mk = lambda x: np.asarray(x, dtype=np.int64)
            return {"input_ids": mk(enc), "attention_mask": mk(masks)}
        if single:
            return {"input_ids": enc[0], "attention_mask": masks[0]}
        return {"input_ids": enc, "attention_mask": masks}

    # ------------------------------------------------------------------ decoding
    def decode(self, ids, skip_special_tokens=False):
        if hasattr(ids, "tolist"):
            ids = ids.tolist()
        parts = []
        nb = len(self._bytes)
        for i in ids:
            i = int(i)
            if i < NUM_SPECIAL:
                if not skip_special_tokens and i >= 0:
                    parts.append(self._special_bytes[i])
            elif i - NUM_SPECIAL < nb:
                parts.append(self._bytes[i - NUM_SPECIAL])
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, seqs, skip_special_tokens=False):
        return [self.decode(s, skip_special_tokens) for s in seqs]

    # ------------------------------------------------------------------ persistence
    def save(self, path):
        obj = {"type": "byte_level_bpe", "version": 1, "specials": SPECIALS,
               "num_special": NUM_SPECIAL, "vocab_size": self.vocab_size,
               "pretoken_regex": PRETOKEN_RE.pattern, "merges": self.merges}
        with open(path, "w") as f:
            json.dump(obj, f)

    def save_pretrained(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        self.save(out_dir / "bpe_model.json")
        with open(out_dir / "tokenizer_config.json", "w") as f:
            json.dump({"eos_token": self.eos_token, "pad_token": self.pad_token,
                       "unk_token": self.unk_token, "eos_token_id": 0, "pad_token_id": 1,
                       "unk_token_id": 2, "vocab_size": self.vocab_size,
                       "padding_side": self.padding_side,
                       "truncation_side": self.truncation_side}, f, indent=2)

    @classmethod
    def load(cls, path, **kw):
        with open(path) as f:
            obj = json.load(f)
        assert obj.get("type") == "byte_level_bpe", "not a byte_level_bpe file"
        assert obj["pretoken_regex"] == PRETOKEN_RE.pattern, "pre-tokenizer regex mismatch"
        return cls(obj["merges"], **kw)

    @classmethod
    def from_pretrained(cls, directory, **kw):
        return cls.load(Path(directory) / "bpe_model.json", **kw)


In [ ]:
%%writefile /kaggle/working/phase6/phase1_models.py
"""
phase1_models.py -- the Phase-1 model code, copied VERBATIM from the Phase-1 training notebooks.

  * cell 13 of every notebook (identical in all four): shared core
    (CausalLMOutput, positions_from_mask, causal_allowed, build_attn_mask, init_weights,
     decoding helpers, CausalLMBase). The only change: its trailing print() was removed so
     that importing this module is silent.
  * cell 15, class definitions only (everything before `MODEL_CLASS, CONFIG_CLASS = ...`):
      - 01_train_gpt2style       -> GPT2StyleConfig, GPT2Attention, GELUMLP, GPT2Block, GPT2StyleLM
      - 02_train_llamastyle      -> LlamaStyleConfig, RMSNorm, RotaryEmbedding, GQAttention, SwiGLU,
                                    LlamaBlock, LlamaStyleLM
        (03_train_mistral_sliding defines the SAME classes; it differs only by a module-level
         SLIDING_WINDOW constant and one comment. The window comes from config.json, so one
         copy serves both M2 and M3.)
      - 04_train_bloom_alibi     -> BloomStyleConfig, alibi_slopes, alibi_bias, BloomAttention,
                                    BloomMLP, BloomBlock, BloomStyleLM

DO NOT EDIT the copied code: state_dict keys of the Phase-1 `model.pt` files depend on it.
Everything NER-specific lives in ner_models.py.
"""

# =============================================================================================
# notebook cell 13 (shared core)
# =============================================================================================
import math
from dataclasses import dataclass, asdict, field
from typing import Optional, List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view


@dataclass
class CausalLMOutput:
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[list] = None      # per layer: (k, v), each (B, n_kv_heads, S, head_dim)


def positions_from_mask(attention_mask, past_len, T, B, device):
    """Position ids for the T new tokens. attention_mask (if given) covers ALL keys: (B, past_len+T)."""
    if attention_mask is None:
        return torch.arange(past_len, past_len + T, device=device).unsqueeze(0).expand(B, T)
    pos = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
    return pos[:, -T:]


def causal_allowed(T, past_len, attention_mask, window, device):
    """Bool (B|1, 1, T, S): True where query t may attend key s. Always materialised."""
    S = past_len + T
    q = torch.arange(past_len, S, device=device)[:, None]      # absolute index of each query
    k = torch.arange(S, device=device)[None, :]
    allowed = k <= q                                             # causal
    if window is not None:
        allowed = allowed & (q - k < window)                     # sliding window: last `window` keys incl. self
    allowed = allowed[None, None]
    if attention_mask is not None:
        allowed = allowed & attention_mask[:, None, None, :].bool()   # drop padding keys
        allowed = allowed | (q == k)[None, None]                 # a pad query would see nothing -> NaN; let it see itself
    return allowed


def build_attn_mask(T, past_len, attention_mask, window, device):
    """Returns (mask, is_causal) for F.scaled_dot_product_attention."""
    S = past_len + T
    window_active = window is not None and window < S
    no_pad = attention_mask is None or bool(attention_mask.all())
    if no_pad and not window_active:
        if past_len == 0:
            return None, True        # fast fused causal kernel (training path)
        if T == 1:
            return None, False       # single cached query sees every key
    return causal_allowed(T, past_len, None if no_pad else attention_mask,
                          window if window_active else None, device), False


def init_weights(model, n_layer, std=0.02):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():       # residual-branch output projections
        if name.endswith("o_proj.weight") or name.endswith("down_proj.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layer))


# ------------------------------------------------------------------ decoding helpers
def _sample(logits, temperature, top_p, top_k):
    logits = logits / max(float(temperature), 1e-5)
    if top_k and top_k > 0:
        kth = torch.topk(logits, min(int(top_k), logits.size(-1)), dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None and top_p < 1.0:
        sl, si = torch.sort(logits, descending=True, dim=-1)
        probs = sl.softmax(-1)
        remove = probs.cumsum(-1) - probs > top_p           # keep the smallest set with mass >= top_p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)


def _banned_ngram_tokens(row, n):
    """Tokens that would complete an n-gram already present in `row` (list of ints)."""
    if n < 2 or len(row) < n:
        return []
    arr = np.asarray(row)
    win = sliding_window_view(arr[:-1], n - 1)           # window i = arr[i:i+n-1], followed by arr[i+n-1]
    hit = np.nonzero((win == arr[-(n - 1):]).all(1))[0]
    return arr[hit + n - 1].tolist()


class CausalLMBase(nn.Module):
    """Loss + generation shared by all four architectures."""

    @staticmethod
    def lm_loss(logits, labels):
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)).float(),
                               labels[:, 1:].reshape(-1), ignore_index=-100)

    def _process_logits(self, logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress):
        if repetition_penalty and repetition_penalty != 1.0:
            sc = torch.gather(logits, 1, seq)
            sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
            logits = logits.scatter(1, seq, sc)
        if no_repeat_ngram_size and no_repeat_ngram_size > 1:
            for b, row in enumerate(rows):
                banned = _banned_ngram_tokens(row, no_repeat_ngram_size)
                if banned:
                    logits[b, banned] = float("-inf")
        if suppress:
            logits[:, list(suppress)] = float("-inf")
        return logits

    @torch.inference_mode()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=256, do_sample=False,
                 temperature=1.0, top_p=1.0, top_k=0, repetition_penalty=1.0,
                 no_repeat_ngram_size=0, num_beams=1, suppress_token_ids=(),
                 stop_fn=None, stop_check_every=16, **_ignored):
        """Returns ONLY the new tokens, shape (B, n_new). stop_fn(new_tokens) -> True ends early."""
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        attention_mask = attention_mask.long()
        rows = [ids[m.bool()].tolist() for ids, m in zip(input_ids, attention_mask)]   # real tokens only
        if num_beams > 1:
            return self._beam_search(input_ids, attention_mask, rows, max_new_tokens, num_beams,
                                     repetition_penalty, no_repeat_ngram_size, suppress_token_ids,
                                     stop_fn, stop_check_every)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
        seq, new = input_ids, []
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty,
                                          no_repeat_ngram_size, suppress_token_ids)
            nxt = _sample(logits, temperature, top_p, top_k) if do_sample else logits.argmax(-1)
            new.append(nxt)
            seq = torch.cat([seq, nxt[:, None]], 1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(nxt)[:, None]], 1)
            for row, t in zip(rows, nxt.tolist()):
                row.append(t)
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(torch.stack(new, 1)):
                break
            out = self(nxt[:, None], attention_mask=attention_mask, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        return torch.stack(new, 1)

    def _beam_search(self, input_ids, attention_mask, rows, max_new_tokens, K, repetition_penalty,
                     no_repeat_ngram_size, suppress, stop_fn, stop_check_every):
        B = input_ids.size(0)
        out = self(input_ids, attention_mask=attention_mask, use_cache=True, last_only=True)
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in out.past_key_values]
        logits = out.logits[:, -1].float().repeat_interleave(K, 0)
        seq = input_ids.repeat_interleave(K, 0)
        am = attention_mask.repeat_interleave(K, 0)
        rows = [list(r) for r in rows for _ in range(K)]
        scores = torch.zeros(B, K, device=input_ids.device)
        scores[:, 1:] = -1e9                       # all beams start identical: expand from beam 0 only
        gen = seq[:, :0]
        for step in range(max_new_tokens):
            logits = self._process_logits(logits, seq, rows, repetition_penalty, no_repeat_ngram_size, suppress)
            logp = logits.log_softmax(-1)
            V = logp.size(-1)
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            gen = torch.cat([gen[src], tok[:, None]], 1)
            seq = torch.cat([seq[src], tok[:, None]], 1)
            am = torch.cat([am[src], torch.ones_like(tok)[:, None]], 1)
            rows = [rows[i] + [t] for i, t in zip(src.tolist(), tok.tolist())]
            past = [(k[src], v[src]) for k, v in past]
            if step == max_new_tokens - 1:
                break
            if stop_fn is not None and (step + 1) % stop_check_every == 0 and stop_fn(gen):
                break
            out = self(tok[:, None], attention_mask=am, past_key_values=past, use_cache=True)
            past, logits = out.past_key_values, out.logits[:, -1].float()
        best = scores.argmax(1)
        return gen.view(B, K, -1)[torch.arange(B, device=best.device), best]


# =============================================================================================
# 01_train_gpt2style, cell 15 (classes)
# =============================================================================================
@dataclass
class GPT2StyleConfig:
    vocab_size: int
    n_positions: int = 2048
    n_embd: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


class GPT2Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.n_head, self.head_dim, self.d = c.n_head, c.n_embd // c.n_head, c.n_embd
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.o_proj = nn.Linear(c.n_embd, c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split(self.d, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # (B, H, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        y = y.transpose(1, 2).reshape(B, T, self.d)                        # merge heads
        return self.o_proj(y), (k, v)


class GELUMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class GPT2Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.attn = GPT2Attention(c)
        self.ln_2 = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.mlp = GELUMLP(c.n_embd, c.mlp_ratio * c.n_embd)

    def forward(self, x, mask, is_causal, past=None):
        a, present = self.attn(self.ln_1(x), mask, is_causal, past)       # pre-LN residual
        x = x + a
        x = x + self.mlp(self.ln_2(x))
        return x, present


class GPT2StyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.wte = nn.Embedding(c.vocab_size, c.n_embd)
        self.wpe = nn.Embedding(c.n_positions, c.n_embd)
        self.blocks = nn.ModuleList([GPT2Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.wte.weight                              # tied input/output embeddings

    def get_input_embeddings(self):
        return self.wte

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        pos = pos.clamp(max=self.config.n_positions - 1)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask, None, input_ids.device)
        x = self.wte(input_ids) + self.wpe(pos)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 02_train_llamastyle, cell 15 (classes) -- also used for 03_train_mistral_sliding
# =============================================================================================
@dataclass
class LlamaStyleConfig:
    vocab_size: int
    max_position_embeddings: int = 2048
    hidden_size: int = 768
    intermediate_size: int = 3072
    num_hidden_layers: int = 12
    num_attention_heads: int = 12
    num_key_value_heads: int = 4
    rope_theta: float = 10000.0
    rms_norm_eps: float = 1e-5
    sliding_window: Optional[int] = None      # None = full causal attention
    initializer_range: float = 0.02


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).to(x.dtype)


class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, theta=10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, pos):                                   # pos: (B, T) int
        freqs = pos[..., None].float() * self.inv_freq        # (B, T, hd/2)
        emb = torch.cat([freqs, freqs], dim=-1)               # (B, T, hd): angle for dim i and i+hd/2
        return emb.cos()[:, None], emb.sin()[:, None]         # (B, 1, T, hd) -> broadcast over heads


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    return (x * cos + rotate_half(x) * sin).to(x.dtype)


class GQAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.num_attention_heads == 0
        assert c.num_attention_heads % c.num_key_value_heads == 0, "KV heads must divide query heads"
        self.h, self.kv = c.num_attention_heads, c.num_key_value_heads
        self.hd = c.hidden_size // c.num_attention_heads
        self.n_rep = self.h // self.kv
        self.q_proj = nn.Linear(c.hidden_size, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(c.hidden_size, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, c.hidden_size, bias=False)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)    # (B, 12, T, hd)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)   # (B,  4, T, hd)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)           # rotate Q and K only
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        present = (k, v)                                                  # cache stays at 4 heads
        if self.n_rep > 1:                                                # kv head j serves q heads j*3..j*3+2
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=is_causal)
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.h * self.hd)), present


class SwiGLU(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d, hidden, bias=False)
        self.up_proj = nn.Linear(d, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class LlamaBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.attn = GQAttention(c)
        self.post_attn_norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.mlp = SwiGLU(c.hidden_size, c.intermediate_size)

    def forward(self, x, cos, sin, mask, is_causal, past=None):
        a, present = self.attn(self.input_norm(x), cos, sin, mask, is_causal, past)
        x = x + a
        x = x + self.mlp(self.post_attn_norm(x))
        return x, present


class LlamaStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embed_tokens = nn.Embedding(c.vocab_size, c.hidden_size)
        self.rotary = RotaryEmbedding(c.hidden_size // c.num_attention_heads, c.rope_theta)
        self.layers = nn.ModuleList([LlamaBlock(c) for _ in range(c.num_hidden_layers)])
        self.norm = RMSNorm(c.hidden_size, c.rms_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        init_weights(self, c.num_hidden_layers, c.initializer_range)
        self.lm_head.weight = self.embed_tokens.weight

    def get_input_embeddings(self):
        return self.embed_tokens

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        pos = positions_from_mask(attention_mask, past_len, T, B, input_ids.device)
        cos, sin = self.rotary(pos)
        mask, is_causal = build_attn_mask(T, past_len, attention_mask,
                                          self.config.sliding_window, input_ids.device)
        x = self.embed_tokens(input_ids)
        presents = []
        for i, layer in enumerate(self.layers):
            x, pr = layer(x, cos, sin, mask, is_causal, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.norm(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# =============================================================================================
# 04_train_bloom_alibi, cell 15 (classes)
# =============================================================================================
@dataclass
class BloomStyleConfig:
    vocab_size: int
    hidden_size: int = 768
    n_layer: int = 14
    n_head: int = 12
    mlp_ratio: int = 4
    layer_norm_eps: float = 1e-5
    initializer_range: float = 0.02


def alibi_slopes(n_head):
    def pow2_slopes(n):
        start = 2.0 ** (-(2.0 ** -(math.log2(n) - 3)))
        return [start * start ** i for i in range(n)]
    if math.log2(n_head).is_integer():
        return pow2_slopes(n_head)
    closest = 2 ** math.floor(math.log2(n_head))
    return pow2_slopes(closest) + alibi_slopes(2 * closest)[0::2][: n_head - closest]


def alibi_bias(pos_q, pos_k, slopes, allowed):
    """pos_q (B,T), pos_k (B,S), slopes (H,), allowed bool (B|1,1,T,S) -> float bias (B,H,T,S)."""
    dist = (pos_q[:, None, :, None] - pos_k[:, None, None, :]).clamp(min=0).float()
    bias = -slopes.view(1, -1, 1, 1) * dist
    return bias.masked_fill(~allowed, float("-inf"))


class BloomAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.hidden_size % c.n_head == 0
        self.h, self.hd, self.d = c.n_head, c.hidden_size // c.n_head, c.hidden_size
        self.query_key_value = nn.Linear(c.hidden_size, 3 * c.hidden_size)
        self.o_proj = nn.Linear(c.hidden_size, c.hidden_size)

    def forward(self, x, bias, past=None):
        B, T, _ = x.shape
        q, k, v = self.query_key_value(x).split(self.d, dim=-1)
        q = q.view(B, T, self.h, self.hd).transpose(1, 2)
        k = k.view(B, T, self.h, self.hd).transpose(1, 2)
        v = v.view(B, T, self.h, self.hd).transpose(1, 2)
        if past is not None:
            k, v = torch.cat([past[0], k], 2), torch.cat([past[1], v], 2)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype))   # additive ALiBi bias
        return self.o_proj(y.transpose(1, 2).reshape(B, T, self.d)), (k, v)


class BloomMLP(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.up_proj = nn.Linear(d, hidden)
        self.down_proj = nn.Linear(hidden, d)

    def forward(self, x):
        return self.down_proj(F.gelu(self.up_proj(x), approximate="tanh"))


class BloomBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.input_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.self_attention = BloomAttention(c)
        self.post_attention_layernorm = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.mlp = BloomMLP(c.hidden_size, c.mlp_ratio * c.hidden_size)

    def forward(self, x, bias, past=None):
        a, present = self.self_attention(self.input_layernorm(x), bias, past)
        x = x + a
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x, present


class BloomStyleLM(CausalLMBase):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.word_embeddings = nn.Embedding(c.vocab_size, c.hidden_size)
        self.blocks = nn.ModuleList([BloomBlock(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.hidden_size, eps=c.layer_norm_eps)
        self.lm_head = nn.Linear(c.hidden_size, c.vocab_size, bias=False)
        self.register_buffer("slopes", torch.tensor(alibi_slopes(c.n_head), dtype=torch.float32),
                             persistent=False)
        init_weights(self, c.n_layer, c.initializer_range)
        self.lm_head.weight = self.word_embeddings.weight

    def get_input_embeddings(self):
        return self.word_embeddings

    def forward(self, input_ids, labels=None, attention_mask=None, past_key_values=None,
                use_cache=False, last_only=False):
        B, T = input_ids.shape
        dev = input_ids.device
        past_len = past_key_values[0][0].size(2) if past_key_values else 0
        S = past_len + T
        if attention_mask is None:                          # all rows identical -> batch dim 1
            pos_k = torch.arange(S, device=dev)[None]
            allowed = causal_allowed(T, past_len, None, None, dev)
        else:
            pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
            allowed = causal_allowed(T, past_len, attention_mask, None, dev)
        bias = alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)    # computed once, shared by all layers
        x = self.word_embeddings(input_ids)                  # no position embedding, no embedding LayerNorm (see note)
        presents = []
        for i, blk in enumerate(self.blocks):
            x, pr = blk(x, bias, past_key_values[i] if past_key_values else None)
            presents.append(pr)
        x = self.ln_f(x)
        if last_only:
            x = x[:, -1:]
        logits = self.lm_head(x)
        loss = self.lm_loss(logits, labels) if labels is not None else None
        return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents if use_cache else None)


# ---------------------------------------------------------------------------------------------
# Registry (added for Phase 2; not part of the copied code)
# ---------------------------------------------------------------------------------------------
MODEL_CLASSES = {
    "GPT2StyleLM": (GPT2StyleLM, GPT2StyleConfig),
    "LlamaStyleLM": (LlamaStyleLM, LlamaStyleConfig),
    "BloomStyleLM": (BloomStyleLM, BloomStyleConfig),
}


In [ ]:
%%writefile /kaggle/working/phase6/ner_common.py
"""
ner_common.py -- small torch-free helpers shared by every Phase-2 module.

  * registries: the four Phase-1 models (M1..M4) and the experiments (E1..E6, extras)
  * file discovery under /kaggle/input (or any root), atomic JSON writes
  * disk guard and output-size accounting (no Hugging Face Hub anywhere in this project)
"""
import glob
import hashlib
import json
import os
import shutil
import time

# ---------------------------------------------------------------------------------------------
# Registries
# ---------------------------------------------------------------------------------------------
# `arch` must equal the "arch" field the Phase-1 notebooks wrote into final/config.json.
MODELS = {
    "M1": {"arch": "gpt2style", "folder": "M1_gpt2style", "kind": "gpt2"},
    "M2": {"arch": "llamastyle", "folder": "M2_llamastyle", "kind": "llama"},
    "M3": {"arch": "mistral-sliding", "folder": "M3_mistral_sliding", "kind": "llama"},
    "M4": {"arch": "bloom-alibi", "folder": "M4_bloom_alibi", "kind": "bloom"},
}
DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]          # strongest Phase-1 LM first

# Layer subsets for the layer-pruned KD students (every second layer; section 8 of the brief).
STUDENT_LAYERS = {
    "M1": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
    "M2": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M3": [0, 2, 4, 6, 8, 10],          # 6 of 12
    "M4": [0, 2, 4, 6, 8, 10, 12],      # 7 of 14
}

# The six required experiments + extras. `technique` selects the code path in ner_peft.py.
EXPERIMENTS = {
    "E1": {"folder": "E1_full_ft", "technique": "full_ft", "title": "Full fine-tuning"},
    "E2": {"folder": "E2_lorma", "technique": "lorma", "title": "LoRMA+ (compulsory)"},
    "E3": {"folder": "E3_lora", "technique": "lora", "title": "LoRA"},
    "E4": {"folder": "E4_adapters", "technique": "adapters", "title": "Houlsby adapters"},
    "E5": {"folder": "E5_partial_ft", "technique": "partial_ft", "title": "Partial / layer-wise FT"},
    "E6": {"folder": "E6_kd", "technique": "kd", "title": "Knowledge distillation"},
    # extras (not counted among the six)
    "X1": {"folder": "X1_bitfit", "technique": "bitfit", "title": "BitFit (M1/M4 only)",
           "models": ["M4", "M1"]},
    "X2": {"folder": "X2_student_no_kd", "technique": "student_hard",
           "title": "Pruned student, hard labels only (KD control)"},
}
REQUIRED_EXPS = ["E1", "E2", "E3", "E4", "E5", "E6"]

# Default learning rates (section 9.3). The LR probe multiplies `lr` by {1/3, 1, 3}.
DEFAULT_LR = {
    "full_ft": 5e-5, "lorma": 1e-3, "lora": 1e-3, "adapters": 1e-3, "bitfit": 2e-3,
    "partial_ft": 1e-4, "kd": 1e-4, "student_hard": 1e-4,
}
DEFAULT_EPOCHS = {
    "full_ft": 6, "kd": 6, "student_hard": 6, "partial_ft": 8,
    "lorma": 10, "lora": 10, "adapters": 10, "bitfit": 10,
}


# ---------------------------------------------------------------------------------------------
# JSON / files
# ---------------------------------------------------------------------------------------------
def write_json(path, obj, indent=2):
    """Atomic JSON write: a crash never leaves a half-written file behind."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=indent, ensure_ascii=False, default=_json_default)
    os.replace(tmp, path)


def _json_default(o):
    try:
        import numpy as np
        if isinstance(o, np.integer):
            return int(o)
        if isinstance(o, np.floating):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
    except ImportError:
        pass
    raise TypeError(f"not JSON serialisable: {type(o)}")


def read_json(path, default=None):
    if not os.path.exists(path):
        return default
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def find_files(name, roots, max_hits=50):
    """All files called `name` below any of `roots` (recursive), sorted for determinism."""
    hits = []
    for r in roots:
        if r and os.path.isdir(r):
            hits.extend(glob.glob(os.path.join(r, "**", name), recursive=True))
    return sorted(set(hits))[:max_hits]


def default_input_roots():
    """Where attached Kaggle datasets live; PHASE2_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE2_INPUT_ROOTS")
    if env:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def iter_finished_runs(out_root, input_roots=()):
    """Yield (exp_folder, model_folder, run_dir) for every finished run (DONE.json):
    directly under `out_root` (<exp>/<model>/DONE.json, whatever out_root is called) and under
    any `phase2_out` folder inside the attached input datasets. First occurrence wins."""
    seen = set()
    out_root = os.path.abspath(out_root) if out_root else None
    if out_root and os.path.isdir(out_root):
        for d in sorted(glob.glob(os.path.join(out_root, "*", "*", "DONE.json"))):
            exp_f, mod_f = d.split(os.sep)[-3], d.split(os.sep)[-2]
            if mod_f != "BEST" and (exp_f, mod_f) not in seen:
                seen.add((exp_f, mod_f))
                yield exp_f, mod_f, os.path.dirname(d)
    for d in find_files("DONE.json", list(input_roots), max_hits=5000):
        parts = d.split(os.sep)
        if "phase2_out" not in parts:
            continue
        i = len(parts) - 1 - parts[::-1].index("phase2_out")
        if len(parts) - i != 4 or parts[-2] == "BEST":
            continue
        if (parts[i + 1], parts[-2]) not in seen:
            seen.add((parts[i + 1], parts[-2]))
            yield parts[i + 1], parts[-2], os.path.dirname(d)


def find_phase1_models(roots=None):
    """Map M1..M4 -> directory holding that model's Phase-1 `model.pt` + `config.json`.

    Identified by the "arch" field the Phase-1 notebooks wrote into config.json, so the
    dataset/folder names do not matter.
    """
    roots = roots or default_input_roots()
    by_arch = {}
    for cfg_path in find_files("config.json", roots, max_hits=10000):
        d = os.path.dirname(cfg_path)
        if not os.path.exists(os.path.join(d, "model.pt")):
            continue
        try:
            cfg = read_json(cfg_path)
        except Exception:
            continue
        if isinstance(cfg, dict) and "arch" in cfg and "model_class" in cfg:
            by_arch.setdefault(cfg["arch"], []).append(d)
    out = {}
    for mid, m in MODELS.items():
        cands = by_arch.get(m["arch"], [])
        if cands:
            # prefer a directory literally called "final" (what the Phase-1 notebooks saved)
            cands = sorted(cands, key=lambda p: (os.path.basename(p) != "final", p))
            out[mid] = cands[0]
    return out


# ---------------------------------------------------------------------------------------------
# Disk guard (section 9.5)
# ---------------------------------------------------------------------------------------------
def free_gb(path="/kaggle/working"):
    p = path
    while p and not os.path.exists(p):
        p = os.path.dirname(p)
    return shutil.disk_usage(p or "/").free / 1e9


def dir_size_gb(path):
    tot = 0
    for dp, _dn, fns in os.walk(path):
        for fn in fns:
            fp = os.path.join(dp, fn)
            if not os.path.islink(fp):
                try:
                    tot += os.path.getsize(fp)
                except OSError:
                    pass
    return tot / 1e9


class DiskFull(RuntimeError):
    pass


def disk_guard(out_root, min_free_gb=4.0, log=print, keep=None):
    """Make sure at least `min_free_gb` is free under `out_root`.

    Deletion order: leftover last.pt / last.tmp of models that are finished (DONE.json) or not
    the one currently running (`keep` = its checkpoint dir), then tmp files. Never touches
    best.pt, submissions or reports. Raises DiskFull if still short (callers stop cleanly).
    """
    if free_gb(out_root) >= min_free_gb:
        return
    removed = []
    for pat in ("**/checkpoints/last.pt", "**/checkpoints/last.tmp", "**/*.tmp*"):
        for f in glob.glob(os.path.join(out_root, pat), recursive=True):
            if keep and os.path.abspath(f).startswith(os.path.abspath(keep)):
                continue
            try:
                sz = os.path.getsize(f) / 1e9
                os.remove(f)
                removed.append((f, sz))
            except OSError:
                pass
        if free_gb(out_root) >= min_free_gb:
            break
    for f, sz in removed:
        log(f"   [disk] removed {f} ({sz:.2f} GB)")
    if free_gb(out_root) < min_free_gb:
        raise DiskFull(f"only {free_gb(out_root):.1f} GB free under {out_root} "
                       f"(need {min_free_gb} GB) even after cleanup")


# ---------------------------------------------------------------------------------------------
# misc
# ---------------------------------------------------------------------------------------------
def now():
    return time.time()


def fmt_h(seconds):
    return f"{seconds / 3600:.2f} h"


def log_line(path, msg, also_print=True):
    """Append a timestamped line to a text log (and print it)."""
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + msg
    if also_print:
        print(line, flush=True)
    if path:
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        with open(path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


In [ ]:
%%writefile /kaggle/working/phase6/ner_models.py
"""
ner_models.py -- Phase-1 trunk wrappers + NER heads.

The Phase-1 models' forward() only returns LM logits and is always causal. For NER we need the
final hidden states, optionally with bidirectional attention. `Trunk` re-implements only the
tiny "embed -> blocks -> final norm" loop and REUSES the original submodule objects (no copies;
lm_head is dropped, which also avoids a (B, T, 32000) logits tensor).

Attention modes (section 5.4):
  causal : exactly the Phase-1 masks (verified equal to the original forward in tests)
  bidir  : M1/M2 key-padding mask only; M3 symmetric sliding window |q-k| < window;
           M4 ALiBi with symmetric distance -slope*|q-k|. Padding is on the right.
"""
import json
import os

import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_common import MODELS
from phase1_models import (MODEL_CLASSES, RMSNorm, alibi_bias, build_attn_mask, causal_allowed,
                           positions_from_mask)

MAX_POSITIONS = 1024        # every Phase-1 model was trained on 1024-token blocks


def torch_load(path, map_location="cpu"):
    """torch.load that works on old and new torch (weights_only appeared in 1.13)."""
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)
    except Exception:
        # weights_only=True refuses non-tensor pickles (our resumable checkpoints)
        try:
            return torch.load(path, map_location=map_location, weights_only=False)
        except TypeError:
            return torch.load(path, map_location=map_location)


def load_phase1_lm(phase1_dir):
    """Rebuild a Phase-1 model from final/{config.json, model.pt}; strict key match."""
    with open(os.path.join(phase1_dir, "config.json")) as f:
        cfg = json.load(f)
    cls, cfg_cls = MODEL_CLASSES[cfg["model_class"]]
    lm = cls(cfg_cls(**cfg["config"]))
    sd = torch_load(os.path.join(phase1_dir, "model.pt"))
    lm.load_state_dict(sd, strict=True)
    return lm, cfg


def kind_of(lm):
    name = type(lm).__name__
    return {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[name]


class Trunk(nn.Module):
    """embed -> selected blocks -> final norm, reusing the Phase-1 submodules."""

    def __init__(self, lm, attn_mode="bidir", layer_ids=None):
        super().__init__()
        assert attn_mode in ("causal", "bidir")
        self.kind = kind_of(lm)
        self.attn_mode = attn_mode
        c = lm.config
        self.window = None
        if self.kind == "gpt2":
            self.embed, self.wpe, blocks, self.norm = lm.wte, lm.wpe, lm.blocks, lm.ln_f
            self.n_positions = c.n_positions
            self.hidden_size = c.n_embd
        elif self.kind == "llama":
            self.embed, self.rotary, blocks, self.norm = lm.embed_tokens, lm.rotary, lm.layers, lm.norm
            self.window = c.sliding_window
            self.hidden_size = c.hidden_size
        else:
            self.embed, blocks, self.norm = lm.word_embeddings, lm.blocks, lm.ln_f
            self.register_buffer("slopes", lm.slopes.detach().clone(), persistent=False)
            self.hidden_size = c.hidden_size
        self.n_layers_original = len(blocks)
        self.layer_ids = list(range(len(blocks))) if layer_ids is None else list(layer_ids)
        assert all(0 <= i < len(blocks) for i in self.layer_ids)
        self.blocks = nn.ModuleList([blocks[i] for i in self.layer_ids])

    # ---------------------------------------------------------------- masks
    def _sdpa_mask(self, T, attention_mask, dev):
        """(mask, is_causal) for M1/M2/M3 blocks."""
        if self.attn_mode == "causal":
            return build_attn_mask(T, 0, attention_mask, self.window, dev)
        no_pad = bool(attention_mask.all())
        win = self.window if (self.window is not None and self.window < T) else None
        if no_pad and win is None:
            return None, False
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = torch.ones(1, 1, T, T, dtype=torch.bool, device=dev)
        if win is not None:
            allowed = allowed & ((q - k).abs() < win)[None, None]
        if not no_pad:
            allowed = allowed & attention_mask[:, None, None, :].bool()
            allowed = allowed | (q == k)[None, None]          # no all-masked rows (pad queries)
        B = attention_mask.size(0)
        return allowed.expand(B, 1, T, T).contiguous(), False

    def _alibi(self, T, attention_mask, dev):
        pos_k = (attention_mask.long().cumsum(-1) - 1).clamp(min=0)
        if self.attn_mode == "causal":                          # identical to BloomStyleLM.forward
            allowed = causal_allowed(T, 0, attention_mask, None, dev)
            return alibi_bias(pos_k[:, -T:], pos_k, self.slopes, allowed)
        dist = (pos_k[:, None, :, None] - pos_k[:, None, None, :]).abs().float()
        bias = -self.slopes.view(1, -1, 1, 1) * dist
        q = torch.arange(T, device=dev)[:, None]
        k = torch.arange(T, device=dev)[None, :]
        allowed = attention_mask[:, None, None, :].bool() | (q == k)[None, None]
        return bias.masked_fill(~allowed, float("-inf"))

    # ---------------------------------------------------------------- forward
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= MAX_POSITIONS, f"sequence of {T} tokens > {MAX_POSITIONS} (Phase-1 context)"
        dev = input_ids.device
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        pos = positions_from_mask(attention_mask, 0, T, B, dev)
        if self.kind == "gpt2":
            pos = pos.clamp(max=self.n_positions - 1)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids) + self.wpe(pos)
            for blk in self.blocks:
                x, _ = blk(x, mask, is_causal, None)
        elif self.kind == "llama":
            cos, sin = self.rotary(pos)
            mask, is_causal = self._sdpa_mask(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, cos, sin, mask, is_causal, None)
        else:
            bias = self._alibi(T, attention_mask, dev)
            x = self.embed(input_ids)
            for blk in self.blocks:
                x, _ = blk(x, bias, None)
        return self.norm(x)

    def norm_modules(self):
        return [m for m in self.modules() if isinstance(m, (nn.LayerNorm, RMSNorm))]


class NERModel(nn.Module):
    """Trunk + dropout + Linear(H, C) [+ CRF]."""

    def __init__(self, trunk, n_labels, head="linear", dropout=0.1):
        super().__init__()
        self.trunk = trunk
        self.n_labels = n_labels
        self.num_tags = 1 + 2 * n_labels
        self.head_type = head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(trunk.hidden_size, self.num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
        self.crf = None
        if head == "crf":
            from ner_crf import CRF
            self.crf = CRF(n_labels)

    def forward(self, input_ids, attention_mask=None, labels=None, class_weights=None):
        h = self.trunk(input_ids, attention_mask)
        logits = self.classifier(self.dropout(h)).float()
        loss = None
        if labels is not None:
            if self.crf is None:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                                       weight=class_weights, ignore_index=-100)
            else:                                   # drop the EOS position, mask = real labels
                lab = labels[:, 1:]
                loss = self.crf.neg_log_likelihood(logits[:, 1:], lab, lab != -100)
        return logits, loss


def build_ner_model(phase1_dir, n_labels, attn_mode="bidir", head="linear", layer_ids=None,
                    dropout=0.1):
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm                                          # unused blocks / lm_head are freed here
    return NERModel(trunk, n_labels, head, dropout), cfg


def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if (p.requires_grad or not trainable_only))


# ---------------------------------------------------------------------------------------------
# trained-model reconstruction (teacher for E6, prediction, ensembles)
# ---------------------------------------------------------------------------------------------
def trainable_state_dict(model, half=False):
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    if half:
        sd = {n: (t.half() if t.is_floating_point() else t) for n, t in sd.items()}
    return sd


def load_trainable_state(model, sd):
    """Load a trainable-only state dict; every key must exist and match a trainable param."""
    params = dict(model.named_parameters())
    missing = [n for n in sd if n not in params]
    if missing:
        raise KeyError(f"checkpoint has unknown parameters: {missing[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            params[n].copy_(t.to(params[n].dtype))
    expected = {n for n, p in params.items() if p.requires_grad}
    absent = expected - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained_model(run_config, best_pt, phase1_dirs, device="cpu"):
    """Recreate the exact module structure of a finished run and load its best.pt."""
    from ner_peft import apply_technique
    mid = run_config["model"]
    p1 = phase1_dirs.get(mid) or run_config.get("phase1_dir")
    if not p1 or not os.path.exists(os.path.join(p1, "model.pt")):
        raise FileNotFoundError(f"Phase-1 weights for {mid} ({MODELS[mid]['arch']}) not found")
    model, _ = build_ner_model(p1, run_config["n_labels"], run_config["attn_mode"],
                               run_config["head"], run_config.get("layer_ids"))
    apply_technique(model, run_config["technique"], run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase6/ner_peft.py
"""
ner_peft.py -- fine-tuning / PEFT techniques, implemented from scratch (no `peft` library).

  full_ft      every trunk parameter + head            (E1; also KD students E6 / X2)
  lorma        LoRMA+  h = (I + s*B*A) W0 x  (+ bias)   (E2, compulsory)
  lora         LoRA    h = W0 x + s*B*A x   (+ bias)    (E3)
  adapters     Houlsby bottleneck after attention-out and MLP-out, + norms (E4)
  partial_ft   top-6 blocks + final norm + head, layer-wise LR decay (E5)
  bitfit       bias terms only (M1/M4)                  (extra X1)

`apply_technique(model, technique, cfg)` freezes everything, inserts the modules, unfreezes
exactly what the method trains, ASSERTS that the set of trainable parameter names equals the
expected set, and returns optimizer parameter groups (no weight decay on 1-D tensors).

LoRMA reference: Bihany, Patel & Modi (2025), "LoRMA: Low-Rank Multiplicative Adaptation for
LLMs", arXiv:2506.07621. We implement the additive-identity rank inflation (LoRMA+): the
frozen output y0 = W0 x is multiplied by (I + s*B*A) with B (d_out x r) = 0 at init, so the
layer starts as the exact frozen layer. The d_out x d_out product is never materialised:
y = y0 + s * (y0 @ A^T) @ B^T. For fused QKV projections each of q, k, v gets its own B, A.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from phase1_models import RMSNorm

# Linear layers adapted by LoRA / LoRMA, per architecture family (paths inside each block).
TARGETS = {
    "gpt2": ["attn.qkv", "attn.o_proj", "mlp.up_proj", "mlp.down_proj"],
    "llama": ["attn.q_proj", "attn.k_proj", "attn.v_proj", "attn.o_proj",
              "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
    "bloom": ["self_attention.query_key_value", "self_attention.o_proj",
              "mlp.up_proj", "mlp.down_proj"],
}
FUSED_QKV = {"gpt2": "attn.qkv", "bloom": "self_attention.query_key_value"}
ADAPTER_SITES = {
    "gpt2": ["attn.o_proj", "mlp.down_proj"],
    "llama": ["attn.o_proj", "mlp.down_proj"],
    "bloom": ["self_attention.o_proj", "mlp.down_proj"],
}


def _get(mod, path):
    for p in path.split("."):
        mod = getattr(mod, p)
    return mod


def _set(mod, path, new):
    parts = path.split(".")
    for p in parts[:-1]:
        mod = getattr(mod, p)
    setattr(mod, parts[-1], new)


# =============================================================================================
# modules
# =============================================================================================
class LoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * B(A(dropout(x)))."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, dropout=0.05):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)),
                                         self.lora_B.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        return self.base.weight + self.scale * (self.lora_B @ self.lora_A).to(self.base.weight.dtype)


class LoRMALinear(nn.Module):
    """LoRMA+ : y = (I + s*B*A) (W0 x) + b, applied separately to each output slice."""

    def __init__(self, base: nn.Linear, r=16, alpha=32, slices=None):
        super().__init__()
        self.base = base
        self.r, self.scale = r, alpha / r
        d_out = base.out_features
        self.slices = slices or [(0, d_out)]
        assert self.slices[0][0] == 0 and self.slices[-1][1] == d_out
        assert all(a[1] == b[0] for a, b in zip(self.slices, self.slices[1:])), "slices must tile"
        self.lorma_A = nn.ParameterList()
        self.lorma_B = nn.ParameterList()
        for (s, e) in self.slices:
            A = nn.Parameter(torch.empty(r, e - s))
            nn.init.kaiming_uniform_(A, a=math.sqrt(5))
            self.lorma_A.append(A)
            self.lorma_B.append(nn.Parameter(torch.zeros(e - s, r)))

    def forward(self, x):
        y0 = F.linear(x, self.base.weight)                  # frozen W0 x, bias added after
        parts = []
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            ys = y0[..., s:e]
            parts.append(ys + self.scale * ((ys @ A.to(ys.dtype).t()) @ B.to(ys.dtype).t()))
        y = parts[0] if len(parts) == 1 else torch.cat(parts, dim=-1)
        if self.base.bias is not None:
            y = y + self.base.bias
        return y

    @torch.no_grad()
    def merged_weight(self):
        W0 = self.base.weight
        W = W0.clone()
        for (s, e), A, B in zip(self.slices, self.lorma_A, self.lorma_B):
            W[s:e] = W0[s:e] + self.scale * (B @ (A @ W0[s:e]))
        return W


class Adapter(nn.Module):
    """Houlsby bottleneck: h + up(gelu(down(h))); up ~ N(0, 1e-3) so it starts near identity."""

    def __init__(self, d, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.down.bias)
        nn.init.normal_(self.up.weight, std=1e-3)
        nn.init.zeros_(self.up.bias)

    def forward(self, h):
        return h + self.up(F.gelu(self.down(h)))


class AdapterWrapped(nn.Module):
    """Wraps a sub-layer's output projection: adapter(base(x)), i.e. BEFORE the residual add."""

    def __init__(self, base: nn.Linear, bottleneck=64):
        super().__init__()
        self.base = base
        self.adapter = Adapter(base.out_features, bottleneck)

    def forward(self, x):
        return self.adapter(self.base(x))


# =============================================================================================
# applying a technique
# =============================================================================================
def _groups(named, lr, wd, tag):
    dec = [p for n, p in named if p.ndim >= 2]
    nodec = [p for n, p in named if p.ndim < 2]
    out = []
    if dec:
        out.append({"params": dec, "lr": lr, "weight_decay": wd, "name": f"{tag}/decay"})
    if nodec:
        out.append({"params": nodec, "lr": lr, "weight_decay": 0.0, "name": f"{tag}/no_decay"})
    return out


def apply_technique(model, technique, cfg, log=print):
    """Mutates `model` in place; returns optimizer param groups."""
    trunk = model.trunk
    kind = trunk.kind
    lr = float(cfg.get("lr", 1e-3))
    head_lr = float(cfg.get("head_lr", 1e-3))
    wd = float(cfg.get("weight_decay", 0.01))
    for p in model.parameters():
        p.requires_grad = False

    head_named = [(n, p) for n, p in model.named_parameters()
                  if n.startswith("classifier.") or n.startswith("crf.")]
    expected = {n for n, _ in head_named}
    groups = []

    if technique in ("full_ft", "kd", "student_hard"):
        named = [(n, p) for n, p in model.named_parameters() if n.startswith("trunk.")]
        expected |= {n for n, _ in named}
        for _, p in named:
            p.requires_grad = True
        groups += _groups(named, lr, wd, "trunk")

    elif technique in ("lora", "lorma"):
        r = int(cfg.get(f"{technique}_r", 16))
        alpha = float(cfg.get(f"{technique}_alpha", 32))
        for bi, blk in enumerate(trunk.blocks):
            for path in TARGETS[kind]:
                base = _get(blk, path)
                assert isinstance(base, nn.Linear), f"{path} is {type(base)}"
                if technique == "lora":
                    new = LoRALinear(base, r, alpha, float(cfg.get("lora_dropout", 0.05)))
                else:
                    slices = None
                    if FUSED_QKV.get(kind) == path:
                        d = trunk.hidden_size
                        assert base.out_features == 3 * d
                        slices = [(0, d), (d, 2 * d), (2 * d, 3 * d)]
                    new = LoRMALinear(base, r, alpha, slices)
                _set(blk, path, new)
        named = [(n, p) for n, p in model.named_parameters()
                 if ".lora_A" in n or ".lora_B" in n or ".lorma_A" in n or ".lorma_B" in n]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, technique)

    elif technique == "adapters":
        bott = int(cfg.get("adapter_bottleneck", 64))
        for blk in trunk.blocks:
            for path in ADAPTER_SITES[kind]:
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
        named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
        norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
        named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "adapters")

    elif technique == "bitfit":
        named = [(n, p) for n, p in model.named_parameters()
                 if n.startswith("trunk.") and n.endswith(".bias")]
        if not named:
            raise ValueError(f"BitFit needs bias terms; the {kind} architecture has none")
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "bitfit")

    elif technique == "partial_ft":
        k = int(cfg.get("partial_top_k", 6))
        decay = float(cfg.get("layer_decay", 0.85))
        L = len(trunk.blocks)
        k = min(k, L)
        for i in range(L - k, L):
            named = [(f"trunk.blocks.{i}.{n}", p) for n, p in trunk.blocks[i].named_parameters()]
            for _, p in named:
                p.requires_grad = True
            expected |= {n for n, _ in named}
            groups += _groups(named, lr * decay ** (L - 1 - i), wd, f"block{i}")
        named = [(f"trunk.norm.{n}", p) for n, p in trunk.norm.named_parameters()]
        for _, p in named:
            p.requires_grad = True
        expected |= {n for n, _ in named}
        groups += _groups(named, lr, wd, "final_norm")
    else:
        raise ValueError(f"unknown technique {technique!r}")

    for _, p in head_named:
        p.requires_grad = True
    groups += _groups(head_named, head_lr, wd, "head")

    actual = {n for n, p in model.named_parameters() if p.requires_grad}
    if actual != expected:
        raise AssertionError(f"trainable-parameter mismatch: unexpected "
                             f"{sorted(actual - expected)[:5]}, missing {sorted(expected - actual)[:5]}")
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    log(f"   [{technique}] trainable {n_tr:,} / {n_all:,} params ({100 * n_tr / n_all:.3f}%) "
        f"in {len(actual)} tensors")
    return groups


def summarize_trainable(model, max_lines=40):
    rows = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
    lines = [f"{n}  {k:,}" for n, k in rows[:max_lines]]
    if len(rows) > max_lines:
        lines.append(f"... and {len(rows) - max_lines} more tensors")
    return lines


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_common.py
"""
cjpe_common.py -- Phase-6 (Abstractive Legal Summarization) spec, registries and small helpers.

Torch-free. Every other cjpe_* module imports its constants from here.

Reused byte-identical from Phases 2/3 (never edited): ner_common.py (file discovery, atomic
JSON, disk guard, Phase-1 model discovery by the `arch` field), ner_models.py (Trunk),
ner_peft.py (LoRA, LoRMA+, adapters, BitFit, partial FT), phase1_models.py, bpe_tokenizer.py.
"""
import csv
import glob
import io
import json
import math
import os
import time
from typing import Dict, Iterator, List, Optional, Tuple

from ner_common import (MODELS, STUDENT_LAYERS, DiskFull, dir_size_gb, disk_guard, file_md5,
                        find_files, find_phase1_models, free_gb, read_json, write_json)

__all__ = ["SPEC", "MODELS", "STUDENT_LAYERS", "EXPERIMENTS", "DEFAULT_LR", "DEFAULT_EPOCHS",
           "MAX_TRAINABLE_PCT", "DEFAULT_MODEL_ORDER", "OUT_ROOT", "CACHE_DIR", "RAW_DIR",
           "CODE_DIR", "EXIT", "JOB_EXIT", "CSV_COLUMNS", "EPOCH_COLUMNS", "default_input_roots",
           "find_files", "find_phase1_models", "read_json", "write_json", "file_md5",
           "disk_guard", "DiskFull", "dir_size_gb", "free_gb", "module_md5s", "log_to",
           "write_csv", "read_csv", "now_str", "student_layers", "iter_out_dirs", "fmt",
           "MODULE_FILES"]

# =============================================================================================
# The brief (CS-781 Capstone, Phase 5 -- Legal Statute Identification). Edit HERE only.
# =============================================================================================
SPEC: Dict = {
    "phase": 6,
    "task": "abstractive_legal_summarization",
    "hf_repo": "Exploration-Lab/CS-781-Capstone",
    "hf_folder": "Phase_6_Summ",
    "files": {"train": "Train/train.json", "test": "Test/test.json"},
    "metric": "bertscore_f1",                # primary; ROUGE-1 / ROUGE-L F1 secondary
    "submission_entry": "prediction.json",   # list format [{"id", "summary"}], zip root
    "submission_zip": "submission.zip",
    "submissions": {"min": 6, "max": 40, "max_per_day": 10},
    "deadline": "2026-11-02 23:59 IST",
    "rules": {"min_experiments": 6, "min_peft_techniques": 5, "compulsory": ["lorma"],
              "min_kd_experiments": 1, "all_models_required": True,
              "foundation_params": [50e6, 300e6]},
}

DEFAULT_MODEL_ORDER = ["M4", "M2", "M3", "M1"]

# The six counted experiments (PDF 12.2.1 list; the technique is the ONLY difference) + extras.
EXPERIMENTS: Dict[str, Dict] = {
    "E1": {"folder": "E1_lorma", "technique": "lorma", "counts": True,
           "title": "LoRMA+ (compulsory)"},
    "E2": {"folder": "E2_lora", "technique": "lora", "counts": True, "title": "LoRA"},
    "E3": {"folder": "E3_adalora", "technique": "adalora", "counts": True, "title": "AdaLoRA"},
    "E4": {"folder": "E4_houlsby", "technique": "adapters", "counts": True,
           "title": "Houlsby adapters (after attention and FFN)"},
    "E5": {"folder": "E5_pfeiffer", "technique": "pfeiffer", "counts": True,
           "title": "Pfeiffer adapters (after FFN only)"},
    "E6": {"folder": "E6_kd", "technique": "kd", "counts": True,
           "title": "Knowledge distillation (full-FT teacher -> layer-pruned students, online "
                    "token-level KD)"},
    "E7": {"folder": "E7_extras", "technique": "extras", "counts": False,
           "title": "Extras: QLoRA, prefix tuning, prompt tuning, long-document and decoding "
                    "ablations"},
    "E8": {"folder": "E8_report", "technique": "report", "counts": False,
           "title": "Report: ALL_RESULTS, upload plan, consensus ensemble, analysis (CPU)"},
}
COUNTED = [e for e, s in EXPERIMENTS.items() if s["counts"]]

DEFAULT_LR: Dict[str, float] = {
    "full_ft": 5e-5, "kd": 2e-4, "student_hard": 2e-4, "lorma": 5e-4, "lora": 5e-4,
    "qlora": 5e-4, "adalora": 1e-3, "adapters": 5e-4, "pfeiffer": 1e-3, "prefix": 3e-3,
    "prompt": 1e-2, "partial_ft": 1e-4, "bitfit": 2e-3,
}
HEAD_LR = 1e-3
DEFAULT_EPOCHS: Dict[str, int] = {
    "full_ft": 3, "kd": 3, "student_hard": 3, "partial_ft": 3, "lorma": 4, "lora": 4,
    "qlora": 4, "adalora": 4, "adapters": 4, "pfeiffer": 4, "prefix": 4, "prompt": 4, "bitfit": 4,
}
MAX_TRAINABLE_PCT: Dict[str, float] = {
    "lorma": 5.0, "lora": 5.0, "adalora": 5.0, "adapters": 5.0, "bitfit": 0.5,
}

# =============================================================================================
# Paths / exit codes
# =============================================================================================
CODE_DIR = os.path.dirname(os.path.abspath(__file__))
OUT_ROOT = "/kaggle/working/phase6_out"
CACHE_DIR = "/kaggle/working/phase6_cache"
RAW_DIR = "/kaggle/working/phase6_raw"
OUT_MARKER = "phase6_out"
EXIT = {"ok": 0, "error": 1, "missing_dependency": 2, "gate_failure": 3,
        "validation_failure": 4}
# exit codes of one training job (cjpe_train.py)
JOB_EXIT = {0: "ok", 1: "error", 2: "does not fit the time budget", 4: "too many NaN steps",
            5: "early-fail watchdog", 6: "critical self-check failed", 7: "OOM at micro-batch 1",
            137: "killed"}

MODULE_FILES = ["bpe_tokenizer.py", "phase1_models.py", "ner_common.py", "ner_models.py",
                "ner_peft.py", "cjpe_common.py", "cjpe_data.py", "cjpe_tok.py", "cjpe_models.py",
                "cjpe_peft.py", "cjpe_eval.py", "cjpe_kd.py", "lsi_ml.py", "cjpe_train.py", "gen_ml.py", "summ_model.py", "summ_data.py",
                "summ_train.py", "cjpe_predict.py",
                "validate_submission.py", "select_best.py", "aggregate.py", "run_experiment.py",
                "selftest.py"]

CSV_COLUMNS = [
    "experiment", "technique", "model_id", "arch", "doc_repr", "chunk_tokens", "summary_budget",
    "max_new", "num_beams", "status", "fail_reason", "selfcheck", "train_docs", "train_seqs",
    "train_fraction", "dev_docs", "dev_gen_docs", "test_docs", "epochs_planned", "epochs_run",
    "best_epoch", "lr", "batch_effective", "score_metric", "dev_score", "dev_bertscore_f1",
    "dev_rouge1", "dev_rouge2", "dev_rougeL", "dev_loss", "dev_ppl", "length_ratio", "cap_words",
    "words", "ref_words", "compression", "repeated_trigram_rate", "novel_bigram_rate",
    "content_support", "number_precision", "citation_precision", "ref_content_recall",
    "empty_outputs", "test_mean_words", "trainable_params", "total_params", "trainable_pct",
    "params_ratio_vs_teacher", "projected_minutes", "train_minutes", "tokens_per_s",
    "gen_tokens_per_s", "ms_per_doc_infer", "peak_mem_gb", "gpu_slot", "attempts", "train_files",
    "leak_guard", "seed", "job", "submission_zip", "started_at", "finished_at"]
EPOCH_COLUMNS = ["experiment", "model_id", "job", "epoch", "step", "train_loss", "dev_loss", "lr",
                 "minutes", "skipped_steps"]


def default_input_roots() -> List[str]:
    """Where attached Kaggle datasets live; PHASE6_INPUT_ROOTS (os.pathsep-separated) overrides."""
    env = os.environ.get("PHASE6_INPUT_ROOTS")
    if env is not None:
        return [p for p in env.split(os.pathsep) if p]
    return ["/kaggle/input"]


def module_md5s(code_dir: str = CODE_DIR) -> Dict[str, Optional[str]]:
    out = {}
    for f in MODULE_FILES:
        p = os.path.join(code_dir, f)
        out[f] = file_md5(p) if os.path.exists(p) else None
    return out


def student_layers(mid: str, phase1_dir: str) -> List[int]:
    """Every second layer (== STUDENT_LAYERS for the real 12/14-layer models)."""
    cfg = read_json(os.path.join(phase1_dir, "config.json"))["config"]
    n = cfg.get("n_layer", cfg.get("num_hidden_layers"))
    ids = list(range(0, n, 2))
    if n in (12, 14):
        assert ids == STUDENT_LAYERS[mid], (ids, STUDENT_LAYERS[mid])
    return ids


def iter_out_dirs(out_root: Optional[str], input_roots=()) -> List[str]:
    """Every phase5_out folder: the working one first, then any inside attached inputs."""
    outs = []
    if out_root and os.path.isdir(out_root):
        outs.append(os.path.abspath(out_root))
    for r in input_roots:
        if r and os.path.isdir(r):
            for d in glob.glob(os.path.join(r, "**", OUT_MARKER), recursive=True):
                if os.path.isdir(d) and os.path.abspath(d) not in outs:
                    outs.append(os.path.abspath(d))
    return outs


def now_str() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")


def log_to(path: Optional[str], msg: str, prefix: str = "") -> None:
    line = f"{prefix}{msg}"
    print(line, flush=True)
    if path:
        try:
            os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
            with open(path, "a", encoding="utf-8") as f:
                f.write(time.strftime("%H:%M:%S ") + line + "\n")
        except OSError:
            pass


def fmt(v) -> str:
    """CSV cell: empty string for missing values, never None / nan."""
    if v is None:
        return ""
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return ""
        return f"{v:.6g}" if abs(v) < 1e5 else f"{v:.1f}"
    return str(v)


def write_csv(path: str, rows: List[Dict], columns: List[str]) -> None:
    """Atomic CSV write (tmp + os.replace); unknown keys are ignored, missing ones are ''."""
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(columns)
    for r in rows:
        w.writerow([fmt(r.get(c)) for c in columns])
    tmp = f"{path}.tmp{os.getpid()}"
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(buf.getvalue())
    os.replace(tmp, path)


def read_csv(path: str) -> List[Dict]:
    if not os.path.exists(path):
        return []
    with open(path, encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_data.py
"""
cjpe_data.py -- Phase-5 data: discovery, Hugging Face fallback download, schema-tolerant loading,
data audit and the leakage guard (sections 2, 3, 4.2, 4.3 of the brief).

Data integrity rules implemented here:
  * training uses train.json ONLY; dev.json only for selection; test.json text only for
    inference (enforced again by cjpe_train at start-up);
  * any train case whose id or text hash also occurs in test is DROPPED from training, any dev
    case that occurs in test is dropped from dev; every count is written to data_audit.json;
  * PROVENANCE.json records md5s of the three files and what each one was used for.
"""
import hashlib
import json
import os
import random
import re
import time
import urllib.error
import urllib.request
from collections import Counter
from typing import Dict, List, Optional, Tuple

from cjpe_common import RAW_DIR, SPEC, file_md5, find_files, log_to, write_json

ID_KEYS = ("id", "case_id", "doc_id", "Id", "ID", "uid", "name")
TEXT_KEYS = ("text", "case_text", "document", "doc", "content", "judgment_text", "raw_text")
LABEL_KEYS = ("labels", "label", "statutes", "statute_ids", "label_ids", "pred", "y", "target")


class DataError(RuntimeError):
    """The data cannot be loaded unambiguously: the pipeline stops and asks (section 13)."""


# =============================================================================================
# Hugging Face token (Kaggle secret) + download
# =============================================================================================
def get_hf_token() -> Optional[str]:
    """Kaggle secret `HF` (or HF_TOKEN / HUGGINGFACE_TOKEN), else the HF_TOKEN env var.
    The token is never printed, logged or written anywhere."""
    try:
        from kaggle_secrets import UserSecretsClient
        us = UserSecretsClient()
        for name in ("HF", "HF_TOKEN", "HUGGINGFACE_TOKEN"):
            try:
                tok = us.get_secret(name)
                if tok:
                    return tok.strip()
            except Exception:
                pass
    except Exception:
        pass
    return (os.environ.get("HF_TOKEN") or "").strip() or None


def _redact(msg: str, token: Optional[str]) -> str:
    return msg.replace(token, "<redacted>") if token else msg


def hf_download(dest_root: str = RAW_DIR, log=print) -> Optional[Dict[str, str]]:
    """Download the three files with urllib (no huggingface_hub). Returns {split: path} or None."""
    token = get_hf_token()
    log(f"   [data] HF token found: {'yes' if token else 'no'}"
        + (f" (length {len(token)})" if token else ""))
    if not token:
        return None
    out = {}
    for split, rel in SPEC["files"].items():
        url = (f"https://huggingface.co/datasets/{SPEC['hf_repo']}/resolve/main/"
               f"{SPEC['hf_folder']}/{rel}")
        dest = os.path.join(dest_root, SPEC["hf_folder"], rel)
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            try:
                with open(dest, encoding="utf-8") as f:
                    json.load(f)
                out[split] = dest
                continue
            except Exception:
                pass
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        last = None
        for attempt in range(3):
            try:
                req = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}",
                                                           "User-Agent": "cs781-phase6"})
                with urllib.request.urlopen(req, timeout=300) as r, open(dest + ".tmp", "wb") as f:
                    while True:
                        b = r.read(1 << 20)
                        if not b:
                            break
                        f.write(b)
                with open(dest + ".tmp", encoding="utf-8") as f:
                    json.load(f)                       # validate before accepting
                os.replace(dest + ".tmp", dest)
                out[split] = dest
                log(f"   [data] downloaded {rel} ({os.path.getsize(dest) / 1e6:.1f} MB)")
                break
            except urllib.error.HTTPError as e:
                last = f"HTTP {e.code}"
                if e.code in (401, 403):
                    log("   [data] HTTP %d: token missing, or this Hugging Face account has not "
                        "accepted the dataset terms" % e.code)
                    return None
            except Exception as e:                      # network / JSON errors: retry
                last = _redact(f"{type(e).__name__}: {e}", token)
            time.sleep(5 * (attempt + 1))
        if split not in out:
            log(f"   [data] download of {rel} failed after 3 attempts ({last})")
            return None
    return out


# =============================================================================================
# discovery
# =============================================================================================
def _peek_records(path: str) -> Optional[Tuple[str, Optional[str], Optional[str]]]:
    """(id_key, text_key, label_key) if `path` looks like a CJPE split, else None."""
    try:
        with open(path, encoding="utf-8") as f:
            obj = json.load(f)
        recs, keyed = _records(obj)
        if not recs:
            return None
        return _detect_keys(recs[:50], keyed)
    except Exception:
        return None


def discover_files(input_roots: List[str], log=print) -> Dict:
    """Find train/dev/test among the attached datasets; fall back to the HF download."""
    found, cands = {}, {}
    for split, rel in SPEC["files"].items():
        name = os.path.basename(rel)
        hits = find_files(name, input_roots, max_hits=200)
        good = []
        for h in hits:
            low = h.lower()
            score = (("phase_4" in low) + ("cjpe" in low) * 2 + (("/test/" in low) if split == "test"
                     else ("/train/" in low)))
            keys = _peek_records(h)
            if keys is not None:
                good.append((score, h))
        good.sort(key=lambda t: (-t[0], t[1]))
        cands[split] = [g[1] for g in good]
        if good:
            found[split] = good[0][1]
    for split, c in cands.items():
        if len(c) > 1:
            log(f"   [data] {len(c)} candidates for {split}: {c[:5]} -> using {found[split]}")
    if len(found) == len(SPEC["files"]):
        return {"source": "kaggle_dataset", "paths": found, "candidates": cands}
    log(f"   [data] attached dataset incomplete (found {sorted(found)}); trying the HF download")
    dl = hf_download(log=log)
    if dl and len(dl) == len(SPEC["files"]):
        return {"source": "huggingface", "paths": dl, "candidates": cands}
    return {"source": None, "paths": found, "candidates": cands}


# =============================================================================================
# schema-tolerant parsing
# =============================================================================================
def _records(obj) -> Tuple[List, bool]:
    """Return (records, keyed). keyed=True when the file was a dict keyed by id."""
    if isinstance(obj, list):
        return obj, False
    if isinstance(obj, dict):
        for k in ("data", "examples", "records", "cases", "rows"):
            if isinstance(obj.get(k), list):
                return obj[k], False
        # dict keyed by id: values are dicts or plain strings
        vals = list(obj.values())
        if vals and all(isinstance(v, (dict, str)) for v in vals[:50]):
            recs = []
            for k, v in obj.items():
                if isinstance(v, str):
                    recs.append({"id": k, "text": v})
                else:
                    r = dict(v)
                    r.setdefault("id", k)
                    recs.append(r)
            return recs, True
    raise DataError(f"unrecognised JSON container: {type(obj).__name__}")


def _detect_keys(recs: List, keyed: bool) -> Tuple[str, Optional[str], Optional[str]]:
    if not all(isinstance(r, dict) for r in recs):
        raise DataError("records are not JSON objects")
    keys = Counter(k for r in recs for k in r.keys())
    n = len(recs)
    idk = next((k for k in ID_KEYS if keys.get(k, 0) == n), None)
    tk = next((k for k in TEXT_KEYS if keys.get(k, 0) == n), None)
    if tk is None:                       # fall back: the single long-string field
        strf = [k for k in keys if keys[k] == n and all(isinstance(r[k], str) for r in recs)]
        strf = [k for k in strf if k not in ID_KEYS]
        if len(strf) == 1:
            tk = strf[0]
    lk = next((k for k in LABEL_KEYS if keys.get(k, 0) == n), None)
    if idk is None or tk is None:
        raise DataError(f"cannot find id/text keys among {dict(keys)}")
    return idk, tk, lk


def _to_label(v) -> List[int]:
    """Statute ids as a sorted unique list of ints (accepts int, numeric str, list, "1,7")."""
    if v is None:
        return []
    if isinstance(v, bool):
        raise DataError(f"label value {v!r} is a bool, not a statute id")
    if isinstance(v, (int, float)) and float(v) == int(v):
        return [int(v)]
    if isinstance(v, str):
        parts = [x for x in v.replace(";", ",").replace(" ", ",").split(",") if x.strip()]
        try:
            return sorted({int(float(x)) for x in parts})
        except ValueError:
            raise DataError(f"label value {v!r} is not a list of statute ids")
    if isinstance(v, (list, tuple)):
        out = set()
        for x in v:
            out.update(_to_label(x))
        return sorted(out)
    raise DataError(f"label value {v!r} is not a list of statute ids")


def load_split(path: str, split: str, log=print) -> Tuple[List[Dict], Dict]:
    """[{id, text, label|None}] + schema description. Raises DataError when ambiguous."""
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    recs, keyed = _records(obj)
    if not recs:
        raise DataError(f"{split}: no records in {path}")
    idk, tk, lk = _detect_keys(recs, keyed)
    if split != "test" and lk is None:
        raise DataError(f"{split}: no label key among {sorted(recs[0].keys())}")
    out = []
    for r in recs:
        t = r[tk]
        if isinstance(t, list):                          # list of sentences/paragraphs
            t = "\n".join(str(x) for x in t)
        if not isinstance(t, str):
            raise DataError(f"{split}: text field {tk!r} is {type(t).__name__}")
        lab = _to_label(r[lk]) if (lk is not None and split != "test") else None
        out.append({"id": str(r[idk]), "text": t, "label": lab})
    schema = {"container": "dict keyed by id" if keyed else "list of objects",
              "id_key": idk, "text_key": tk, "label_key": lk, "n": len(out),
              "fields": sorted(recs[0].keys()), "md5": file_md5(path), "path": path}
    return out, schema


# =============================================================================================
# audit + leak guard
# =============================================================================================
def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", "replace")).hexdigest()


def norm_hash(s: str) -> str:
    return sha1(re.sub(r"\s+", " ", s).strip().lower())


def _pct(xs: List[float], q: float) -> float:
    if not xs:
        return 0.0
    xs = sorted(xs)
    return float(xs[min(len(xs) - 1, int(q * (len(xs) - 1) + 0.5))])


def length_stats(vals: List[float]) -> Dict:
    if not vals:
        return {}
    return {"mean": sum(vals) / len(vals), "median": _pct(vals, 0.5), "p90": _pct(vals, 0.9),
            "p99": _pct(vals, 0.99), "max": max(vals), "min": min(vals)}


def leak_guard(data: Dict[str, List[Dict]], log=print) -> Dict:
    """Drop train/dev cases that also occur in test (by id or raw / normalised text hash)."""
    h = {s: [(d["id"], sha1(d["text"]), norm_hash(d["text"])) for d in data[s]] for s in data}
    sets = {s: {"id": {a for a, _, _ in v}, "sha1": {b for _, b, _ in v},
                "norm": {c for _, _, c in v}} for s, v in h.items()}
    rep = {}
    for a, b in (("train", "test"), ("dev", "test"), ("train", "dev")):
        rep[f"{a}_x_{b}"] = {k: len(sets[a][k] & sets[b][k]) for k in ("id", "sha1", "norm")}
    dropped = {}
    for s in ("train", "dev"):
        keep, drop = [], []
        for d, (i, x, y) in zip(data[s], h[s]):
            if i in sets["test"]["id"] or x in sets["test"]["sha1"] or y in sets["test"]["norm"]:
                drop.append(i)
            else:
                keep.append(d)
        dropped[s] = drop
        if drop:
            log(f"   !!! [leak-guard] {len(drop)} {s} case(s) also occur in test -> DROPPED from "
                f"{s} (ids {drop[:5]}...)")
        data[s] = keep
    rep["dropped_from_train"] = len(dropped["train"])
    rep["dropped_from_dev"] = len(dropped["dev"])
    rep["dropped_ids"] = {k: v[:200] for k, v in dropped.items()}
    n_train = len(data["train"]) + len(dropped["train"])
    rep["train_test_overlap_pct"] = 100.0 * len(dropped["train"]) / max(1, n_train)
    rep["status"] = "passed"
    return rep


def basic_audit(data: Dict[str, List[Dict]], schemas: Dict) -> Dict:
    a = {"schema": schemas, "splits": {}}
    for s, docs in data.items():
        ids = [d["id"] for d in docs]
        texts = [d["text"] for d in docs]
        e = {"n": len(docs), "empty_text": sum(1 for t in texts if not t.strip()),
             "duplicate_ids": len(ids) - len(set(ids)),
             "duplicate_texts": len(texts) - len({sha1(t) for t in texts}),
             "chars": length_stats([float(len(t)) for t in texts]),
             "words": length_stats([float(len(t.split())) for t in texts])}
        if s != "test":
            c = Counter(x for d in docs for x in (d["label"] or []))
            nl = [len(d["label"] or []) for d in docs]
            freq = sorted(c.values())
            e["n_labels"] = len(c)
            e["labels_per_doc"] = length_stats([float(x) for x in nl])
            e["docs_without_labels"] = sum(1 for x in nl if x == 0)
            e["label_freq_quantiles"] = {q: _pct([float(x) for x in freq], q)
                                         for q in (0.1, 0.25, 0.5, 0.75, 0.9)} if freq else {}
            e["pct_labels_lt5"] = 100.0 * sum(1 for x in freq if x < 5) / max(1, len(freq))
            e["top_labels"] = c.most_common(10)
        a["splits"][s] = e
    return a


def token_audit(data: Dict[str, List[Dict]], tok, sample: int = 2000, seed: int = 42,
                n_proc: int = 1) -> Dict:
    """BPE length statistics on a seeded sample (full tokenisation); chars-per-token ratio."""
    from cjpe_tok import count_tokens_many
    rng = random.Random(seed)
    out = {}
    pool = []
    for s, docs in data.items():
        idx = list(range(len(docs)))
        rng.shuffle(idx)
        pool += [(s, i) for i in idx[:max(1, sample // 3)]]
    counts = count_tokens_many([data[s][i]["text"] for s, i in pool], tok, n_proc)
    by = {}
    for (s, i), n in zip(pool, counts):
        by.setdefault(s, []).append((n, len(data[s][i]["text"]), data[s][i]["label"]))
    tot_c = sum(c for v in by.values() for _, c, _ in v)
    tot_t = sum(n for v in by.values() for n, _, _ in v)
    ratio = tot_c / max(1, tot_t)
    out["chars_per_token"] = ratio
    out["sample_size"] = len(pool)
    for s, v in by.items():
        toks = [float(n) for n, _, _ in v]
        e = {"tokens": length_stats(toks)}
        for thr in (512, 1024, 2048, 4096, 8192):
            e[f"pct_over_{thr}"] = 100.0 * sum(1 for t in toks if t > thr) / max(1, len(toks))
        if s != "test":
            buckets = {}
            for n, _, lab in v:
                b = bucket_of(n)
                buckets.setdefault(b, []).append(lab)
            e["labels_per_doc_by_length"] = {b: {"n": len(l), "mean_labels":
                                                 sum(len(x or []) for x in l) / len(l)}
                                         for b, l in sorted(buckets.items())}
        out[s] = e
    return out


BUCKETS = [(512, "<=512"), (1024, "<=1024"), (2048, "<=2048"), (4096, "<=4096"),
           (10 ** 12, ">4096")]


def bucket_of(n_tokens: float) -> str:
    for thr, name in BUCKETS:
        if n_tokens <= thr:
            return name
    return BUCKETS[-1][1]


def provenance(schemas: Dict, leak: Dict, n_train_used: int, source: str) -> Dict:
    return {
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "data_source": source,
        "md5": {s: schemas[s]["md5"] for s in schemas},
        "paths": {s: schemas[s]["path"] for s in schemas},
        "used_for": {"train.json": "training (the ONLY file whose labels are used for gradient "
                                   "updates)",
                     "dev.json": "model selection, early stopping, threshold tuning and local "
                                 "evaluation only (never trained on, no train+dev refit)",
                     "test.json": "inference only (text); no test-time training, no pseudo "
                                  "labels, nothing tuned on test predictions"},
        "train_cases_used": n_train_used,
        "train_cases_dropped_by_leak_guard": leak.get("dropped_from_train", 0),
        "dev_cases_dropped_by_leak_guard": leak.get("dropped_from_dev", 0),
        "leak_guard": leak.get("status", "unknown"),
    }


def print_audit(a: Dict, log=print) -> None:
    log("   [audit] split   n      empty dup_id dup_text  pos_rate  chars(median)")
    for s, e in a["splits"].items():
        pr = e.get("positive_rate")
        log(f"   [audit] {s:<6} {e['n']:>6} {e['empty_text']:>6} {e['duplicate_ids']:>6} "
            f"{e['duplicate_texts']:>8}  {'' if pr is None else f'{pr:.3f}':>8}  "
            f"{e['chars'].get('median', 0):,.0f}")
    t = a.get("tokens")
    if t:
        log(f"   [audit] chars/token {t['chars_per_token']:.2f} (sample of {t['sample_size']})")
        for s in ("train", "dev", "test"):
            if s in t:
                e = t[s]
                log(f"   [audit] {s} tokens: median {e['tokens'].get('median', 0):,.0f}, p90 "
                    f"{e['tokens'].get('p90', 0):,.0f}, max {e['tokens'].get('max', 0):,.0f}; "
                    f">512 {e['pct_over_512']:.0f}%  >1024 {e['pct_over_1024']:.0f}%  "
                    f">2048 {e['pct_over_2048']:.0f}%  >4096 {e['pct_over_4096']:.0f}%")
    lg = a.get("leak_guard")
    if lg:
        log(f"   [leak-guard] train x test {lg['train_x_test']}, dev x test {lg['dev_x_test']}, "
            f"train x dev {lg['train_x_dev']}; dropped train {lg['dropped_from_train']}, "
            f"dev {lg['dropped_from_dev']}")


def print_examples(data: Dict[str, List[Dict]], log=print) -> None:
    for s, docs in data.items():
        if docs:
            d = docs[0]
            log(f"   [example] {s}: id={d['id']!r} label={d['label']!r} text={d['text'][:300]!r}")


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_tok.py
"""
cjpe_tok.py -- fast head/tail tokenisation, the token cache and the long-document
representations (section 5 of the brief).

Why slices: the Phase-1 BPE is pure Python. A judgment can be 50k+ tokens but every
representation we train on only needs the first and/or last few 511-token chunks, so we cut
character slices at *token boundaries* and tokenise only those.

Token-boundary rule (proved in the docstring of `cut_head` and tested in selftest): the
pre-tokeniser regex never lets a non-whitespace run continue into whitespace, so position p is
a pre-token boundary of the FULL text whenever text[p] is whitespace and text[p-1] is not. The
regex only looks forward, so tokenising text[:p] gives exactly the full-text tokens before p,
and tokenising text[p:] gives exactly the full-text tokens from p on.

Representations (`doc_repr`), each chunk = [EOS] + <= 511 body tokens (<= 512 total):
  hier     floor(K/2) first chunks + ceil(K/2) last chunks of the document, in order (K=1 ->
           the LAST chunk: the court's conclusion is at the end; Malik et al. 2021 (ILDC) found
           the last 512 tokens the most informative single window)
  first_k  first K chunks          last_k   last K chunks
  headtail ONE sequence of <= 1024 tokens: [EOS] first 383 [EOS] last 639
  sections first chunk + 2 chunks starting at the last "conclusion-like" heading in the last
           70% of the text (ORDER / CONCLUSION / HELD / RESULT / "for the foregoing reasons"
           ...), falling back to `hier` when there is none
  full     consecutive chunks from the start, up to 16 (8176 tokens; ablation only)
  spread   (Phase 5) K chunks evenly spaced over the cached head (up to 16 chunks) + the last
           chunk of the document, in order
  cite     (Phase 5) first chunk + 2 chunks starting at the densest window of statute citations
           ("section 302", "Article 14", "I.P.C.", "Act, 1956" ...) + last chunk; falls back to
           `hier` when the document has no citation
"""
import hashlib
import json
import os
import pickle
import re
import time
from multiprocessing import get_context
from typing import Dict, List, Optional

import numpy as np

from cjpe_common import file_md5, write_json

BODY = 511                     # body tokens per chunk; + EOS = 512
EOS_ID, PAD_ID = 0, 1
HT_HEAD, HT_TAIL = 383, 639    # headtail: 1 + 383 + 1 + 639 = 1024
CACHE_VERSION = 3
CITE_RE = re.compile(
    r"(?i)\b(?:section|sec\.|s\.|sections|article|art\.|rule|order)\s*\d+[a-z]?\b"
    r"|\b(?:i\.?\s?p\.?\s?c|cr\.?\s?p\.?\s?c|c\.?\s?p\.?\s?c|constitution)\b"
    r"|\bact,?\s*(?:18|19|20)\d\d\b")
SECTION_RE = re.compile(
    r"(?im)^\s*(?:ORDER|CONCLUSION|CONCLUSIONS|HELD|RESULT|DECISION|OPERATIVE PORTION)\b"
    r"|for the (?:foregoing|aforesaid|above|reasons) (?:reasons|stated|discussed)"
    r"|in the (?:result|circumstances)\b|we (?:accordingly )?(?:allow|dismiss|set aside)\b")

_TOK = None                    # per-process tokenizer (pool workers)


# =============================================================================================
# boundary-safe slicing
# =============================================================================================
def _ws(c: str) -> bool:
    return c.isspace()


def cut_head(text: str, j: int) -> int:
    """Smallest p >= j that is a safe boundary (text[p] whitespace, text[p-1] not); len(text)
    if none. text[:p] then tokenises exactly like the first tokens of the full text."""
    n = len(text)
    p = max(1, j)
    while p < n and not (_ws(text[p]) and not _ws(text[p - 1])):
        p += 1
    return min(p, n)


def cut_tail(text: str, i: int) -> int:
    """Largest p <= i that is a safe boundary; 0 if none (then the whole text is used)."""
    p = min(i, len(text) - 1)
    while p > 0 and not (_ws(text[p]) and not _ws(text[p - 1])):
        p -= 1
    return max(p, 0)


def _init_worker(bpe_dir: str) -> None:
    global _TOK
    from bpe_tokenizer import BPETokenizer
    _TOK = BPETokenizer.from_pretrained(bpe_dir)


def _encode(text: str) -> List[int]:
    return _TOK.encode(text)


def encode_doc(text: str, need_head: int, need_tail: int, want_sections: bool = False,
               need_sect: int = 2 * BODY, tok=None) -> Dict:
    """Head/tail token ids of one document; exact token count when the doc was fully encoded."""
    enc = (tok or _TOK).encode
    L = len(text)
    need = need_head + need_tail
    out = {"n_chars": L, "sect": None}
    full_thr = need * 9                              # short documents: encode everything
    if L <= full_thr:
        ids = enc(text)
        out.update(complete=True, head=np.asarray(ids, np.uint16), tail=None, n_tokens=len(ids))
    else:
        hc = cut_head(text, need_head * 6)
        while True:
            head = enc(text[:hc])
            if len(head) >= need_head or hc >= L:
                break
            hc = cut_head(text, hc * 2)
        tc = cut_tail(text, L - need_tail * 6)
        while True:
            tail = enc(text[tc:])
            if len(tail) >= need_tail or tc <= 0:
                break
            tc = cut_tail(text, L - (L - tc) * 2)
        if hc >= tc:                                 # slices meet: encode the whole document
            ids = enc(text)
            out.update(complete=True, head=np.asarray(ids, np.uint16), tail=None,
                       n_tokens=len(ids))
        else:
            seen_chars = hc + (L - tc)
            est = (len(head) + len(tail)) / max(1, seen_chars) * L
            out.update(complete=False, head=np.asarray(head[:need_head], np.uint16),
                       tail=np.asarray(tail[-need_tail:] if need_tail else [], np.uint16),
                       n_tokens=int(round(est)))
    if want_sections:
        pos = None
        for m in SECTION_RE.finditer(text):
            if m.start() >= 0.3 * L:
                pos = m.start()
        if pos is not None:
            s = cut_tail(text, pos) if pos > 0 else 0
            e = cut_head(text, s + need_sect * 6)
            ids = enc(text[s:e])
            out["sect"] = np.asarray(ids[:need_sect], np.uint16)
        # Phase 5: densest window of statute citations (window = ~2 chunks of characters)
        pos = [m.start() for m in CITE_RE.finditer(text)]
        out["cite"] = None
        if pos:
            win = need_sect * 5
            best, bi, j = 0, 0, 0
            for i in range(len(pos)):
                while pos[i] - pos[j] > win:
                    j += 1
                if i - j + 1 > best:
                    best, bi = i - j + 1, j
            s = cut_tail(text, max(0, pos[bi] - 200)) if pos[bi] > 200 else 0
            e = cut_head(text, s + need_sect * 6)
            ids = enc(text[s:e])
            out["cite"] = np.asarray(ids[:need_sect], np.uint16)
            out["n_cites"] = len(pos)
    return out


def _encode_doc_star(args):
    return encode_doc(*args)


def _count(text: str) -> int:
    return len(_TOK.encode(text))


def _pool(bpe_dir: str, n_proc: int):
    return get_context("fork").Pool(n_proc, initializer=_init_worker, initargs=(bpe_dir,))


def count_tokens_many(texts: List[str], tok, n_proc: int = 1) -> List[int]:
    if n_proc <= 1 or len(texts) < 50 or not getattr(tok, "_bpe_dir", None):
        return [len(tok.encode(t)) for t in texts]
    with _pool(tok._bpe_dir, n_proc) as p:
        return p.map(_count, texts, chunksize=8)


# =============================================================================================
# the cache
# =============================================================================================
def cache_key(file_md5s: Dict[str, str], tok_md5: str) -> str:
    """Base key: data files + tokenizer (+ CACHE_VERSION). The head/tail sizes and the sections
    flag are part of the FILE name, so a larger cache (e.g. from 00) serves smaller needs."""
    s = json.dumps({"v": CACHE_VERSION, "files": file_md5s, "tok": tok_md5}, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:16]


def cache_name(key: str, need_head: int, need_tail: int, sections: bool) -> str:
    return f"cache_{key}_h{need_head}_t{need_tail}_s{int(bool(sections))}.pkl"


def build_cache(data: Dict[str, List[Dict]], bpe_dir: str, out_dir: str, key: str,
                need_head: int, need_tail: int, sections: bool, n_proc: int, log=print) -> str:
    """Tokenise every document once; writes <out_dir>/cache_<key>.pkl + cache_meta.json."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, cache_name(key, need_head, need_tail, sections))
    t0 = time.time()
    res = {}
    for split, docs in data.items():
        args = [(d["text"], need_head, need_tail, sections) for d in docs]
        if n_proc > 1 and len(args) >= 64:
            with _pool(bpe_dir, n_proc) as p:
                enc = p.map(_encode_doc_star, args, chunksize=16)
        else:
            _init_worker(bpe_dir)
            enc = [_encode_doc_star(a) for a in args]
        rows = []
        for d, e in zip(docs, enc):
            e["id"], e["label"] = d["id"], d["label"]
            rows.append(e)
        res[split] = rows
        log(f"   [cache] {split}: {len(rows)} docs tokenised ({time.time() - t0:.0f}s so far)")
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(res, f, protocol=4)
    os.replace(tmp, path)
    write_json(os.path.join(out_dir, "cache_meta.json"),
               {"key": key, "file": os.path.basename(path), "need_head": need_head,
                "need_tail": need_tail, "sections": sections, "seconds": time.time() - t0,
                "n": {s: len(v) for s, v in res.items()}})
    return path


def find_cache(key: str, need_head: int, need_tail: int, sections: bool,
               roots: List[str]) -> Optional[str]:
    """Any cache of the same data + tokenizer that holds at least the needed tokens."""
    pat = re.compile(rf"^cache_{re.escape(key)}_h(\d+)_t(\d+)_s([01])\.pkl$")
    best = None
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        for dp, _dn, fns in os.walk(r):
            for fn in fns:
                m = pat.match(fn)
                if not m:
                    continue
                h, t, sc = int(m.group(1)), int(m.group(2)), m.group(3) == "1"
                if h >= need_head and t >= need_tail and (sc or not sections):
                    size = h + t
                    if best is None or size < best[0]:
                        best = (size, os.path.join(dp, fn))
    return best[1] if best else None


def load_cache(path: str) -> Dict[str, List[Dict]]:
    with open(path, "rb") as f:
        return pickle.load(f)


# =============================================================================================
# representations
# =============================================================================================
def _with_eos(a: np.ndarray) -> np.ndarray:
    return np.concatenate([np.zeros(1, np.int64), a.astype(np.int64)])


def _first_chunks(ids: np.ndarray, n: int) -> List[np.ndarray]:
    return [ids[i * BODY:(i + 1) * BODY] for i in range(n) if len(ids[i * BODY:(i + 1) * BODY])]


def _last_chunks(ids: np.ndarray, n: int) -> List[np.ndarray]:
    out = []
    L = len(ids)
    for j in range(n, 0, -1):                  # in document order
        s, e = max(0, L - j * BODY), L - (j - 1) * BODY
        if e > s:
            out.append(ids[s:e])
    return out


def doc_chunks(e: Dict, repr_: str, K: int) -> List[np.ndarray]:
    """The list of model inputs (each starts with EOS, <= 512 tokens; headtail <= 1024)."""
    head = e["head"]
    tail = e["tail"] if e["tail"] is not None else head
    complete = bool(e["complete"])
    n_tok = len(head) if complete else None
    if repr_ == "headtail":
        if complete and n_tok <= 1023:
            return [_with_eos(head)]
        h = head[:HT_HEAD]
        t = tail[-HT_TAIL:]
        return [np.concatenate([_with_eos(h), _with_eos(t)])]
    if repr_ == "cite" and e.get("cite") is not None and len(e["cite"]):
        parts = _first_chunks(head, 1) + _first_chunks(e["cite"], 2) + _last_chunks(tail, 1)
        return [_with_eos(p) for p in parts if len(p)]
    if repr_ == "spread":
        n_all = -(-len(head) // BODY)
        if complete and n_all <= K:
            return [_with_eos(p) for p in _first_chunks(head, n_all)]
        avail = _first_chunks(head, min(n_all, 16))
        whole = complete and n_all <= 16             # avail already ends at the document end
        k = max(1, min(K if whole else K - 1, len(avail)))
        pick = sorted({int(round(i * (len(avail) - 1) / max(1, k - 1))) for i in range(k)})
        parts = [avail[i] for i in pick] + ([] if whole else _last_chunks(tail, 1))
        return [_with_eos(p) for p in parts if len(p)]
    if repr_ == "sections" and e.get("sect") is not None and len(e["sect"]):
        parts = _first_chunks(head, 1) + _first_chunks(e["sect"], 2)
        return [_with_eos(p) for p in parts]
    if repr_ in ("hier", "sections", "cite"):
        a, b = K // 2, K - K // 2
    elif repr_ == "first_k":
        a, b = K, 0
    elif repr_ == "last_k":
        a, b = 0, K
    elif repr_ == "full":
        return [_with_eos(p) for p in _first_chunks(head, K)]
    else:
        raise ValueError(f"unknown doc_repr {repr_!r}")
    if complete:
        ids = head
        n_all = -(-len(ids) // BODY)
        if n_all <= a + b:                       # short document: every chunk, in order
            return [_with_eos(p) for p in _first_chunks(ids, n_all)]
        parts = _first_chunks(ids, a) + _last_chunks(ids, b)
    else:
        parts = _first_chunks(head, a) + _last_chunks(tail, b)
    return [_with_eos(p) for p in parts if len(p)]


def needs_for(repr_: str, K: int):
    """(need_head, need_tail) tokens the cache must hold for a representation."""
    if repr_ == "full":
        return K * BODY, 0
    if repr_ == "headtail":
        return 2 * BODY, 2 * BODY
    if repr_ == "first_k":
        return K * BODY, BODY
    if repr_ == "last_k":
        return BODY, K * BODY
    a, b = K // 2, K - K // 2
    return max(a, 1) * BODY, max(b, 1) * BODY


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_models.py
"""
cjpe_models.py -- the document classifier (section 6 of the brief).

    chunks of one document --Phase-1 Trunk (shared, reused submodules)--> hidden states
    -> masked mean over each chunk's tokens (fp32)            = chunk vectors c_1..c_n
    -> learned attention pooling  a_i = softmax_i(w . tanh(W c_i)) over the REAL chunks
    -> dropout 0.1 -> Linear(H, 2)                              = 2 logits (fp32)

All new parameters live in `model.classifier` so that ner_peft.apply_technique (which trains
every `classifier.*` parameter as "the head") works unchanged for every technique.
Padding chunks never reach the trunk: only real chunks are encoded, then scattered into a
(B, n_max, H) grid whose missing slots get exactly zero attention weight.
"""
import json
import os
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_models import Trunk, count_params, load_phase1_lm, torch_load

PAD_ID = 1


class DocHead(nn.Module):
    def __init__(self, d: int, n_classes: int = 2, dropout: float = 0.1, pool_dim: int = 256):
        super().__init__()
        self.att_proj = nn.Linear(d, pool_dim)
        self.att_vec = nn.Linear(pool_dim, 1, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(d, n_classes)
        for lin in (self.att_proj, self.att_vec, self.out):
            nn.init.normal_(lin.weight, std=0.02)
            if lin.bias is not None:
                nn.init.zeros_(lin.bias)

    def attention(self, c: torch.Tensor, cmask: torch.Tensor) -> torch.Tensor:
        """c (B, n, H) fp32, cmask (B, n) bool -> weights (B, n); masked slots exactly 0."""
        s = self.att_vec(torch.tanh(self.att_proj(c))).squeeze(-1)
        s = s.masked_fill(~cmask, float("-inf"))
        w = torch.softmax(s, dim=-1)
        return torch.where(cmask, w, torch.zeros_like(w))

    def forward(self, c, cmask):
        w = self.attention(c, cmask)
        doc = (w.unsqueeze(-1) * c).sum(1)
        return self.out(self.dropout(doc)), w


class DocClassifier(nn.Module):
    def __init__(self, trunk: Trunk, n_classes: int = 2, dropout: float = 0.1):
        super().__init__()
        self.trunk = trunk
        self.classifier = DocHead(trunk.hidden_size, n_classes, dropout)

    def chunk_vectors(self, input_ids, attention_mask):
        h = self.trunk(input_ids, attention_mask)                      # (C, T, H)
        m = attention_mask.unsqueeze(-1).float()
        return (h.float() * m).sum(1) / m.sum(1).clamp(min=1.0)        # (C, H) fp32

    def forward(self, input_ids, attention_mask, doc_index, slot, n_docs: int, n_slots: int):
        v = self.chunk_vectors(input_ids, attention_mask)
        H = v.shape[-1]
        grid = torch.zeros(n_docs, n_slots, H, device=v.device, dtype=v.dtype)
        cmask = torch.zeros(n_docs, n_slots, device=v.device, dtype=torch.bool)
        grid[doc_index, slot] = v
        cmask[doc_index, slot] = True
        # the head runs in fp32 even under autocast (section 6: logits and loss in fp32)
        with torch.autocast(device_type=v.device.type, enabled=False):
            logits, w = self.classifier(grid.float(), cmask)
        return logits.float(), w


def collate_docs(chunk_lists: List[List[np.ndarray]], device=None) -> Dict:
    """Flatten the chunks of several documents into one right-padded (C, T) batch."""
    flat, doc_index, slot = [], [], []
    for d, chunks in enumerate(chunk_lists):
        for s, c in enumerate(chunks):
            flat.append(c)
            doc_index.append(d)
            slot.append(s)
    T = max(len(c) for c in flat)
    assert T <= 1024, f"sequence of {T} tokens > 1024"
    ids = np.full((len(flat), T), PAD_ID, np.int64)
    mask = np.zeros((len(flat), T), np.int64)
    for i, c in enumerate(flat):
        ids[i, :len(c)] = c
        mask[i, :len(c)] = 1
    b = {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(mask),
         "doc_index": torch.tensor(doc_index, dtype=torch.long),
         "slot": torch.tensor(slot, dtype=torch.long),
         "n_docs": len(chunk_lists), "n_slots": max(len(c) for c in chunk_lists),
         "n_tokens": int(mask.sum())}
    if device is not None:
        for k in ("input_ids", "attention_mask", "doc_index", "slot"):
            b[k] = b[k].to(device, non_blocking=True)
    return b


def forward_batch(model: DocClassifier, b: Dict):
    return model(b["input_ids"], b["attention_mask"], b["doc_index"], b["slot"], b["n_docs"],
                 b["n_slots"])


def build_doc_model(phase1_dir: str, attn_mode: str = "bidir", layer_ids=None,
                    dropout: float = 0.1, n_labels: int = 2, bias_init=None):
    """Phase 5: Linear(H, n_labels) multi-label head; bias initialised to the label log-odds
    prior (rare labels start near 0 probability: stable BCE)."""
    lm, cfg = load_phase1_lm(phase1_dir)
    trunk = Trunk(lm, attn_mode, layer_ids)
    del lm
    model = DocClassifier(trunk, int(n_labels), dropout)
    if bias_init is not None:
        with torch.no_grad():
            model.classifier.out.bias.copy_(torch.as_tensor(np.asarray(bias_init, np.float32)))
    return model, cfg


def model_size(model: nn.Module) -> Dict[str, float]:
    tot = count_params(model)
    tr = count_params(model, trainable_only=True)
    return {"total_params": tot, "trainable_params": tr,
            "trainable_pct": 100.0 * tr / max(1, tot)}


def device_report(model: nn.Module) -> Dict[str, int]:
    out: Dict[str, int] = {}
    for p in list(model.parameters()) + list(model.buffers()):
        out[str(p.device)] = out.get(str(p.device), 0) + 1
    return out


def assert_one_device(model: nn.Module, device) -> None:
    """Lesson L1: after .to(device) EVERY parameter and buffer must be on that device."""
    want = torch.device(device)
    bad = [n for n, p in list(model.named_parameters()) + list(model.named_buffers())
           if p.device.type != want.type or (want.index is not None and p.device.index not in
                                             (None, want.index))]
    if bad:
        raise AssertionError(f"{len(bad)} tensors not on {want}: {bad[:5]}")


# ---------------------------------------------------------------------------------------------
# checkpoints of trained models
# ---------------------------------------------------------------------------------------------
STATE_BUFFER_TAGS = (".ada_mask", ".ada_ipt_", ".ada_unc_")


def trainable_state_dict(model: nn.Module, half: bool = False) -> Dict[str, torch.Tensor]:
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    for n, b in model.named_buffers():
        if any(t in n for t in STATE_BUFFER_TAGS):
            sd[n] = b.detach().cpu()
    if half:
        sd = {n: (t.half() if t.is_floating_point() and ".ada_mask" not in n else t)
              for n, t in sd.items()}
    return sd


def load_trainable_state(model: nn.Module, sd: Dict[str, torch.Tensor]) -> None:
    params = dict(model.named_parameters())
    bufs = dict(model.named_buffers())
    unknown = [n for n in sd if n not in params and n not in bufs]
    if unknown:
        raise KeyError(f"checkpoint has unknown tensors: {unknown[:5]}")
    with torch.no_grad():
        for n, t in sd.items():
            dst = params.get(n, bufs.get(n))
            dst.copy_(t.to(dst.dtype))
    absent = {n for n, p in params.items() if p.requires_grad} - set(sd)
    if absent:
        raise KeyError(f"checkpoint lacks trainable parameters: {sorted(absent)[:5]}")


def rebuild_trained(run_config: Dict, best_pt: str, phase1_dir: Optional[str] = None,
                    device="cpu"):
    """Recreate a finished run's module structure and load its best.pt (teacher reuse, E8)."""
    from cjpe_peft import apply_technique
    p1 = phase1_dir or run_config["phase1_dir"]
    model, _ = build_doc_model(p1, run_config["attn_mode"], run_config.get("layer_ids"),
                               n_labels=len(run_config["label_ids"]))
    tech = "full_ft" if run_config["technique"] in ("kd", "student_hard") else \
        run_config["technique"]
    apply_technique(model, tech, run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_peft.py
"""
cjpe_peft.py -- fine-tuning techniques for the document classifier (pure PyTorch, no `peft`).

  full_ft / kd / student_hard   every trunk parameter + head       (ner_peft, Phase 2)
  lorma      LoRMA+ y = (I + s*B*A) W0 x, rank 16, alpha 32       (ner_peft, Phase 2; compulsory)
             Bihany, Patel & Modi (2025), arXiv:2506.07621
  lora       LoRA r=16, alpha 32                                  (ner_peft; extra E7)
  adapters   Houlsby bottleneck 64 after attention-out and MLP-out + all norms (ner_peft)
  partial_ft top-6 blocks + final norm + head, layer-wise LR decay 0.85 (ner_peft)
  bitfit     bias terms only (M1/M4)                              (ner_peft; extra E7)
  adalora    dW = P diag(lambda) Q with an orthogonality regulariser, sensitivity importance
             (EMA + uncertainty) and a cubic budget schedule 16 -> 8 average rank.
             Zhang et al. (2023), ICLR, arXiv:2303.10512. Code copied from Phase 3 rr_peft.py
             (AdaLoRALinear + AdaLoRA), adapted only in where the head lives (`classifier.*`).

The head (`classifier.*`: attention pooling + output layer) is always trainable.

Lesson L1 (device-order bug of the Phase-3 pilot): the reused ner_peft modules create their
parameters on the default (CPU) device. `apply_technique` therefore moves the whole model to the
device of the frozen base (`trunk.embed.weight.device`) after inserting the modules, and then
asserts that every parameter and buffer sits on that one device, so the result is correct in
both orders ("apply then .to(device)" in production, ".to(device) then apply" in checks).
"""
import copy
import math
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from cjpe_common import DEFAULT_LR, HEAD_LR
from cjpe_models import assert_one_device, model_size
from ner_peft import (FUSED_QKV, TARGETS, LoRALinear, LoRMALinear, AdapterWrapped, _get,
                      _groups, _set)
from ner_peft import apply_technique as phase2_apply

PHASE2 = ("full_ft", "kd", "student_hard", "lorma", "lora", "adapters", "partial_ft", "bitfit")
ALL_TECHNIQUES = PHASE2 + ("adalora",)


# =============================================================================================
# AdaLoRA (copied from Phase 3 rr_peft.py)
# =============================================================================================
class AdaLoRALinear(nn.Module):
    """y = W0 x + b + (alpha/r) * P diag(lambda * mask) Q dropout(x); lambda = 0 at init."""

    def __init__(self, base: nn.Linear, r: int = 16, alpha: float = 32, dropout: float = 0.05):
        super().__init__()
        self.base = base
        dev = base.weight.device
        self.r, self.scale = r, alpha / r
        self.ada_P = nn.Parameter(torch.randn(base.out_features, r, device=dev) * 0.02)
        self.ada_E = nn.Parameter(torch.zeros(r, device=dev))
        self.ada_Q = nn.Parameter(torch.randn(r, base.in_features, device=dev) * 0.02)
        self.register_buffer("ada_mask", torch.ones(r, device=dev))
        for n in ("P", "E", "Q"):
            self.register_buffer(f"ada_ipt_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
            self.register_buffer(f"ada_unc_{n}", torch.zeros_like(getattr(self, f"ada_{n}")))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        y = self.base(x)
        lam = (self.ada_E * self.ada_mask).to(x.dtype)
        u = F.linear(self.drop(x), self.ada_Q.to(x.dtype)) * lam
        return y + self.scale * F.linear(u, self.ada_P.to(x.dtype))

    @torch.no_grad()
    def merged_weight(self):
        d = self.ada_P @ torch.diag(self.ada_E * self.ada_mask) @ self.ada_Q
        return self.base.weight + self.scale * d.to(self.base.weight.dtype)

    def orth_reg(self) -> torch.Tensor:
        P, Q = self.ada_P.float(), self.ada_Q.float()
        I = torch.eye(self.r, device=P.device)
        return ((P.t() @ P - I) ** 2).sum() + ((Q @ Q.t() - I) ** 2).sum()


class Technique:
    """No-op hooks (Phase-2 techniques)."""
    name = "base"

    def extra_loss(self, model) -> Optional[torch.Tensor]:
        return None

    def after_backward(self, model, step: int, total: int) -> None:
        return None

    def selectable(self, step: int, total: int) -> bool:
        return True

    def force_final(self, model) -> None:
        return None

    def final_report(self, model) -> Dict:
        return {}


class AdaLoRA(Technique):
    name = "adalora"

    def setup(self, model, cfg, log=print):
        trunk = model.trunk
        self.r = int(cfg.get("adalora_init_r", 16))
        self.target_r = int(cfg.get("adalora_target_r", 8))
        alpha = float(cfg.get("adalora_alpha", 32))
        self.tinit_frac = float(cfg.get("adalora_tinit_frac", 0.1))
        self.tfinal_frac = float(cfg.get("adalora_tfinal_frac", 0.2))
        self.delta_t = int(cfg.get("adalora_delta_t", 10))
        self.beta1 = float(cfg.get("adalora_beta1", 0.85))
        self.beta2 = float(cfg.get("adalora_beta2", 0.85))
        self.orth_w = float(cfg.get("adalora_orth", 0.1))
        for p in model.parameters():
            p.requires_grad = False
        for blk in trunk.blocks:
            for path in TARGETS[trunk.kind]:
                _set(blk, path, AdaLoRALinear(_get(blk, path), self.r, alpha,
                                              float(cfg.get("lora_dropout", 0.05))))
        self.modules_ = [(n, m) for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)]
        self.b0 = self.r * len(self.modules_)
        self.bT = self.target_r * len(self.modules_)
        named = [(n, p) for n, p in model.named_parameters() if ".ada_" in n]
        for _, p in named:
            p.requires_grad = True
        log(f"   [adalora] {len(self.modules_)} matrices, rank {self.r} -> target "
            f"{self.target_r} (budget {self.b0} -> {self.bT} singular values)")
        return _groups(named, float(cfg["lr"]), float(cfg.get("weight_decay", 0.01)), "adalora")

    def extra_loss(self, model):
        if not self.modules_ or self.orth_w <= 0:
            return None
        return self.orth_w * sum(m.orth_reg() for _, m in self.modules_) / len(self.modules_)

    def phases(self, total: int):
        tf_start = max(0, min(total - int(self.tfinal_frac * total), total - 1))
        return min(int(self.tinit_frac * total), tf_start), tf_start

    def budget(self, step: int, total: int) -> int:
        ti, tf_start = self.phases(total)
        if step < ti:
            return self.b0
        if step >= tf_start:
            return self.bT
        mul = 1.0 - (step - ti) / max(1, tf_start - ti)
        return int(self.bT + (self.b0 - self.bT) * mul ** 3)

    def selectable(self, step, total):
        return step > self.phases(total)[1]

    @torch.no_grad()
    def _update_importance(self):
        for _, m in self.modules_:
            for n in ("P", "E", "Q"):
                p = getattr(m, f"ada_{n}")
                if p.grad is None:
                    continue
                ipt = (p * p.grad).abs()
                ema = getattr(m, f"ada_ipt_{n}")
                unc = getattr(m, f"ada_unc_{n}")
                ema.mul_(self.beta1).add_((1 - self.beta1) * ipt)
                unc.mul_(self.beta2).add_((1 - self.beta2) * (ipt - ema).abs())

    @torch.no_grad()
    def _scores(self) -> List[torch.Tensor]:
        out = []
        for _, m in self.modules_:
            s = lambda n: getattr(m, f"ada_ipt_{n}") * getattr(m, f"ada_unc_{n}")  # noqa: E731
            out.append(s("E") + s("P").mean(0) + s("Q").mean(1))
        return out

    @torch.no_grad()
    def _prune_to(self, b: int):
        scores = self._scores()
        flat = torch.cat(scores)
        if b >= flat.numel():
            return
        thr = torch.kthvalue(flat.float().cpu(), flat.numel() - b).values.item()
        for (_, m), s in zip(self.modules_, scores):
            keep = (s > thr).float()
            m.ada_mask.copy_(keep)
            m.ada_E.mul_(keep)

    @torch.no_grad()
    def after_backward(self, model, step, total):
        ok = all(p.grad is None or torch.isfinite(p.grad).all()
                 for _, m in self.modules_ for p in (m.ada_P, m.ada_E, m.ada_Q))
        if not ok:
            return
        self._update_importance()
        ti, tf_start = self.phases(total)
        if step < ti or step > tf_start:
            return
        if step != tf_start and (step - ti) % self.delta_t != 0:
            return
        self._prune_to(self.budget(step, total))

    def force_final(self, model):
        """A run stopped early by the time guard still ends at the target budget."""
        kept = sum(int(m.ada_mask.sum()) for _, m in self.modules_)
        if kept > self.bT:
            self._prune_to(self.bT)

    def final_report(self, model):
        ranks = {n.replace("trunk.blocks.", "L"): int(m.ada_mask.sum().item())
                 for n, m in model.named_modules() if isinstance(m, AdaLoRALinear)}
        per_layer: Dict[str, int] = {}
        for n, r in ranks.items():
            per_layer[n.split(".")[0]] = per_layer.get(n.split(".")[0], 0) + r
        return {"adalora_rank_per_layer": per_layer,
                "adalora_budget_final": int(sum(ranks.values())),
                "adalora_budget_initial": self.b0, "adalora_budget_target": self.bT}


# =============================================================================================
# entry point
# =============================================================================================
def default_hparams(technique: str) -> Dict:
    lr = DEFAULT_LR.get(technique, 1e-3)
    head = 10 * lr if technique in ("full_ft", "kd", "student_hard") else HEAD_LR
    return {"lr": lr, "head_lr": head}


def apply_technique(model: nn.Module, technique: str, cfg: Dict, log=print
                    ) -> Tuple[List[Dict], Technique]:
    """Insert the technique, set requires_grad, return (optimizer groups, hook object)."""
    if technique not in ALL_TECHNIQUES:
        raise ValueError(f"unknown technique {technique!r}")
    dev = model.trunk.embed.weight.device
    if technique == "adalora":
        tech = AdaLoRA()
        groups = tech.setup(model, cfg, log)
        head = [(n, p) for n, p in model.named_parameters() if n.startswith("classifier.")]
        for _, p in head:
            p.requires_grad = True
        groups += _groups(head, float(cfg.get("head_lr", HEAD_LR)),
                          float(cfg.get("weight_decay", 0.01)), "head")
        expected = {n for n, _ in head} | {n for n, p in model.named_parameters() if ".ada_" in n}
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != expected:
            raise AssertionError(f"[adalora] trainable mismatch {sorted(actual ^ expected)[:5]}")
    else:
        p2 = "full_ft" if technique in ("kd", "student_hard") else technique
        groups = phase2_apply(model, p2, cfg, log=lambda *a: None)
        tech = Technique()
        tech.name = technique
    model.to(dev)                                         # lesson L1
    assert_one_device(model, dev)
    s = model_size(model)
    log(f"   [{technique}] trainable {s['trainable_params']:,} / {s['total_params']:,} "
        f"({s['trainable_pct']:.3f}%)")
    return groups, tech


# =============================================================================================
# merge (used by the self-checks: merged == unmerged)
# =============================================================================================
def _clone_linear(base: nn.Linear) -> nn.Linear:
    lin = nn.Linear(base.in_features, base.out_features, bias=base.bias is not None,
                    device=base.weight.device, dtype=base.weight.dtype)
    with torch.no_grad():
        lin.weight.copy_(base.weight)
        if base.bias is not None:
            lin.bias.copy_(base.bias)
    return lin


def merge_model(model: nn.Module) -> int:
    """Fold LoRA / LoRMA / AdaLoRA updates into plain Linear layers. Returns the count."""
    hits = [(n, m) for n, m in model.named_modules()
            if isinstance(m, (LoRALinear, LoRMALinear, AdaLoRALinear))]
    for n, m in hits:
        lin = _clone_linear(m.base)
        with torch.no_grad():
            lin.weight.copy_(m.merged_weight())
        _set(model, n, lin)
    return len(hits)


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_eval.py
"""
cjpe_eval.py -- binary Macro-F1 (implemented here, no sklearn), threshold tuning on dev, and the
per-run analysis tables (section 11.1).

Macro-F1 = mean of the F1 of class 0 and class 1 (both always counted: a class that is never
predicted gets F1 = 0, so predicting only the majority class is punished).
"""
from typing import Dict, List, Optional, Sequence

import numpy as np

from cjpe_data import BUCKETS, bucket_of


def confusion(gold: Sequence[int], pred: Sequence[int]) -> Dict[str, int]:
    g = np.asarray(gold, dtype=np.int64)
    p = np.asarray(pred, dtype=np.int64)
    return {"tp": int(((g == 1) & (p == 1)).sum()), "fp": int(((g == 0) & (p == 1)).sum()),
            "fn": int(((g == 1) & (p == 0)).sum()), "tn": int(((g == 0) & (p == 0)).sum())}


def _prf(tp: int, fp: int, fn: int):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def binary_scores(gold: Sequence[int], pred: Sequence[int]) -> Dict[str, float]:
    c = confusion(gold, pred)
    p1, r1, f1 = _prf(c["tp"], c["fp"], c["fn"])            # class 1 = positive
    p0, r0, f0 = _prf(c["tn"], c["fn"], c["fp"])            # class 0 viewed as positive
    n = max(1, sum(c.values()))
    return {"macro_f1": (f0 + f1) / 2, "macro_p": (p0 + p1) / 2, "macro_r": (r0 + r1) / 2,
            "accuracy": (c["tp"] + c["tn"]) / n, "f1_class0": f0, "f1_class1": f1,
            "p_class0": p0, "p_class1": p1, "r_class0": r0, "r_class1": r1,
            "precision": p1, "recall": r1, "f1_positive": f1, **c}


def macro_f1(gold, pred) -> float:
    return binary_scores(gold, pred)["macro_f1"]


def tune_tau(p1: Sequence[float], gold: Sequence[int], lo: float = 0.05, hi: float = 0.95,
             step: float = 0.01, min_frac: float = 0.05) -> Dict:
    """Best macro-F1 threshold on dev, keeping >= min_frac of predictions in each class.
    Falls back to 0.5 when no threshold beats it."""
    p = np.asarray(p1, dtype=np.float64)
    g = np.asarray(gold, dtype=np.int64)
    base = macro_f1(g, (p >= 0.5).astype(int))
    best_t, best_f = 0.5, base
    n = len(p)
    for t in np.arange(lo, hi + 1e-9, step):
        pr = (p >= t).astype(int)
        pos = pr.mean() if n else 0
        if pos < min_frac or pos > 1 - min_frac:
            continue
        f = macro_f1(g, pr)
        if f > best_f + 1e-12:
            best_t, best_f = float(round(t, 4)), f
    return {"tau": best_t, "f1_tuned": best_f, "f1_at_0p5": base,
            "used": "tuned" if best_t != 0.5 else "0.5"}


def length_table(gold, pred, n_tokens) -> List[Dict]:
    rows = []
    by: Dict[str, List[int]] = {}
    for i, n in enumerate(n_tokens):
        by.setdefault(bucket_of(n), []).append(i)
    for _, name in BUCKETS:
        idx = by.get(name, [])
        if not idx:
            continue
        s = binary_scores([gold[i] for i in idx], [pred[i] for i in idx])
        rows.append({"bucket": name, "n": len(idx), "macro_f1": s["macro_f1"],
                     "accuracy": s["accuracy"], "pos_rate_gold": float(np.mean([gold[i] for i in idx]))})
    return rows


def reliability(p1, gold, bins: int = 10) -> List[Dict]:
    p = np.asarray(p1, dtype=np.float64)
    g = np.asarray(gold, dtype=np.float64)
    rows = []
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        m = (p >= lo) & ((p < hi) if b < bins - 1 else (p <= hi))
        if m.any():
            rows.append({"bin": f"{lo:.1f}-{hi:.1f}", "n": int(m.sum()),
                         "mean_p1": float(p[m].mean()), "observed_pos_rate": float(g[m].mean())})
    return rows


def ece(p1, gold, bins: int = 10) -> float:
    rel = reliability(p1, gold, bins)
    n = sum(r["n"] for r in rel)
    return float(sum(r["n"] * abs(r["mean_p1"] - r["observed_pos_rate"]) for r in rel) / max(1, n))


def full_report(ids, gold, p1, tau: float, n_tokens) -> Dict:
    pred = [int(x >= tau) for x in p1]
    s = binary_scores(gold, pred)
    s05 = binary_scores(gold, [int(x >= 0.5) for x in p1])
    return {"tau": tau, "scores": s, "scores_at_0p5": s05,
            "by_length": length_table(gold, pred, n_tokens),
            "reliability": reliability(p1, gold), "ece": ece(p1, gold),
            "n": len(ids), "pred_pos_rate": float(np.mean(pred)) if pred else 0.0}


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_kd.py
"""
cjpe_kd.py -- knowledge distillation for E6 (section 7.2).

Teacher : full fine-tuning of TEACHER_MODEL (default M4) trained inside E6 (job
          `teacher_<model>`), or a finished E1 run of that model found in the attached inputs
          whose fingerprint (data, tokenizer, representation) matches.
          After training, the teacher's logits for every train / dev / test document are cached
          once in `teacher_logits.npz` (eval mode, no dropout, no_grad).
Students: the four Phase-1 models pruned to every second layer (Trunk(layer_ids=...)), full
          fine-tuning, same `hier` representation, so the logits align per DOCUMENT.
Loss    : L = a * BCE(student, y) + (1 - a) * T^2 * BCE(s/T, sigmoid(t/T))  (multi-label),
          a = 0.5, T = 2 (Hinton et al., 2015, "Distilling the Knowledge in a Neural Network").
"""
import glob
import os
from typing import Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import read_json


def kd_loss(s_logits: torch.Tensor, t_logits: torch.Tensor, labels: torch.Tensor,
            alpha: float = 0.5, T: float = 2.0, weight: Optional[torch.Tensor] = None
            ) -> torch.Tensor:
    """Phase 5 (multi-label): a * BCE(student, gold) + (1-a) * T^2 * soft-target BCE against
    sigmoid(teacher / T); `weight` = per-label pos_weight."""
    from lsi_ml import kd_loss_ml
    return kd_loss_ml(s_logits, t_logits, labels, alpha, T, weight)


def save_teacher_logits(path: str, logits: Dict[str, Dict[str, np.ndarray]]) -> None:
    """logits[split] = {"ids": array(str), "logits": (n, 2) float32}."""
    flat = {}
    for s, d in logits.items():
        flat[f"{s}_ids"] = np.asarray(d["ids"])
        flat[f"{s}_logits"] = np.asarray(d["logits"], np.float32)
    tmp = path + ".tmp.npz"
    np.savez_compressed(tmp, **flat)
    os.replace(tmp, path)


def load_teacher_logits(path: str) -> Dict[str, Dict[str, np.ndarray]]:
    z = np.load(path, allow_pickle=False)
    out = {}
    for k in z.files:
        if k.endswith("_ids"):
            s = k[:-4]
            out[s] = {"ids": z[k], "logits": z[f"{s}_logits"]}
    return out


def teacher_map(path: str, split: str = "train") -> Dict[str, np.ndarray]:
    d = load_teacher_logits(path)[split]
    return {str(i): l for i, l in zip(d["ids"], d["logits"])}


def find_reusable_teacher(input_roots, model_folder: str, fingerprint: Dict) -> Optional[str]:
    """A finished E1 run of the teacher model with an identical fingerprint, or None."""
    for r in input_roots:
        if not r or not os.path.isdir(r):
            continue
        pat = os.path.join(r, "**", "phase5_out", "E1_full_ft", model_folder, "DONE.json")
        for done in glob.glob(pat, recursive=True):
            run = os.path.dirname(done)
            rc = read_json(os.path.join(run, "logs", "run_config.json"), {}) or {}
            if not os.path.exists(os.path.join(run, "checkpoints", "best.pt")):
                continue
            if all(rc.get(k) == v for k, v in fingerprint.items()):
                return run
    return None


In [ ]:
%%writefile /kaggle/working/phase6/lsi_ml.py
"""
lsi_ml.py -- Phase 5 (Legal Statute Identification) multi-label core: targets, losses,
metrics, dev-only decision rules and the per-run analysis (brief sections 3.3, 3.5, 5).

Labels inside the pipeline are multi-hot uint8 rows over the TRAIN label map (`label_ids`,
sorted statute ids seen in train.json). Statute ids that occur only in dev are never predicted
(the evaluator only accepts valid ids) and are reported as `dev_only_labels`.
"""
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F

GRID = np.round(np.arange(0.02, 0.901, 0.02), 4)
RARE, FREQ = 5, 20                       # train-support buckets: <5 rare, 5-20 medium, >20 freq


# =============================================================================================
# labels / targets
# =============================================================================================
def multihot(label_lists: Sequence, lab2i: Dict[int, int], L: int) -> np.ndarray:
    Y = np.zeros((len(label_lists), L), np.uint8)
    for r, labs in enumerate(label_lists):
        for x in labs or []:
            j = lab2i.get(int(x))
            if j is not None:
                Y[r, j] = 1
    return Y


def targets(labels, idx, device) -> torch.Tensor:
    return torch.from_numpy(np.stack([labels[i] for i in idx]).astype(np.float32)).to(device)


def pos_weight(Y: np.ndarray, mode: str = "sqrt_inv", cap: float = 30.0) -> Optional[np.ndarray]:
    if mode == "none":
        return None
    pos = Y.sum(0).astype(np.float64)
    neg = len(Y) - pos
    r = neg / np.maximum(pos, 1.0)
    w = np.sqrt(r) if mode == "sqrt_inv" else r
    w = np.clip(w, 1.0, cap)
    w[pos == 0] = 1.0
    return w.astype(np.float32)


def prior_logit(Y: np.ndarray) -> np.ndarray:
    p = (Y.sum(0).astype(np.float64) + 0.5) / (len(Y) + 1.0)
    p = np.clip(p, 1e-4, 0.5)
    return np.log(p / (1 - p)).astype(np.float32)


# =============================================================================================
# losses (per document = sum over labels; fp32)
# =============================================================================================
def ml_loss(logits: torch.Tensor, Y: torch.Tensor, pw: Optional[torch.Tensor] = None,
            kind: str = "bce") -> torch.Tensor:
    x = logits.float()
    if kind == "asl":                    # asymmetric loss (Ridnik et al. 2021): g-=4, g+=0, clip
        p = torch.sigmoid(x)
        pm = (p - 0.05).clamp(min=0)
        lp = Y * torch.log(p.clamp(min=1e-8))
        ln = (1 - Y) * torch.log((1 - pm).clamp(min=1e-8)) * pm.pow(4)
        return -(lp + ln).sum(1)
    if kind == "focal":                  # focal BCE, gamma 2, with pos_weight on positives
        b = F.binary_cross_entropy_with_logits(x, Y, reduction="none")
        p = torch.sigmoid(x)
        pt = Y * p + (1 - Y) * (1 - p)
        w = (1 - pt).pow(2)
        if pw is not None:
            w = w * (1 + (pw - 1) * Y)
        return (w * b).sum(1)
    return F.binary_cross_entropy_with_logits(x, Y, pos_weight=pw, reduction="none").sum(1)


def kd_loss_ml(s_logits, t_logits, Y, alpha=0.5, T=2.0, pw=None) -> torch.Tensor:
    """a * BCE(student, gold) + (1-a) * T^2 * BCE(student/T, sigmoid(teacher/T)); mean/doc."""
    s = s_logits.float()
    hard = ml_loss(s, Y, pw).mean()
    soft = F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t_logits.float() / T),
                                              reduction="none").sum(1).mean()
    return alpha * hard + (1 - alpha) * T * T * soft


def sigmoid_np(z: np.ndarray) -> np.ndarray:
    return (1.0 / (1.0 + np.exp(-np.clip(z, -40, 40)))).astype(np.float32)


# =============================================================================================
# metrics
# =============================================================================================
def counts(G: np.ndarray, P: np.ndarray):
    G = G.astype(bool)
    P = P.astype(bool)
    tp = (G & P).sum(0).astype(np.int64)
    fp = (~G & P).sum(0).astype(np.int64)
    fn = (G & ~P).sum(0).astype(np.int64)
    return tp, fp, fn


def _f1(tp, fp, fn):
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0)
    r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    f = np.where(p + r > 0, 2 * p * r / np.maximum(p + r, 1e-12), 0.0)
    return p, r, f


def ml_scores(G: np.ndarray, P: np.ndarray, train_support: Optional[np.ndarray] = None,
              n_dev_only_pos: int = 0) -> Dict:
    """macro_f1 = mean F1 over labels with dev support (what an evaluator built from the dev
    reference would see); macro_f1_all = over every train label (0 for unseen+unpredicted)."""
    tp, fp, fn = counts(G, P)
    p, r, f = _f1(tp, fp, fn)
    sup = G.sum(0) > 0
    out = {"macro_f1": float(f[sup].mean()) if sup.any() else 0.0,
           "macro_p": float(p[sup].mean()) if sup.any() else 0.0,
           "macro_r": float(r[sup].mean()) if sup.any() else 0.0,
           "macro_f1_all": float(f.mean()) if len(f) else 0.0}
    TP, FP, FN = int(tp.sum()), int(fp.sum()), int(fn.sum()) + int(n_dev_only_pos)
    mp = TP / max(1, TP + FP)
    mr = TP / max(1, TP + FN)
    out.update({"micro_f1": 2 * mp * mr / max(1e-12, mp + mr), "micro_p": mp, "micro_r": mr,
                "subset_acc": float((G.astype(bool) == P.astype(bool)).all(1).mean())
                if len(G) else 0.0,
                "mean_pred_labels": float(P.sum(1).mean()) if len(P) else 0.0,
                "mean_gold_labels": float(G.sum(1).mean()) if len(G) else 0.0,
                "n_labels_with_dev_support": int(sup.sum()), "tp": TP, "fp": FP, "fn": FN})
    if train_support is not None:
        for name, m in (("rare", train_support < RARE),
                        ("medium", (train_support >= RARE) & (train_support <= FREQ)),
                        ("frequent", train_support > FREQ)):
            mm = m & sup
            out[f"macro_f1_{name}"] = float(f[mm].mean()) if mm.any() else None
            out[f"n_{name}"] = int(mm.sum())
    return out


def macro(G, P) -> float:
    tp, fp, fn = counts(G, P)
    f = _f1(tp, fp, fn)[2]
    sup = G.sum(0) > 0
    return float(f[sup].mean()) if sup.any() else 0.0


# =============================================================================================
# decision rules (dev only)
# =============================================================================================
def apply_rule(Pr: np.ndarray, taus: np.ndarray, kmax: int) -> np.ndarray:
    """threshold per label, then >= 1 label per document (top-1) and <= kmax labels."""
    pred = Pr >= taus[None, :]
    if len(Pr) == 0:
        return pred
    top1 = Pr.argmax(1)
    pred[np.arange(len(Pr)), top1] = True
    over = np.where(pred.sum(1) > kmax)[0]
    for i in over:
        keep = np.argsort(-np.where(pred[i], Pr[i], -1.0))[:kmax]
        pred[i] = False
        pred[i, keep] = True
    return pred


def fit_global(Pr, G, kmax) -> (float, float):
    best_t, best_f = 0.5, -1.0
    L = Pr.shape[1]
    for t in GRID:
        f = macro(G, apply_rule(Pr, np.full(L, t, np.float32), kmax))
        if f > best_f + 1e-12:
            best_t, best_f = float(t), f
    return best_t, best_f


def fit_perlabel(Pr, G, tg: float, m: int = 5, lam: float = 0.5) -> np.ndarray:
    L = Pr.shape[1]
    taus = np.full(L, tg, np.float32)
    sup = G.sum(0)
    cand = np.where(sup >= m)[0]
    if not len(cand):
        return taus
    best_f = np.full(len(cand), -1.0)
    best_t = np.full(len(cand), tg)
    Gc = G[:, cand].astype(bool)
    for t in GRID:
        Pc = Pr[:, cand] >= t
        tp = (Gc & Pc).sum(0)
        fp = (~Gc & Pc).sum(0)
        fn = (Gc & ~Pc).sum(0)
        f = _f1(tp, fp, fn)[2]
        better = f > best_f + 1e-12
        best_f[better] = f[better]
        best_t[better] = t
    taus[cand] = lam * best_t + (1 - lam) * tg
    return taus


def choose_rule(Pr: np.ndarray, G: np.ndarray, kmax: int, seed: int = 0,
                allow_perlabel: bool = True) -> Dict:
    L = Pr.shape[1]
    tg, fg = fit_global(Pr, G, kmax)
    taus_pl = fit_perlabel(Pr, G, tg)
    f_pl_in = macro(G, apply_rule(Pr, taus_pl, kmax))
    # honest estimate: 2-fold cross-fitting on dev
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(Pr))
    halves = [perm[: len(perm) // 2], perm[len(perm) // 2:]]
    cv_g, cv_p = [], []
    for a, b in ((0, 1), (1, 0)):
        A, B = halves[a], halves[b]
        if len(A) < 4 or len(B) < 4:
            continue
        t_a, _ = fit_global(Pr[A], G[A], kmax)
        pl_a = fit_perlabel(Pr[A], G[A], t_a)
        cv_g.append(macro(G[B], apply_rule(Pr[B], np.full(L, t_a, np.float32), kmax)))
        cv_p.append(macro(G[B], apply_rule(Pr[B], pl_a, kmax)))
    cvg = float(np.mean(cv_g)) if cv_g else None
    cvp = float(np.mean(cv_p)) if cv_p else None
    use_pl = bool(allow_perlabel and cvg is not None and cvp is not None and cvp > cvg + 0.002)
    taus = taus_pl if use_pl else np.full(L, tg, np.float32)
    return {"name": "perlabel" if use_pl else "global", "tau_global": tg, "kmax": int(kmax),
            "taus": [round(float(x), 4) for x in taus],
            "dev_macro_global": fg, "dev_macro_perlabel_insample": f_pl_in,
            "cv_macro_global": cvg, "cv_macro_perlabel": cvp}


def rule_taus(rule: Dict, L: int) -> np.ndarray:
    t = np.asarray(rule.get("taus") or [rule.get("tau_global", 0.5)] * L, np.float32)
    return t if len(t) == L else np.full(L, float(rule.get("tau_global", 0.5)), np.float32)


# =============================================================================================
# per-run analysis
# =============================================================================================
def per_label_rows(G, P, taus, label_ids, train_support) -> List[Dict]:
    tp, fp, fn = counts(G, P)
    p, r, f = _f1(tp, fp, fn)
    return [{"label": int(label_ids[j]), "train_support": int(train_support[j]),
             "dev_support": int(G[:, j].sum()), "precision": float(p[j]), "recall": float(r[j]),
             "f1": float(f[j]), "predicted_count": int(P[:, j].sum()), "tau": float(taus[j])}
            for j in range(G.shape[1])]


def confused_pairs(G, P, label_ids, top: int = 30) -> List[Dict]:
    from collections import Counter
    c = Counter()
    Gb, Pb = G.astype(bool), P.astype(bool)
    for i in range(len(G)):
        miss = np.where(Gb[i] & ~Pb[i])[0]
        spur = np.where(Pb[i] & ~Gb[i])[0]
        for a in miss[:20]:
            for b in spur[:20]:
                c[(int(label_ids[a]), int(label_ids[b]))] += 1
    return [{"missed": a, "predicted_instead": b, "docs": n} for (a, b), n in c.most_common(top)]


def length_rows(G, P, n_tokens) -> List[Dict]:
    from cjpe_data import BUCKETS, bucket_of
    by: Dict[str, List[int]] = {}
    for i, n in enumerate(n_tokens):
        by.setdefault(bucket_of(n), []).append(i)
    rows = []
    for _, name in BUCKETS:
        idx = by.get(name, [])
        if idx:
            s = ml_scores(G[idx], P[idx])
            rows.append({"bucket": name, "n": len(idx), "macro_f1": s["macro_f1"],
                         "micro_f1": s["micro_f1"], "mean_gold_labels": s["mean_gold_labels"]})
    return rows


def ece_ml(Pr, G, bins: int = 10, max_n: int = 2_000_000) -> float:
    p = Pr.ravel()
    g = G.ravel().astype(np.float64)
    if len(p) > max_n:
        sel = np.random.default_rng(0).choice(len(p), max_n, replace=False)
        p, g = p[sel], g[sel]
    e, n = 0.0, len(p)
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        m = (p >= lo) & ((p < hi) if b < bins - 1 else (p <= hi))
        if m.any():
            e += m.sum() * abs(float(p[m].mean()) - float(g[m].mean()))
    return float(e / max(1, n))


def full_report(G, Pr, rule, n_tokens, label_ids, train_support, n_dev_only_pos=0) -> Dict:
    L = Pr.shape[1]
    taus = rule_taus(rule, L)
    P = apply_rule(Pr, taus, rule["kmax"])
    P05 = apply_rule(Pr, np.full(L, 0.5, np.float32), rule["kmax"])
    Pg = apply_rule(Pr, np.full(L, rule["tau_global"], np.float32), rule["kmax"])
    return {"rule": {k: v for k, v in rule.items() if k != "taus"},
            "scores": ml_scores(G, P, train_support, n_dev_only_pos),
            "scores_at_0p5": ml_scores(G, P05, train_support, n_dev_only_pos),
            "scores_global": ml_scores(G, Pg, train_support, n_dev_only_pos),
            "by_length": length_rows(G, P, n_tokens),
            "confused_pairs": confused_pairs(G, P, label_ids),
            "ece": ece_ml(Pr, G), "n": int(len(G)),
            "_P": P, "_per_label": per_label_rows(G, P, taus, label_ids, train_support)}


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_train.py
"""
cjpe_train.py -- ONE training job (one model x one technique), run as a subprocess on ONE GPU
(the orchestrator sets CUDA_VISIBLE_DEVICES). Reads RUN/logs/run_config.json.

    python cjpe_train.py --run-dir RUN [--mode train|smoke|overfit|teacher_reuse]

train  : self-checks on the real device -> throughput pre-flight -> time plan (section 8.4)
         -> training with dev evaluation (2x per epoch), threshold tuning, early stopping,
         30-minute resumable checkpoints, time guard -> best.pt -> final dev report, test
         prediction, validated submission.zip -> DONE.json (or status DEGENERATE)
         (+ teacher logits for train/dev/test when run_config["is_teacher"])
smoke  : self-checks + 20-step throughput only (00 notebook)
overfit: 32 training documents must reach loss < 0.1 within 100 steps (selftest)
teacher_reuse : load a finished E1 run, write its teacher logits + dev report

Exit codes: 0 ok, 1 error, 2 does not fit the time budget, 4 too many NaN steps,
5 early-fail watchdog, 6 critical self-check failed, 7 OOM at one document per micro-batch.
Every state change is written to RUN/status.json (also from `finally`), so a crash always
leaves a diagnosable file (rule R3).
"""
import argparse
import contextlib
import copy
import json
import math
import os
import random
import sys
import time
import traceback
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import (DEFAULT_EPOCHS, MAX_TRAINABLE_PCT, SPEC, disk_guard, DiskFull,
                         log_to, read_json, write_json, now_str)
from lsi_ml import (apply_rule, choose_rule, macro, ml_loss, ml_scores, multihot, pos_weight,
                    prior_logit, rule_taus, sigmoid_np, targets)
from cjpe_kd import kd_loss, save_teacher_logits, teacher_map
from cjpe_models import (assert_one_device, build_doc_model, collate_docs, forward_batch,
                         load_trainable_state, model_size, rebuild_trained, trainable_state_dict)
from cjpe_peft import apply_technique, merge_model
from cjpe_predict import (batches_by_chunks, predict_logits, softmax1, write_dev_outputs,
                          write_submission)
from cjpe_tok import doc_chunks, load_cache
from ner_models import torch_load


class JobExit(Exception):
    def __init__(self, code: int, reason: str):
        super().__init__(reason)
        self.code, self.reason = code, reason


# =============================================================================================
# small helpers
# =============================================================================================
class Ctx:
    def __init__(self, run_dir: str, cfg: Dict):
        self.run_dir = run_dir
        self.cfg = cfg
        self.log_path = os.path.join(run_dir, "logs", "console.log")
        self.t0 = time.time()
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def log(self, msg: str) -> None:
        log_to(self.log_path, msg, prefix=f"[{self.cfg.get('job', '?')}] ")

    def status(self, state: str, reason: str = "", **extra) -> None:
        p = os.path.join(self.run_dir, "status.json")
        st = read_json(p, {}) or {}
        hist = st.get("history", [])
        hist.append({"state": state, "time": now_str(), "reason": reason[:500]})
        st.update({"state": state, "reason": reason[:2000], "time": now_str(),
                   "history": hist[-50:], **extra})
        write_json(p, st)

    def jlog(self, rec: Dict) -> None:
        with open(os.path.join(self.run_dir, "logs", "train_log.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")


def amp_setup(device):
    """The Phase-1 AMP pattern: fp16 autocast + GradScaler on T4 (no bf16), bf16 on Ampere+."""
    if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8:
        dtype = torch.bfloat16
    elif device.type == "cuda":
        dtype = torch.float16
    else:
        dtype = None

    def amp_ctx():
        if dtype is None:
            return contextlib.nullcontext()
        return torch.autocast(device_type="cuda", dtype=dtype)
    return dtype, amp_ctx


def make_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def is_oom(e: BaseException) -> bool:
    return "out of memory" in str(e).lower()


def peak_gb() -> float:
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def set_seed(s: int) -> None:
    random.seed(s)
    np.random.seed(s % (2 ** 32))
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# =============================================================================================
# data
# =============================================================================================
def load_split_docs(cfg: Dict, K: Optional[int] = None):
    """{split: {"ids", "labels", "chunks", "n_tokens"}} for the configured representation."""
    cache = load_cache(cfg["cache_path"])
    K = int(K or cfg["K"])
    lab = [int(x) for x in cfg["label_ids"]]
    lab2i = {x: i for i, x in enumerate(lab)}
    out = {}
    for split in ("train", "dev", "test"):
        rows = cache[split]
        labs = [list(r["label"] or []) if split != "test" else [] for r in rows]
        out[split] = {"ids": [r["id"] for r in rows],
                      "labels": multihot(labs, lab2i, len(lab)),
                      "dev_only_pos": sum(1 for l in labs for x in l if int(x) not in lab2i),
                      "chunks": [doc_chunks(r, cfg["doc_repr"], K) for r in rows],
                      "n_tokens": [int(r["n_tokens"]) for r in rows]}
    return out


def assert_data_integrity(d: Dict, cfg: Dict) -> Dict:
    """Section 4.3: training ids are train ids only, disjoint from dev and test."""
    tr, dv, te = set(d["train"]["ids"]), set(d["dev"]["ids"]), set(d["test"]["ids"])
    assert not (tr & te), f"{len(tr & te)} training ids are test ids"
    assert not (tr & dv), f"{len(tr & dv)} training ids are dev ids"
    L = len(cfg["label_ids"])
    for s_ in ("train", "dev", "test"):
        assert d[s_]["labels"].shape == (len(d[s_]["ids"]), L), f"{s_}: label matrix shape"
    assert int(d["train"]["labels"].sum()) > 0, "train has no positive labels"
    for s in ("train", "dev", "test"):
        for ch in d[s]["chunks"][:2000]:
            assert ch, f"{s}: a document has no chunk"
            assert all(len(c) <= 1024 for c in ch), f"{s}: sequence > 1024 tokens"
            assert all(int(c.max()) < 32000 for c in ch), f"{s}: token id >= 32000"
    return {"train_ids_subset_of_train_json": True, "train_x_dev": 0, "train_x_test": 0}


def stratified_subsample(labels, frac: float, seed: int) -> List[int]:
    """Seeded subsample of documents; every document that carries a rare label (train support
    < 5) is always kept so rare statutes are not lost when the time budget forces frac < 1."""
    n = len(labels)
    if frac >= 0.999:
        return list(range(n))
    rng = np.random.default_rng(seed)
    keep = set(rng.choice(n, max(1, int(round(frac * n))), replace=False).tolist())
    Y = np.asarray(labels)
    if Y.ndim == 2 and n:
        rare = Y.sum(0) < 5
        if rare.any():
            keep |= set(np.where(Y[:, rare].any(1))[0].tolist())
    return sorted(keep)


def class_weights(labels, mode: str, device) -> Optional[torch.Tensor]:
    """Per-label BCE pos_weight: clip(sqrt(neg/pos), 1, 30) (mode sqrt_inv), neg/pos (inv)."""
    w = pos_weight(np.asarray(labels), "none" if mode == "none" else
                   ("inv" if mode == "inv" else "sqrt_inv"))
    return None if w is None else torch.tensor(w, dtype=torch.float32, device=device)


# =============================================================================================
# self-checks on the REAL device (lesson L1 / L2)
# =============================================================================================
def job_checks(base: Optional[torch.nn.Module], model, tech, cfg: Dict, d: Dict, device,
               amp_ctx) -> Dict:
    """critical: device placement, output shape, finite loss and gradients, data sanity.
    advisory: zero-init equals the base, merged equals unmerged, trainable-% cap."""
    res = {"checks": [], "critical_failed": [], "advisory_failed": []}

    def add(name, kind, fn):
        rec = {"name": name, "kind": kind}
        try:
            val = fn()
            if isinstance(val, dict):
                rec.update(val)
            rec.setdefault("verdict", "pass")
        except Exception as e:
            rec["verdict"] = "fail"
            rec["error"] = f"{type(e).__name__}: {e}"[:1500]
            rec["trace"] = traceback.format_exc()[-1500:]
        if rec["verdict"] != "pass":
            (res["critical_failed"] if kind == "critical" else res["advisory_failed"]).append(name)
        res["checks"].append(rec)

    tr = d["train"]
    idx = [i for i in range(min(len(tr["ids"]), 50))][:2]
    b = collate_docs([tr["chunks"][i] for i in idx], device)
    y = targets(tr["labels"], idx, device)
    L = len(cfg["label_ids"])

    add("all_tensors_on_one_device", "critical", lambda: (assert_one_device(model, device),
                                                         {"device": str(device)})[1])

    def fwd():
        model.train()
        with amp_ctx():
            logits, w = forward_batch(model, b)
        assert logits.shape == (len(idx), L), f"logits {tuple(logits.shape)}"
        assert torch.isfinite(logits).all(), "non-finite logits"
        loss = ml_loss(logits.float(), y).mean()
        ex = tech.extra_loss(model)
        if ex is not None:
            loss = loss + ex
        assert torch.isfinite(loss), "non-finite loss"
        loss.backward()
        gr = [p.grad for p in model.parameters() if p.requires_grad]
        assert any(g is not None and float(g.abs().sum()) > 0 for g in gr), "no gradient"
        assert all(g is None or bool(torch.isfinite(g).all()) for g in gr), "non-finite grad"
        model.zero_grad(set_to_none=True)
        wz = w[~(torch.arange(w.shape[1], device=w.device)[None] <
                 torch.bincount(b["doc_index"], minlength=w.shape[0])[:, None])]
        assert float(wz.detach().abs().sum()) == 0.0, "masked chunk got attention weight"
        return {"loss": float(loss), "logits_shape": list(logits.shape)}
    add("forward_backward_finite", "critical", fwd)

    def tiny_batch():
        ch = [[c[:64] for c in tr["chunks"][i][:1]] for i in idx]
        return collate_docs(ch, device)

    if base is not None and cfg["technique"] in ("lorma", "lora", "adalora", "adapters"):
        def zero_init():
            tb = tiny_batch()
            model.eval()
            base.eval()
            with torch.no_grad():
                a = model.chunk_vectors(tb["input_ids"], tb["attention_mask"])
                c = base.chunk_vectors(tb["input_ids"], tb["attention_mask"])
            diff = float((a - c).abs().max())
            scale = max(1.0, float(c.abs().max()))
            thr = (5e-2 if cfg["technique"] == "adapters" else 1e-4) * scale
            return {"value": diff, "threshold": thr, "verdict": "pass" if diff <= thr else "warn"}
        add("zero_init_equals_base", "advisory", zero_init)
    if cfg["technique"] in ("lorma", "lora", "adalora"):
        def merge_eq():
            m = copy.deepcopy(model)
            g = torch.Generator().manual_seed(7)
            with torch.no_grad():
                for n, p in m.named_parameters():
                    if any(t in n for t in ("lora_B", "lorma_B", "ada_E")):
                        p.add_((torch.randn(p.shape, generator=g) * 0.02).to(p.device, p.dtype))
            tb = tiny_batch()
            m.eval()
            with torch.no_grad():
                u = m.chunk_vectors(tb["input_ids"], tb["attention_mask"])
                n_merged = merge_model(m)
                v = m.chunk_vectors(tb["input_ids"], tb["attention_mask"])
            diff = float((u - v).abs().max())
            thr = 1e-3 * max(1.0, float(u.abs().max()))
            del m
            return {"value": diff, "threshold": thr, "merged_layers": n_merged,
                    "verdict": "pass" if diff <= thr else "warn"}
        add("merged_equals_unmerged", "advisory", merge_eq)
    s = model_size(model)
    real = s["total_params"] >= SPEC["rules"]["foundation_params"][0] * 0.4
    if cfg["technique"] in MAX_TRAINABLE_PCT and real:
        cap = MAX_TRAINABLE_PCT[cfg["technique"]]
        add("trainable_pct_cap", "advisory",
            lambda: {"value": s["trainable_pct"], "threshold": cap,
                     "verdict": "pass" if s["trainable_pct"] <= cap else "warn"})
    model.zero_grad(set_to_none=True)
    res["ok"] = not res["critical_failed"]
    res["selfcheck"] = "fail" if res["critical_failed"] else (
        "warn" if res["advisory_failed"] else "pass")
    return res


# =============================================================================================
# one optimizer step (with OOM back-off)
# =============================================================================================
class Stepper:
    def __init__(self, ctx, model, opt, sched, scaler, tech, amp_ctx, cw, cfg, tmap=None):
        self.ctx, self.model, self.opt, self.sched, self.scaler = ctx, model, opt, sched, scaler
        self.tech, self.amp_ctx, self.cw, self.cfg, self.tmap = tech, amp_ctx, cw, cfg, tmap
        self.micro = int(cfg["micro_chunks"])
        self.skipped = 0
        self.overflow = 0
        self.params = [p for p in model.parameters() if p.requires_grad]

    def loss_of(self, logits, y, ids):
        if self.tmap is not None:
            t = torch.tensor(np.stack([self.tmap[i] for i in ids]), device=logits.device)
            return kd_loss(logits, t, y, float(self.cfg.get("kd_alpha", 0.5)),
                           float(self.cfg.get("kd_T", 2.0)), self.cw) * len(ids)
        per = ml_loss(logits, y, self.cw, self.cfg.get("loss", "bce"))
        return per.sum()

    def step(self, data, doc_idx: List[int], global_step: int, total_steps: int):
        """Forward/backward over `doc_idx` (one effective batch) + optimizer step.
        Returns (mean loss, tokens) or (None, tokens) for a skipped non-finite step."""
        while True:
            try:
                return self._step(data, doc_idx, global_step, total_steps)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                self.opt.zero_grad(set_to_none=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                one_doc = max(len(data["chunks"][i]) for i in doc_idx)
                if self.micro <= one_doc:
                    raise JobExit(7, "CUDA OOM with one document per micro-batch")
                self.micro = max(one_doc, self.micro // 2)
                self.ctx.log(f"   [oom] micro-batch -> {self.micro} chunks (accumulation grows)")

    def _step(self, data, doc_idx, global_step, total_steps):
        model = self.model
        model.train()
        n = len(doc_idx)
        tot, toks = 0.0, 0
        for g in batches_by_chunks(data["chunks"], self.micro, doc_idx):
            b = collate_docs([data["chunks"][i] for i in g], self.ctx.device)
            y = targets(data["labels"], g, self.ctx.device)
            with self.amp_ctx():
                logits, _ = forward_batch(model, b)
            loss = self.loss_of(logits, y, [data["ids"][i] for i in g]) / n
            ex = self.tech.extra_loss(model)
            if ex is not None:
                loss = loss + ex * (len(g) / n)
            self.scaler.scale(loss).backward()
            tot += float(loss.detach())
            toks += b["n_tokens"]
        self.scaler.unscale_(self.opt)
        gn = torch.nn.utils.clip_grad_norm_(self.params, float(self.cfg.get("grad_clip", 1.0)))
        if not math.isfinite(tot):
            # a non-finite LOSS is a real problem (counted; > 5% of steps aborts the job)
            self.skipped += 1
            self.opt.zero_grad(set_to_none=True)
            self.scaler.update()
            return None, toks
        if not torch.isfinite(gn):
            # finite loss but inf/nan gradients = normal fp16 loss-scale overflow: GradScaler
            # skips the optimizer step itself and lowers the scale (not counted as a failure)
            self.overflow += 1
        self.tech.after_backward(model, global_step, total_steps)
        self.scaler.step(self.opt)
        self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        if self.sched is not None:
            self.sched.step()
        return tot, toks


def make_optimizer(groups):
    for g in groups:
        g["initial_lr"] = g["lr"]
    return torch.optim.AdamW([{k: v for k, v in g.items() if k != "name"} for g in groups],
                             betas=(0.9, 0.98), eps=1e-8)


def make_scheduler(opt, total_steps: int, warmup_frac: float = 0.06):
    warm = max(1, int(round(warmup_frac * total_steps)))

    def f(step):
        if step < warm:
            return (step + 1) / warm
        return max(0.0, 1.0 - (step - warm) / max(1, total_steps - warm))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


# =============================================================================================
# throughput + time plan (section 8.4)
# =============================================================================================
def measure_throughput(ctx, stepper, d, n_steps: int = 6):
    """Training tokens/s with real batches (no optimizer change: grads are dropped) and
    inference tokens/s."""
    model = stepper.model
    tr = d["train"]
    eff = int(ctx.cfg["effective_batch"])
    rng = np.random.default_rng(1)
    order = rng.permutation(len(tr["ids"])).tolist()
    sel = order[:max(1, min(len(order), eff))]
    params = stepper.params

    def one():
        toks = 0
        for g in batches_by_chunks(tr["chunks"], stepper.micro, sel):
            b = collate_docs([tr["chunks"][i] for i in g], ctx.device)
            y = targets(tr["labels"], g, ctx.device)
            with stepper.amp_ctx():
                logits, _ = forward_batch(model, b)
            stepper.scaler.scale(ml_loss(logits.float(), y).mean()).backward()
            toks += b["n_tokens"]
        for p in params:
            p.grad = None
        return toks
    model.train()
    while True:
        try:
            one()                                  # warm-up (allocations, kernels)
            break
        except RuntimeError as e:
            if not is_oom(e):
                raise
            for p in params:
                p.grad = None
            torch.cuda.empty_cache()
            one_doc = max(len(tr["chunks"][i]) for i in sel)
            if stepper.micro <= one_doc:
                raise JobExit(7, "CUDA OOM with one document per micro-batch (pre-flight)")
            stepper.micro = max(one_doc, stepper.micro // 2)
            ctx.log(f"   [oom] pre-flight: micro-batch -> {stepper.micro} chunks")
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    toks = 0
    for _ in range(n_steps):
        toks += one()
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    r_train = toks / max(1e-6, time.time() - t)
    # inference speed
    dv = d["dev"]
    sub = list(range(min(len(dv["ids"]), 64)))
    predict_logits(model, [dv["chunks"][i] for i in sub[:8]], ctx.device, stepper.amp_ctx,
                   int(ctx.cfg["infer_chunks"]))                     # warm-up
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    t = time.time()
    ntok = sum(sum(len(c) for c in dv["chunks"][i]) for i in sub)
    predict_logits(model, [dv["chunks"][i] for i in sub], ctx.device, stepper.amp_ctx,
                   int(ctx.cfg["infer_chunks"]))
    if ctx.device.type == "cuda":
        torch.cuda.synchronize()
    r_inf = ntok / max(1e-6, time.time() - t)
    return r_train, max(r_inf, r_train)


def tokens_of(chunks, idx=None) -> int:
    it = chunks if idx is None else (chunks[i] for i in idx)
    return sum(sum(len(c) for c in ch) for ch in it)


def make_plan(ctx, d_by_K: Dict[int, Dict], r_train: float, r_inf: float) -> Dict:
    cfg = ctx.cfg
    budget = float(cfg["budget_s"])
    used = time.time() - ctx.t0
    margin = max(150.0, 0.05 * budget)
    avail = budget - used - margin
    e_max = int(cfg.get("epochs_max") or DEFAULT_EPOCHS.get(cfg["technique"], 4))
    e_min = int(cfg.get("min_epochs", 2))
    evals = int(cfg.get("evals_per_epoch", 2))
    rows = []
    choice = None
    fixed = cfg.get("fixed_plan")
    for K in sorted(d_by_K, reverse=True):
        d = d_by_K[K]
        ep = tokens_of(d["train"]["chunks"]) / r_train
        dev_inf = tokens_of(d["dev"]["chunks"]) / r_inf
        once = (tokens_of(d["test"]["chunks"]) + tokens_of(d["dev"]["chunks"])) / r_inf
        if cfg.get("is_teacher"):
            once += tokens_of(d["train"]["chunks"]) / r_inf
        per_epoch = ep + evals * dev_inf
        e_fit = int(max(0.0, avail - once) // max(1e-6, per_epoch))
        rows.append({"K": K, "epoch_min": per_epoch / 60, "inference_min": once / 60,
                     "epochs_fit": e_fit})
        if fixed:
            if K == max(d_by_K):
                choice = {"K": K, "epochs": int(fixed["epochs"]),
                          "train_fraction": float(fixed.get("frac", 1.0)), "reason": "fixed plan"}
            continue
        if choice is None and e_fit >= e_min:
            choice = {"K": K, "epochs": min(e_max, e_fit), "train_fraction": 1.0,
                      "reason": "fits" if K == max(d_by_K) else "K reduced to fit"}
    if choice is None:
        K = min(d_by_K)
        r = rows[-1]
        per_epoch = r["epoch_min"] * 60
        frac = (avail - r["inference_min"] * 60) / max(1e-6, e_min * per_epoch)
        min_frac = float(cfg.get("min_train_fraction", 0.3))
        if frac >= min_frac:
            choice = {"K": K, "epochs": e_min, "train_fraction": round(min(1.0, frac), 3),
                      "reason": "stratified subsample to fit"}
        else:
            e = 1 if (avail - r["inference_min"] * 60) >= min_frac * per_epoch else 0
            if e == 0:
                raise JobExit(2, f"not even {min_frac:.0%} of one epoch fits: available "
                                 f"{avail / 60:.1f} min, one epoch {per_epoch / 60:.1f} min")
            choice = {"K": K, "epochs": 1, "train_fraction": min_frac,
                      "reason": "minimum subsample, 1 epoch (time guard will cut if needed)"}
    d = d_by_K[choice["K"]]
    n_tr = len(d["train"]["ids"])
    n_used = len(stratified_subsample(d["train"]["labels"], choice["train_fraction"],
                                      int(cfg["seed"])))
    proj = (choice["epochs"] * (tokens_of(d["train"]["chunks"]) * n_used / max(1, n_tr) / r_train
                                + evals * tokens_of(d["dev"]["chunks"]) / r_inf)
            + (tokens_of(d["test"]["chunks"]) + tokens_of(d["dev"]["chunks"])) / r_inf + used)
    plan = {**choice, "K_requested": int(cfg["K"]), "budget_min": budget / 60,
            "available_min": avail / 60, "tokens_per_s_train": r_train,
            "tokens_per_s_infer": r_inf, "train_docs": n_tr, "train_docs_used": n_used,
            "projected_minutes": proj / 60, "options": rows, "epochs_max": e_max,
            "epochs_min": e_min, "micro_chunks": None}
    return plan


def print_plan(ctx, plan: Dict) -> None:
    ctx.log(f"   [plan] budget {plan['budget_min']:.0f} min, available {plan['available_min']:.0f} "
            f"min, train {plan['tokens_per_s_train']:,.0f} tok/s, infer "
            f"{plan['tokens_per_s_infer']:,.0f} tok/s")
    ctx.log("   [plan]   K  epoch(min)  inference(min)  epochs that fit")
    for r in plan["options"]:
        ctx.log(f"   [plan]  {r['K']:>2}  {r['epoch_min']:>10.1f}  {r['inference_min']:>14.1f}  "
                f"{r['epochs_fit']:>15}")
    flag = "" if plan["reason"] == "fits" else "   <<< " + plan["reason"].upper()
    ctx.log(f"   [plan] CHOSEN: K={plan['K']} epochs={plan['epochs']} train_fraction="
            f"{plan['train_fraction']} ({plan['train_docs_used']}/{plan['train_docs']} docs), "
            f"projected {plan['projected_minutes']:.0f} min{flag}")


# =============================================================================================
# evaluation during training
# =============================================================================================
def dev_eval(ctx, model, d, amp_ctx) -> Dict:
    """sigmoid probabilities on dev -> dev-only decision rule (global tau / cross-fitted
    per-label taus, >= 1 and <= kmax labels per document) -> Macro-F1."""
    dv = d["dev"]
    logits = predict_logits(model, dv["chunks"], ctx.device, amp_ctx,
                            int(ctx.cfg["infer_chunks"]))
    Pr = sigmoid_np(logits)
    G = dv["labels"]
    kmax = int(ctx.cfg.get("kmax", 10))
    rule = choose_rule(Pr, G, kmax, int(ctx.cfg["seed"]),
                       allow_perlabel=bool(ctx.cfg.get("tau_scan", True)))
    P = apply_rule(Pr, rule_taus(rule, Pr.shape[1]), kmax)
    s = ml_scores(G, P)
    s["accuracy"] = s["subset_acc"]
    L = Pr.shape[1]
    loss = float(ml_loss(torch.from_numpy(logits), torch.from_numpy(G.astype(np.float32)))
                 .mean().item())
    return {"p1": Pr, "logits": logits, "tau": float(rule["tau_global"]), "rule": rule,
            "scores": s, "f1": s["macro_f1"],
            "f1_at_0p5": macro(G, apply_rule(Pr, np.full(L, 0.5, np.float32), kmax)),
            "dev_loss": loss, "p1_std": float(Pr.std())}


# =============================================================================================
# checkpoints
# =============================================================================================
def save_best(ctx, model) -> None:
    p = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    full = ctx.cfg["technique"] in ("full_ft", "kd", "student_hard", "partial_ft")
    torch.save(trainable_state_dict(model, half=full), p + ".tmp")
    os.replace(p + ".tmp", p)


def save_last(ctx, model, opt, sched, scaler, st: Dict) -> None:
    p = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    os.makedirs(os.path.dirname(p), exist_ok=True)
    try:
        disk_guard(os.path.dirname(ctx.run_dir), 3.0, log=ctx.log,
                   keep=os.path.join(ctx.run_dir, "checkpoints"))
    except DiskFull as e:
        ctx.log(f"   [disk] {e}: skipping last.pt")
        return
    obj = {"model": trainable_state_dict(model), "opt": opt.state_dict(),
           "sched": sched.state_dict(), "scaler": scaler.state_dict(), "st": st,
           "rng": {"py": random.getstate(), "np": np.random.get_state(),
                   "torch": torch.get_rng_state()}}
    torch.save(obj, os.path.join(ctx.run_dir, "checkpoints", "last.tmp"))
    os.replace(os.path.join(ctx.run_dir, "checkpoints", "last.tmp"), p)


# =============================================================================================
# the job
# =============================================================================================
def run_train(ctx: Ctx, mode: str) -> int:
    cfg = ctx.cfg
    set_seed(int(cfg["seed"]))
    dev = ctx.device
    amp_dtype, amp_ctx = amp_setup(dev)
    ctx.status("preflight")
    ctx.log(f"start {cfg['experiment']}/{cfg['technique']}/{cfg['model']} on {dev}"
            f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}, "
            f"budget {cfg['budget_s'] / 60:.0f} min, attempt {cfg.get('attempt', 1)}")
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    inj = os.environ.get("PHASE5_INJECT_FAIL", "")          # tests only: "<job>:<exit code>"
    if inj and inj.split(":")[0] == cfg["job"] and int(cfg.get("attempt", 1)) == 1:
        raise JobExit(int(inj.split(":")[1]), "injected failure (test)")

    # ---- data (all candidate K values, for the planner) ---------------------------------
    K0 = int(cfg["K"])
    reducible = cfg["doc_repr"] in ("hier", "first_k", "last_k")
    Ks = [K0] + ([k for k in (2, 1) if k < K0] if reducible and mode == "train" else [])
    d_by_K = {K: load_split_docs(cfg, K) for K in Ks}
    d = d_by_K[K0]
    integ = assert_data_integrity(d, cfg)
    write_json(os.path.join(ctx.run_dir, "logs", "integrity.json"), integ)
    ctx.log(f"   data: train {len(d['train']['ids'])}, dev {len(d['dev']['ids'])}, test "
            f"{len(d['test']['ids'])} docs; repr {cfg['doc_repr']} K={K0}; integrity ok")

    # ---- teacher logits (KD students) ---------------------------------------------------
    tmap = None
    if cfg["technique"] == "kd":
        tl = cfg.get("teacher_logits")
        if not tl or not os.path.exists(tl):
            raise JobExit(1, f"teacher logits not found ({tl})")
        tmap = teacher_map(tl, "train")
        miss = [i for i in d["train"]["ids"] if i not in tmap]
        if miss:
            raise JobExit(1, f"{len(miss)} train docs have no teacher logits")

    # ---- model ---------------------------------------------------------------------------
    model, _ = build_doc_model(cfg["phase1_dir"], cfg["attn_mode"], cfg.get("layer_ids"),
                               float(cfg.get("dropout", 0.1)), len(cfg["label_ids"]),
                               prior_logit(d["train"]["labels"]))
    need_base = (not cfg.get("skip_job_checks")) and cfg["technique"] in (
        "lorma", "lora", "adalora", "adapters")
    base = copy.deepcopy(model) if need_base else None
    groups, tech = apply_technique(model, cfg["technique"], cfg, log=ctx.log)
    model.to(dev)
    assert_one_device(model, dev)
    size = model_size(model)
    write_json(os.path.join(ctx.run_dir, "logs", "model_size.json"), size)

    checks = {"skipped": True, "selfcheck": "skipped"}
    if not cfg.get("skip_job_checks"):
        if base is not None:
            base.to(dev)
        checks = job_checks(base, model, tech, cfg, d, dev, amp_ctx)
        del base
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    write_json(os.path.join(ctx.run_dir, "logs", "job_checks.json"), checks)
    for c in checks.get("checks", []):
        v = c.get("value")
        ctx.log(f"   [check] {c['name']:<28} {c['kind']:<9} {c['verdict']}"
                + (f" value={v:.3g} thr={c.get('threshold'):.3g}" if isinstance(v, float) else "")
                + (f"  {c.get('error', '')[:200]}" if c["verdict"] == "fail" else ""))
    if checks.get("critical_failed"):
        raise JobExit(6, f"critical self-check failed: {checks['critical_failed']}")
    if checks.get("advisory_failed"):
        ctx.log(f"   WARNING advisory self-checks: {checks['advisory_failed']} (continuing)")

    tr_all = d["train"]
    cw = class_weights(tr_all["labels"], cfg.get("class_weight", "sqrt_inv"), dev)
    opt = make_optimizer(groups)
    scaler = make_scaler(amp_dtype == torch.float16)
    stepper = Stepper(ctx, model, opt, None, scaler, tech, amp_ctx, cw, cfg, tmap)

    if mode == "overfit":
        return run_overfit(ctx, stepper, d)

    # ---- pre-flight throughput + plan ----------------------------------------------------
    plan_path = os.path.join(ctx.run_dir, "logs", "plan.json")
    last_pt = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    resume = None
    if os.path.exists(last_pt) and os.path.exists(plan_path) and cfg.get("resume", True):
        try:
            resume = torch_load(last_pt)
            plan = read_json(plan_path)
            ctx.log(f"   [resume] from step {resume['st']['step']} (plan reused)")
        except Exception as e:
            ctx.log(f"   [resume] unreadable last.pt ({e}); starting fresh")
            resume = None
    if resume is None:
        r_train, r_inf = measure_throughput(ctx, stepper, d)
        ctx.log(f"   [preflight] train {r_train:,.0f} tok/s, inference {r_inf:,.0f} tok/s, "
                f"micro-batch {stepper.micro} chunks, peak {peak_gb():.2f} GB")
        if mode == "smoke":
            write_json(os.path.join(ctx.run_dir, "logs", "smoke.json"),
                       {"tokens_per_s_train": r_train, "tokens_per_s_infer": r_inf,
                        "micro_chunks": stepper.micro, "peak_mem_gb": peak_gb(),
                        "selfcheck": checks.get("selfcheck"), **size})
            ctx.status("smoke_ok", "smoke ok")
            return 0
        plan = make_plan(ctx, d_by_K, r_train, r_inf)
        plan["micro_chunks"] = stepper.micro
        plan["selfcheck"] = checks.get("selfcheck")
        write_json(plan_path, plan)
    print_plan(ctx, plan)
    stepper.micro = int(plan.get("micro_chunks") or stepper.micro)
    if plan["K"] != K0:
        d = d_by_K[plan["K"]]
    del d_by_K

    keep = stratified_subsample(d["train"]["labels"], float(plan["train_fraction"]),
                                int(cfg["seed"]))
    eff = int(cfg["effective_batch"])
    steps_per_epoch = max(1, math.ceil(len(keep) / eff))
    total_steps = steps_per_epoch * int(plan["epochs"])
    sched = make_scheduler(opt, total_steps, float(cfg.get("warmup_frac", 0.06)))
    stepper.sched = sched
    evals = int(cfg.get("evals_per_epoch", 2))
    eval_at = sorted({max(1, round(steps_per_epoch * (k + 1) / evals)) for k in range(evals)})
    patience = int(cfg.get("patience", 2)) * evals

    st = {"step": 0, "epoch": 0, "pos": 0, "best_f1": -1.0, "best_epoch": None, "bad": 0,
          "evals": [], "skipped": 0, "train_seconds": 0.0}
    if resume is not None:
        load_trainable_state(model, resume["model"])
        opt.load_state_dict(resume["opt"])
        sched.load_state_dict(resume["sched"])
        scaler.load_state_dict(resume["scaler"])
        st.update(resume["st"])
        try:
            random.setstate(resume["rng"]["py"])
            np.random.set_state(resume["rng"]["np"])
            torch.set_rng_state(resume["rng"]["torch"])
        except Exception:
            pass
        with open(os.path.join(ctx.run_dir, "logs", "resume.json"), "w") as f:
            json.dump({"resumed_at_step": st["step"]}, f)
        del resume

    # inference reserve after the training deadline: final dev + test (+ teacher train logits)
    r_inf = float(plan["tokens_per_s_infer"])
    reserve = (tokens_of(d["dev"]["chunks"]) + tokens_of(d["test"]["chunks"])) / r_inf
    if cfg.get("is_teacher"):
        reserve += tokens_of(d["train"]["chunks"]) / r_inf
    reserve = 1.25 * reserve + 90.0
    deadline = ctx.t0 + float(cfg["budget_s"]) - reserve
    ctx.status("training", plan=plan["reason"])
    ctx.log(f"   [train] {len(keep)} docs/epoch, {steps_per_epoch} steps/epoch x "
            f"{plan['epochs']} epochs = {total_steps} steps; eval at steps {eval_at} of each "
            f"epoch; training deadline in {(deadline - time.time()) / 60:.0f} min")

    last_ckpt = time.time()
    t_train = time.time()
    stop_reason = "completed"
    kill_at = int(cfg.get("kill_at_step", -1))
    ema = None
    while st["epoch"] < int(plan["epochs"]):
        rng = np.random.default_rng(int(cfg["seed"]) + 1000 * st["epoch"])
        order = [keep[i] for i in rng.permutation(len(keep))]
        while st["pos"] < steps_per_epoch:
            if time.time() > deadline:
                stop_reason = "time guard"
                break
            s_in = st["pos"]
            batch = order[s_in * eff:(s_in + 1) * eff]
            if not batch:
                st["pos"] = steps_per_epoch
                break
            loss, toks = stepper.step(d["train"], batch, st["step"], total_steps)
            st["step"] += 1
            st["pos"] += 1
            if loss is None:
                st["skipped"] += 1
                if st["step"] >= 20 and st["skipped"] > 0.05 * st["step"]:
                    raise JobExit(4, f"{st['skipped']} of {st['step']} steps non-finite")
            else:
                ema = loss if ema is None else 0.98 * ema + 0.02 * loss
            if st["step"] % 50 == 0:
                ctx.log(f"   step {st['step']}/{total_steps} ep {st['epoch'] + 1} loss "
                        f"{ema:.4f} lr {sched.get_last_lr()[0]:.2e} peak {peak_gb():.1f} GB "
                        f"skipped {st['skipped']} fp16-overflow {stepper.overflow}")
            if st["step"] == kill_at:
                save_last(ctx, model, opt, sched, scaler, st)
                os._exit(137)
            if st["pos"] in eval_at:
                ev = dev_eval(ctx, model, d, amp_ctx)
                frac_ep = st["epoch"] + st["pos"] / steps_per_epoch
                rec = {"type": "eval", "step": st["step"], "epoch": round(frac_ep, 3),
                       "train_loss": ema, "dev_loss": ev["dev_loss"], "dev_macro_f1": ev["f1"],
                       "dev_acc": ev["scores"]["accuracy"], "tau": ev["tau"],
                       "f1_at_0p5": ev["f1_at_0p5"], "lr": sched.get_last_lr()[0],
                       "minutes": (time.time() - ctx.t0) / 60, "skipped_steps": st["skipped"]}
                ctx.jlog(rec)
                st["evals"].append(rec)
                sel = tech.selectable(st["step"], total_steps)
                ctx.log(f"   [eval] epoch {frac_ep:.2f}: dev macro-F1 {ev['f1']:.4f} (tau "
                        f"{ev['tau']:.2f}; @0.5 {ev['f1_at_0p5']:.4f}) loss {ev['dev_loss']:.4f}"
                        f"{'' if sel else ' (not selectable yet)'}")
                if sel and ev["f1"] > st["best_f1"] + 1e-9:
                    st["best_f1"], st["best_epoch"], st["bad"] = ev["f1"], round(frac_ep, 3), 0
                    save_best(ctx, model)
                elif sel:
                    st["bad"] += 1
                write_json(os.path.join(ctx.run_dir, "logs", "progress.json"),
                           {"best_dev_macro_f1": st["best_f1"], "best_epoch": st["best_epoch"],
                            "step": st["step"], "total_steps": total_steps,
                            "epochs_run": round(frac_ep, 3)})
                if st["epoch"] == 0 and st["pos"] == eval_at[-1] and ev["f1"] < 0.40 \
                        and ev["p1_std"] < 1e-3:
                    raise JobExit(5, f"watchdog: dev macro-F1 {ev['f1']:.3f} with a constant "
                                     f"output after epoch 1")
                if sel and st["bad"] >= patience:
                    stop_reason = "early stopping"
                    break
            if time.time() - last_ckpt > 60 * float(cfg.get("ckpt_every_min", 30)):
                save_last(ctx, model, opt, sched, scaler, st)
                last_ckpt = time.time()
        if stop_reason != "completed":
            break
        st["epoch"] += 1
        st["pos"] = 0
    st["train_seconds"] = time.time() - t_train
    ctx.log(f"   [train] stopped: {stop_reason} at step {st['step']}/{total_steps}")

    # ---- final model ---------------------------------------------------------------------
    best_pt = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    tech.force_final(model)
    if not os.path.exists(best_pt) or (cfg["technique"] == "adalora" and st["best_f1"] < 0):
        ev = dev_eval(ctx, model, d, amp_ctx)
        st["best_f1"], st["best_epoch"] = ev["f1"], st["epoch"] + st["pos"] / steps_per_epoch
        save_best(ctx, model)
        ctx.log(f"   [final] no selectable checkpoint during training -> current weights "
                f"(dev macro-F1 {ev['f1']:.4f})")
    load_trainable_state(model, torch_load(best_pt))
    lp = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    if os.path.exists(lp):
        os.remove(lp)
    return finish(ctx, model, tech, d, amp_ctx, st, plan, size, checks, stop_reason)


def finish(ctx, model, tech, d, amp_ctx, st, plan, size, checks, stop_reason) -> int:
    cfg = ctx.cfg
    ctx.status("evaluating")
    t = time.time()
    ev = dev_eval(ctx, model, d, amp_ctx)
    snips = load_snippets(cfg)
    label_ids = [int(x) for x in cfg["label_ids"]]
    tsup = d["train"]["labels"].sum(0)
    rep = write_dev_outputs(ctx.run_dir, d["dev"]["ids"], d["dev"]["labels"], ev["p1"],
                            ev["rule"], d["dev"]["n_tokens"], snips, label_ids, tsup,
                            int(d["dev"].get("dev_only_pos", 0)))
    ctx.status("predicting")
    tl = predict_logits(model, d["test"]["chunks"], ctx.device, amp_ctx, int(cfg["infer_chunks"]))
    dt = time.time() - t
    p_test = sigmoid_np(tl)
    vrep = write_submission(ctx.run_dir, d["test"]["ids"], p_test, ev["rule"], label_ids)
    write_json(os.path.join(ctx.run_dir, "dev", "rule.json"), ev["rule"])
    if cfg.get("is_teacher"):
        trl = predict_logits(model, d["train"]["chunks"], ctx.device, amp_ctx,
                             int(cfg["infer_chunks"]))
        save_teacher_logits(os.path.join(ctx.run_dir, "teacher_logits.npz"),
                            {"train": {"ids": d["train"]["ids"], "logits": trl},
                             "dev": {"ids": d["dev"]["ids"], "logits": ev["logits"]},
                             "test": {"ids": d["test"]["ids"], "logits": tl}})
        ctx.log("   [teacher] logits cached for train / dev / test")
    n_docs = len(d["dev"]["ids"]) + len(d["test"]["ids"])
    s = rep["scores"]
    r = ev["rule"]
    metrics = {
        "dev": s, "dev_macro_f1": s["macro_f1"], "dev_macro_f1_at_0p5":
            rep["scores_at_0p5"]["macro_f1"], "dev_macro_f1_global":
            rep["scores_global"]["macro_f1"], "rule": r["name"], "tau": r["tau_global"],
        "kmax": r["kmax"], "cv_macro_global": r.get("cv_macro_global"),
        "cv_macro_perlabel": r.get("cv_macro_perlabel"), "ece": rep["ece"],
        "n_labels": len(label_ids), "dev_only_labels": int(d["dev"].get("dev_only_pos", 0)),
        "best_epoch": st["best_epoch"],
        "epochs_run": st["evals"][-1]["epoch"] if st["evals"] else 0,
        "steps": st["step"], "skipped_steps": st["skipped"], "stop_reason": stop_reason,
        "train_minutes": st["train_seconds"] / 60, "total_minutes": (time.time() - ctx.t0) / 60,
        "docs_per_s_infer": n_docs / max(1e-6, dt), "ms_per_doc_infer": 1000 * dt / max(1, n_docs),
        "peak_mem_gb": peak_gb(), "test_mean_labels": vrep.get("mean_labels"),
        "K_effective": plan["K"],
        "train_fraction": plan["train_fraction"], "epochs_planned": plan["epochs"],
        "train_docs_used": plan["train_docs_used"],
        "tokens_per_s": plan["tokens_per_s_train"], "projected_minutes":
            plan["projected_minutes"], "selfcheck": checks.get("selfcheck"),
        "validation_ok": vrep["ok"], "degenerate": vrep.get("degenerate"),
        "technique_report": tech.final_report(model), **size,
    }
    write_json(os.path.join(ctx.run_dir, "metrics.json"), metrics)
    if not vrep["ok"]:
        ctx.status("failed", f"submission validation failed: {vrep['errors']}")
        return 4
    if vrep.get("degenerate"):
        # multi-label: still a valid submission; keep it (ranked by dev macro-F1 like any other
        # run) so a weak model never leaves an experiment without an upload. Flagged in the log,
        # metrics.json ("degenerate": true) and the CSV selfcheck column.
        ctx.log("   !!! WARNING: every test document got the same label set (degenerate)")
    write_json(os.path.join(ctx.run_dir, "DONE.json"),
               {"finished": now_str(), "dev_macro_f1": s["macro_f1"], "rule": r["name"],
                "stop_reason": stop_reason, "submission": os.path.join(
                    ctx.run_dir, "submission", "submission.zip")})
    ctx.status("done", f"dev macro-F1 {s['macro_f1']:.4f}")
    ctx.log(f"   DONE dev macro-F1 {s['macro_f1']:.4f} (micro {s['micro_f1']:.4f}, subset acc "
            f"{s['subset_acc']:.4f}, rare {s.get('macro_f1_rare')}, rule {r['name']} "
            f"tau {r['tau_global']:.2f}, cv global {r.get('cv_macro_global')} vs per-label "
            f"{r.get('cv_macro_perlabel')}); test mean labels {vrep.get('mean_labels'):.2f}; "
            f"submission validated")
    return 0


def load_snippets(cfg: Dict) -> Dict[str, str]:
    p = cfg.get("snippets_path")
    if p and os.path.exists(p):
        return read_json(p, {}) or {}
    return {}


def run_overfit(ctx, stepper, d) -> int:
    """32 train docs, effective batch 8, up to 100 steps: loss must go below 0.1."""
    tr = d["train"]
    idx = list(range(min(32, len(tr["ids"]))))
    for g in stepper.opt.param_groups:
        g["lr"] = g["initial_lr"] = max(g["lr"], float(ctx.cfg.get("overfit_lr", 3e-3)))
    stepper.sched = None
    last = None
    n = 0
    for n in range(1, 101):
        rng = np.random.default_rng(n)
        order = [idx[i] for i in rng.permutation(len(idx))]
        losses = []
        for k in range(0, len(order), 8):
            l, _ = stepper.step(tr, order[k:k + 8], n, 100)
            if l is not None:
                losses.append(l)
        last = float(np.mean(losses)) if losses else float("nan")
        if last < 0.1:
            break
    sub = {k: (v[idx] if k == "labels" else [v[i] for i in idx]) for k, v in tr.items()
           if k in ("ids", "labels", "chunks")}
    _dt, amp_ctx = amp_setup(ctx.device)
    Pr = sigmoid_np(predict_logits(stepper.model, sub["chunks"], ctx.device, amp_ctx, 32))
    f_fit = macro(sub["labels"], Pr >= 0.5)
    res = {"passed": bool((last is not None and last < 0.1) or f_fit >= 0.9),
           "final_loss": last, "macro_f1_on_fit_docs": f_fit, "epochs": n, "n_docs": len(idx)}
    write_json(os.path.join(ctx.run_dir, "logs", "overfit.json"), res)
    ctx.status("done", f"overfit {res}")
    return 0 if res["passed"] else 1


def run_teacher_reuse(ctx: Ctx) -> int:
    cfg = ctx.cfg
    dev = ctx.device
    _dt, amp_ctx = amp_setup(dev)
    src = cfg["reuse_from"]
    rc = read_json(os.path.join(src, "logs", "run_config.json"))
    model = rebuild_trained(rc, os.path.join(src, "checkpoints", "best.pt"), cfg["phase1_dir"],
                            dev)
    d = load_split_docs(cfg)
    logits = {s: predict_logits(model, d[s]["chunks"], dev, amp_ctx, int(cfg["infer_chunks"]))
              for s in ("train", "dev", "test")}
    save_teacher_logits(os.path.join(ctx.run_dir, "teacher_logits.npz"),
                        {s: {"ids": d[s]["ids"], "logits": logits[s]} for s in logits})
    st = {"best_epoch": None, "epoch": 0, "pos": 0, "step": 0, "skipped": 0, "evals": [],
          "train_seconds": 0.0}
    rm = read_json(os.path.join(src, "metrics.json"), {}) or {}
    plan = {"K": cfg["K"], "train_fraction": rm.get("train_fraction", 1.0),
            "epochs": rm.get("epochs_planned"), "train_docs_used": rm.get("train_docs_used"),
            "tokens_per_s_train": rm.get("tokens_per_s"), "projected_minutes": 0.0}
    ctx.log(f"   [teacher] reused finished E1 run {src}")
    return finish(ctx, model, __import__("cjpe_peft").Technique(), d, amp_ctx, st, plan,
                  model_size(model), {"selfcheck": "reused"}, f"reused {src}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", default="train", choices=["train", "smoke", "overfit",
                                                        "teacher_reuse"])
    a = ap.parse_args()
    cfg = read_json(os.path.join(a.run_dir, "logs", "run_config.json"))
    ctx = Ctx(a.run_dir, cfg)
    code, reason = 1, "unknown"
    try:
        code = run_teacher_reuse(ctx) if a.mode == "teacher_reuse" else run_train(ctx, a.mode)
        reason = "ok" if code == 0 else f"exit {code}"
    except JobExit as e:
        code, reason = e.code, e.reason
        ctx.log(f"   JOB EXIT {code}: {reason}")
    except Exception as e:
        code = 7 if is_oom(e) else 1
        reason = f"{type(e).__name__}: {e}"
        ctx.log(f"   JOB ERROR: {reason}\n{traceback.format_exc()[-3000:]}")
    finally:
        st = read_json(os.path.join(a.run_dir, "status.json"), {}) or {}
        if code != 0 and st.get("state") not in ("failed",):
            ctx.status("failed", reason, exit_code=code)
        elif code == 0 and st.get("state") not in ("done", "degenerate", "smoke_ok"):
            ctx.status("done", reason)
    sys.stdout.flush()
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase6/gen_ml.py
"""
gen_ml.py -- Phase-6 (abstractive summarisation) text utilities and metrics. Torch-free.

  sentence_spans   split a summary into sentences at TOKEN-SAFE boundaries (whitespace preceded
                   by non-whitespace), so encode(s1)+encode(s2)+... == encode(summary)
  align_sentences  DANCER-style alignment (Gidiotis & Tsoumakas 2020): every reference sentence
                   goes to the document chunk with the highest content-word recall
  rouge_n / rouge_l  pure-Python ROUGE-1/2/L F1 (lower-cased alphanumeric tokens, no stemmer);
                   LCS by bit-parallel big-integer arithmetic (Hyyroe 2004) -> fast on long texts
  bertscore        BERTScore F1 via the `bert-score` package if importable (roberta-large, the
                   package default for lang="en"); returns None + reason otherwise (never raises)
  diagnostics      length, compression, repeated-trigram rate (redundancy), novel bigrams
                   (abstractiveness), source support of content words, number / citation
                   precision against the source (factuality proxy)
  length_curve     dev-tuned word cap: truncating at sentence boundaries to r x median train
                   summary length; the best r on dev is applied to test (generation length control)
"""
import math
import re
from collections import Counter
from typing import Dict, List, Optional, Sequence, Tuple

WORD_RE = re.compile(r"[a-z0-9]+")
SENT_END_RE = re.compile(r"[.!?;:][\"')\]]*(\s+)|\n\s*\n(\s*)")
NUM_RE = re.compile(r"\b\d[\d,./-]*\d\b|\b\d\b")
CITE_RE = re.compile(
    r"(?i)\b(?:section|sec\.|s\.|sections|article|art\.|rule|order)\s*\d+[a-z]?\b"
    r"|\b(?:i\.?\s?p\.?\s?c|cr\.?\s?p\.?\s?c|c\.?\s?p\.?\s?c)\b|\bact,?\s*(?:18|19|20)\d\d\b")
STOP = set("""a an the of to in and or for on at by with from as is are was were be been being
it its this that these those which who whom whose what when where why how not no nor but if then
than so such into onto upon over under about above below between through during before after
he she they them his her their we our you your i me my also any all each other some can could
would should may might must shall will has have had do does did there here very more most""".split())


def words(text: str) -> List[str]:
    return WORD_RE.findall(text.lower())


def content_words(text: str) -> List[str]:
    return [w for w in words(text) if w not in STOP and (len(w) > 2 or w.isdigit())]


# =============================================================================================
# sentences + alignment
# =============================================================================================
def sentence_spans(text: str) -> List[str]:
    """Pieces whose concatenation is exactly `text`; every cut is at a whitespace character that
    follows a non-whitespace character (a BPE pre-token boundary)."""
    cuts = []
    for m in SENT_END_RE.finditer(text):
        g = 1 if m.group(1) is not None else 2
        p = m.start(g) if m.group(g) else m.start()
        while p > 0 and text[p - 1].isspace():
            p -= 1
        if 0 < p < len(text) and text[p].isspace() and not text[p - 1].isspace():
            if not cuts or p > cuts[-1]:
                cuts.append(p)
    out, s = [], 0
    for c in cuts:
        out.append(text[s:c])
        s = c
    out.append(text[s:])
    return [x for x in out if x]


def align_sentences(sent_texts: Sequence[str], chunk_texts: Sequence[str]) -> List[int]:
    """Chunk index for every sentence: argmax of content-word recall (ties -> earliest chunk);
    sentences without content words follow the previous sentence."""
    csets = [set(content_words(t)) for t in chunk_texts]
    out, prev = [], 0
    for s in sent_texts:
        w = set(content_words(s))
        if not w or not csets:
            out.append(prev)
            continue
        sc = [len(w & c) for c in csets]
        best = max(range(len(sc)), key=lambda i: (sc[i], -i))
        prev = best if sc[best] > 0 else prev
        out.append(prev)
    return out


# =============================================================================================
# ROUGE
# =============================================================================================
def _ngrams(t: List[str], n: int) -> Counter:
    return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))


def _prf(hit: float, n_c: int, n_r: int) -> Dict[str, float]:
    p = hit / n_c if n_c else 0.0
    r = hit / n_r if n_r else 0.0
    return {"p": p, "r": r, "f": (2 * p * r / (p + r)) if p + r > 0 else 0.0}


def rouge_n(cand: str, ref: str, n: int = 1) -> Dict[str, float]:
    c, r = _ngrams(words(cand), n), _ngrams(words(ref), n)
    hit = sum((c & r).values())
    return _prf(hit, sum(c.values()), sum(r.values()))


def lcs_len(a: List[str], b: List[str]) -> int:
    """Bit-parallel LCS length (Hyyroe 2004): V <- (V + (V & M_t)) | (V - (V & M_t))."""
    if not a or not b:
        return 0
    masks: Dict[str, int] = {}
    for i, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << i)
    full = (1 << len(a)) - 1
    V = full
    for t in b:
        U = V & masks.get(t, 0)
        V = ((V + U) | (V - U)) & full
    return len(a) - bin(V).count("1")


def rouge_l(cand: str, ref: str) -> Dict[str, float]:
    c, r = words(cand), words(ref)
    return _prf(lcs_len(c, r), len(c), len(r))


def rouge_all(cands: List[str], refs: List[str]) -> Dict[str, float]:
    if not cands:
        return {"rouge1": None, "rouge2": None, "rougeL": None}
    r1 = [rouge_n(c, r, 1)["f"] for c, r in zip(cands, refs)]
    r2 = [rouge_n(c, r, 2)["f"] for c, r in zip(cands, refs)]
    rl = [rouge_l(c, r)["f"] for c, r in zip(cands, refs)]
    m = lambda x: float(sum(x) / len(x))  # noqa: E731
    return {"rouge1": m(r1), "rouge2": m(r2), "rougeL": m(rl), "rouge1_each": r1,
            "rougeL_each": rl}


# =============================================================================================
# BERTScore
# =============================================================================================
_SCORER = {}


def bertscore(cands: List[str], refs: List[str], device: str = "cpu", batch_size: int = 16,
              log=print) -> Tuple[Optional[List[float]], str]:
    """(per-pair F1 list, info) or (None, reason). Uses the package defaults for lang='en'
    (roberta-large, layer 17, no baseline rescaling) like a default evaluator would."""
    if not cands:
        return None, "no candidates"
    try:
        from bert_score import BERTScorer
    except Exception as e:                                   # package missing
        return None, f"bert-score not importable ({type(e).__name__})"
    try:
        key = (device, batch_size)
        if key not in _SCORER:
            _SCORER[key] = BERTScorer(lang="en", device=device, batch_size=batch_size)
        c = [x if x.strip() else "." for x in cands]
        r = [x if x.strip() else "." for x in refs]
        _p, _r, f = _SCORER[key].score(c, r, verbose=False)
        vals = [float(v) for v in f.tolist()]
        if any(math.isnan(v) for v in vals):
            return None, "BERTScore returned NaN"
        return vals, "bert-score roberta-large (lang=en)"
    except Exception as e:                                   # no internet / OOM / ...
        return None, f"bert-score failed: {type(e).__name__}: {str(e)[:200]}"


def free_bertscore() -> None:
    _SCORER.clear()


# =============================================================================================
# length control + diagnostics
# =============================================================================================
def cap_words(text: str, n_words: Optional[int]) -> str:
    """Keep whole sentences while the word count stays <= n_words (always >= 1 sentence)."""
    if not n_words or len(words(text)) <= n_words:
        return text
    out, n = [], 0
    for s in sentence_spans(text):
        k = len(words(s))
        if out and n + k > n_words:
            break
        out.append(s)
        n += k
    return "".join(out).strip()


LENGTH_RATIOS = (0.5, 0.75, 1.0, 1.25, None)          # x median train summary words; None = no cap


def diagnostics(summ: str, source: Optional[str] = None, ref: Optional[str] = None) -> Dict:
    w = words(summ)
    tri = _ngrams(w, 3)
    n_tri = sum(tri.values())
    d = {"words": len(w), "sentences": len(sentence_spans(summ.strip())) if summ.strip() else 0,
         "repeated_trigram_rate": (1.0 - len(tri) / n_tri) if n_tri else 0.0}
    if source is not None:
        sw = words(source)
        sb = set(_ngrams(sw, 2))
        cb = list(_ngrams(w, 2))
        d["compression"] = len(w) / max(1, len(sw))
        d["novel_bigram_rate"] = (sum(1 for b in cb if b not in sb) / len(cb)) if cb else 0.0
        cw = content_words(summ)
        sset = set(sw)
        d["content_support"] = (sum(1 for x in cw if x in sset) / len(cw)) if cw else 1.0
        nums = set(NUM_RE.findall(summ))
        snums = set(NUM_RE.findall(source))
        d["number_precision"] = (len(nums & snums) / len(nums)) if nums else None
        cites = {re.sub(r"\s+", " ", c.lower()) for c in CITE_RE.findall(summ)}
        scites = {re.sub(r"\s+", " ", c.lower()) for c in CITE_RE.findall(source)}
        d["citation_precision"] = (len(cites & scites) / len(cites)) if cites else None
    if ref is not None:
        rw = set(content_words(ref))
        d["ref_content_recall"] = (len(rw & set(content_words(summ))) / len(rw)) if rw else None
        d["ref_words"] = len(words(ref))
    return d


def mean_of(rows: List[Dict], key: str) -> Optional[float]:
    v = [r[key] for r in rows if r.get(key) is not None]
    return float(sum(v) / len(v)) if v else None


def primary(rec: Dict) -> Tuple[Optional[float], str]:
    """The model-selection score of a dev evaluation: BERTScore F1 when available, else the mean
    of ROUGE-1 and ROUGE-L F1 (reported with its name so runs are never mixed silently)."""
    if rec.get("bertscore_f1") is not None:
        return float(rec["bertscore_f1"]), "bertscore_f1"
    if rec.get("rouge1") is not None:
        return 0.5 * (float(rec["rouge1"]) + float(rec["rougeL"])), "rouge1_rougeL_mean"
    return None, "none"


def consensus_pick(cands: List[str]) -> int:
    """MBR-style ensemble: index of the candidate with the highest mean ROUGE-L F1 to the others."""
    if len(cands) <= 1:
        return 0
    best, bi = -1.0, 0
    for i, c in enumerate(cands):
        s = sum(rouge_l(c, o)["f"] for j, o in enumerate(cands) if j != i) / (len(cands) - 1)
        if s > best:
            best, bi = s, i
    return bi


In [ ]:
%%writefile /kaggle/working/phase6/summ_model.py
"""
summ_model.py -- the Phase-1 causal LM as a summariser + the Phase-6 techniques + decoding.

SummLM keeps the Phase-1 LM intact (its own forward, KV cache, positions, ALiBi, sliding window)
and registers a `Trunk` VIEW of the same blocks so the tested Phase-2..5 PEFT code
(cjpe_peft.apply_technique: LoRMA+, LoRA, AdaLoRA, Houlsby adapters, full FT, ...) inserts its
modules into the very blocks the LM runs. `classifier` is an empty module (no new head: the
tied LM head generates the summary).

New techniques (Phase-6 list, PDF 12.2.1):
  pfeiffer  Pfeiffer adapter: ONE bottleneck (d/16) after the FFN output of every block + norms
  qlora     LoRA r=16 on top of NF4-quantised frozen linears (blockwise absmax, block 64, two
            4-bit codes per byte; Dettmers et al. 2023), dequantised on the fly
  prefix    prefix tuning (Li & Liang 2021): P learned key/value vectors per layer, used as the
            KV-cache "past"; initialised from the frozen model's own keys/values of real text
  prompt    prompt tuning (Lester et al. 2021): P learned input embeddings; their keys/values are
            computed through the frozen network on every forward (exactly "prepend P embeddings"
            for a causal decoder), initialised from real token embeddings
Training loss: token cross-entropy on the SUMMARY tokens only; logits are computed only at the
positions that predict a summary token (SelectHead) -> no (B, T, V) tensor.
"""
import math
import os
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from ner_models import Trunk, count_params, load_phase1_lm, torch_load
from ner_peft import ADAPTER_SITES, AdapterWrapped, LoRALinear, _get, _groups, _set
from phase1_models import _banned_ngram_tokens

EOS_ID, PAD_ID = 0, 1
CTX = 1024
BLOCK_ATTR = {"gpt2": "blocks", "llama": "layers", "bloom": "blocks"}
NEW_TECHNIQUES = ("pfeiffer", "qlora", "prefix", "prompt")
NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
       -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
       0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]


class SelectHead(nn.Module):
    """lm_head that (when `sel` is set) only computes logits at the selected positions."""

    def __init__(self, head: nn.Linear):
        super().__init__()
        self.head = head
        self.sel = None

    def forward(self, x):
        if self.sel is not None:
            x = x[self.sel]
        return self.head(x)


# =============================================================================================
# virtual tokens (prefix / prompt tuning)
# =============================================================================================
def _attn_dims(trunk) -> Tuple[int, int]:
    a = trunk.blocks[0]
    if trunk.kind == "gpt2":
        return a.attn.n_head, a.attn.head_dim
    if trunk.kind == "llama":
        return a.attn.kv, a.attn.hd
    return a.self_attention.h, a.self_attention.hd


class Prefix(nn.Module):
    def __init__(self, past: List[Tuple[torch.Tensor, torch.Tensor]]):
        super().__init__()
        self.k = nn.ParameterList([nn.Parameter(k[0].detach().float().clone()) for k, _ in past])
        self.v = nn.ParameterList([nn.Parameter(v[0].detach().float().clone()) for _, v in past])
        self.length = int(past[0][0].shape[2])

    def past(self, model, B, dtype):
        return [(k.to(dtype)[None].expand(B, -1, -1, -1), v.to(dtype)[None].expand(B, -1, -1, -1))
                for k, v in zip(self.k, self.v)]


class Prompt(nn.Module):
    def __init__(self, emb: torch.Tensor):
        super().__init__()
        self.emb = nn.Parameter(emb.detach().float().clone())
        self.length = int(emb.shape[0])

    def past(self, model, B, dtype):
        e = self.emb
        h = model.trunk.embed.register_forward_hook(lambda m, i, o: e[None].to(o.dtype))
        try:
            ids = torch.zeros(1, self.length, dtype=torch.long, device=e.device)
            out = model.lm(ids, use_cache=True)
        finally:
            h.remove()
        return [(k.expand(B, -1, -1, -1), v.expand(B, -1, -1, -1))
                for k, v in out.past_key_values]


# =============================================================================================
# NF4 linear (QLoRA base)
# =============================================================================================
class NF4Linear(nn.Module):
    def __init__(self, base: nn.Linear, block: int = 64):
        super().__init__()
        W = base.weight.detach().float()
        self.in_features, self.out_features = base.in_features, base.out_features
        n = W.numel()
        pad = (-n) % block
        flat = torch.cat([W.reshape(-1), W.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1).clamp(min=1e-8)
        table = torch.tensor(NF4, device=W.device)
        codes = (flat / absmax[:, None]).unsqueeze(-1).sub(table).abs().argmin(-1).to(torch.uint8)
        codes = codes.view(-1)
        if codes.numel() % 2:
            codes = torch.cat([codes, codes.new_zeros(1)])
        self.register_buffer("q_packed", (codes[0::2] << 4) | codes[1::2])
        self.register_buffer("q_absmax", absmax.half())
        self.register_buffer("q_table", table)
        self.n, self.block = n, block
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(),
                                                                requires_grad=False)

    @property
    def weight(self):
        p = self.q_packed
        codes = torch.stack([p >> 4, p & 15], 1).view(-1).long()
        vals = self.q_table[codes][: self.q_absmax.numel() * self.block].view(-1, self.block)
        W = (vals * self.q_absmax.float()[:, None]).view(-1)[: self.n]
        return W.view(self.out_features, self.in_features)

    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype), None if self.bias is None else
                        self.bias.to(x.dtype))


# =============================================================================================
# the model
# =============================================================================================
class SummLM(nn.Module):
    def __init__(self, lm, layer_ids=None):
        super().__init__()
        kind = {"GPT2StyleLM": "gpt2", "LlamaStyleLM": "llama", "BloomStyleLM": "bloom"}[
            type(lm).__name__]
        if layer_ids is not None:
            blocks = getattr(lm, BLOCK_ATTR[kind])
            setattr(lm, BLOCK_ATTR[kind], nn.ModuleList([blocks[i] for i in layer_ids]))
        self.trunk = Trunk(lm, "causal")                 # PEFT view: the SAME block modules
        self.lm = lm
        lm.lm_head = SelectHead(lm.lm_head)
        self.classifier = nn.Module()                     # no new head
        self.virtual = None
        self.kind = kind

    @property
    def n_virtual(self) -> int:
        return 0 if self.virtual is None else self.virtual.length

    def forward(self, input_ids, attention_mask=None, past_key_values=None, use_cache=False,
                last_only=False, sel=None):
        B = input_ids.shape[0]
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        if self.virtual is not None:
            if past_key_values is None:
                dt = self.trunk.embed.weight.dtype
                if input_ids.is_cuda and torch.is_autocast_enabled():
                    try:
                        dt = torch.get_autocast_dtype("cuda")
                    except AttributeError:                       # torch < 2.4
                        dt = torch.get_autocast_gpu_dtype()
                past_key_values = self.virtual.past(self, B, dt)
            attention_mask = torch.cat([attention_mask.new_ones(B, self.n_virtual),
                                        attention_mask], 1)
        self.lm.lm_head.sel = sel
        try:
            return self.lm(input_ids, attention_mask=attention_mask,
                           past_key_values=past_key_values, use_cache=use_cache,
                           last_only=last_only)
        finally:
            self.lm.lm_head.sel = None

    def lm_logits_at_targets(self, input_ids, attention_mask, labels):
        """(logits (N, V) fp32, targets (N,)) at the positions that predict a summary token."""
        tgt = labels[:, 1:]
        sel = torch.zeros_like(labels, dtype=torch.bool)
        sel[:, :-1] = tgt != -100
        out = self(input_ids, attention_mask, sel=sel)
        return out.logits.float(), tgt[tgt != -100]


def build_summ_model(phase1_dir: str, layer_ids=None):
    lm, cfg = load_phase1_lm(phase1_dir)
    return SummLM(lm, layer_ids), cfg


# =============================================================================================
# techniques
# =============================================================================================
def _sample_ids(sample_ids, n):
    ids = [int(x) for x in (sample_ids if sample_ids is not None else [])][:n]
    while len(ids) < n:
        ids.append(3 + len(ids) % 200)
    return ids


def apply_summ_technique(model: SummLM, technique: str, cfg: Dict, log=print, sample_ids=None):
    """Returns (optimizer groups, technique hooks). `sample_ids`: real tokens for prefix/prompt
    initialisation."""
    from cjpe_models import assert_one_device, model_size
    from cjpe_peft import Technique, apply_technique
    if technique not in NEW_TECHNIQUES:
        return apply_technique(model, technique, cfg, log=log)
    dev = model.trunk.embed.weight.device
    lr = float(cfg["lr"])
    wd = float(cfg.get("weight_decay", 0.01))
    if technique == "qlora":
        groups, tech = apply_technique(model, "lora", cfg, log=lambda *a: None)
        n = 0
        for name, m in list(model.named_modules()):
            if isinstance(m, LoRALinear) and isinstance(m.base, nn.Linear):
                m.base = NF4Linear(m.base, int(cfg.get("qlora_block", 64)))
                n += 1
        tech.name = "qlora"
        log(f"   [qlora] {n} frozen linears stored as NF4 (4-bit) + LoRA r={cfg.get('lora_r', 16)}")
    else:
        for p in model.parameters():
            p.requires_grad = False
        trunk = model.trunk
        if technique == "pfeiffer":
            bott = int(cfg.get("pfeiffer_bottleneck") or max(8, trunk.hidden_size // 16))
            for blk in trunk.blocks:
                path = ADAPTER_SITES[trunk.kind][1]                     # FFN output only
                _set(blk, path, AdapterWrapped(_get(blk, path), bott))
            named = [(n, p) for n, p in model.named_parameters() if ".adapter." in n]
            norm_ids = {id(p) for m in trunk.norm_modules() for p in m.parameters()}
            named += [(n, p) for n, p in model.named_parameters() if id(p) in norm_ids]
        else:
            P = int(cfg.get("n_virtual", 20))
            ids = torch.tensor([_sample_ids(sample_ids, P)], dtype=torch.long, device=dev)
            with torch.no_grad():
                if technique == "prefix":
                    model.virtual = Prefix(model.lm(ids, use_cache=True).past_key_values)
                else:
                    model.virtual = Prompt(trunk.embed(ids)[0])
            named = [(n, p) for n, p in model.named_parameters() if n.startswith("virtual.")]
        for _, p in named:
            p.requires_grad = True
        groups = _groups(named, lr, wd if technique == "pfeiffer" else 0.0, technique)
        tech = Technique()
        tech.name = technique
        actual = {n for n, p in model.named_parameters() if p.requires_grad}
        if actual != {n for n, _ in named}:
            raise AssertionError(f"[{technique}] trainable-parameter mismatch")
    model.to(dev)
    assert_one_device(model, dev)
    s = model_size(model)
    log(f"   [{technique}] trainable {s['trainable_params']:,} / {s['total_params']:,} "
        f"({s['trainable_pct']:.3f}%)")
    return groups, tech


def rebuild_summ(run_config: Dict, best_pt: str, phase1_dir: Optional[str] = None,
                 device="cpu") -> SummLM:
    """A finished run's model (KD teacher)."""
    from cjpe_models import load_trainable_state
    model, _ = build_summ_model(phase1_dir or run_config["phase1_dir"],
                                run_config.get("layer_ids"))
    tech = "full_ft" if run_config["technique"] in ("kd", "student_hard") else \
        run_config["technique"]
    apply_summ_technique(model, tech, run_config, log=lambda *a: None)
    load_trainable_state(model, torch_load(best_pt))
    return model.to(device).eval()


# =============================================================================================
# batching + decoding
# =============================================================================================
def collate_train(seqs: List[Tuple[np.ndarray, np.ndarray]], device=None) -> Dict:
    """seqs: (prompt incl. EOS-start and separator, target) -> right-padded batch with labels
    -100 everywhere except the target tokens."""
    T = max(len(p) + len(t) for p, t in seqs)
    assert T <= CTX, f"sequence of {T} tokens > {CTX}"
    ids = np.full((len(seqs), T), PAD_ID, np.int64)
    mask = np.zeros((len(seqs), T), np.int64)
    lab = np.full((len(seqs), T), -100, np.int64)
    for i, (p, t) in enumerate(seqs):
        n = len(p) + len(t)
        ids[i, :len(p)] = p
        ids[i, len(p):n] = t
        mask[i, :n] = 1
        lab[i, len(p):n] = t
    b = {"input_ids": torch.from_numpy(ids), "attention_mask": torch.from_numpy(mask),
         "labels": torch.from_numpy(lab), "n_tokens": int(mask.sum()),
         "n_targets": int((lab != -100).sum())}
    if device is not None:
        for k in ("input_ids", "attention_mask", "labels"):
            b[k] = b[k].to(device, non_blocking=True)
    return b


def collate_prompts(prompts: List[np.ndarray], device) -> Tuple[torch.Tensor, torch.Tensor]:
    T = max(len(p) for p in prompts)
    ids = np.full((len(prompts), T), PAD_ID, np.int64)
    mask = np.zeros((len(prompts), T), np.int64)
    for i, p in enumerate(prompts):                         # LEFT padding for generation
        ids[i, T - len(p):] = p
        mask[i, T - len(p):] = 1
    return torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)


def _ban(logits, rows, finished, n_ngram):
    if n_ngram and n_ngram > 1:
        for b, row in enumerate(rows):
            if not finished[b] and len(row) >= n_ngram:
                bad = _banned_ngram_tokens(row, n_ngram)
                if bad:
                    logits[b, bad] = float("-inf")


@torch.inference_mode()
def decode_batch(model, ids, mask, max_new: int, min_new: int = 0, num_beams: int = 1,
                 no_repeat_ngram: int = 4, length_penalty: float = 1.0) -> List[List[int]]:
    """Greedy (num_beams=1) or beam search with n-gram blocking over the GENERATED tokens only
    (copying phrases from the document stays possible). Returns token lists without EOS."""
    B = ids.shape[0]
    K = max(1, int(num_beams))
    out = model(ids, attention_mask=mask, use_cache=True, last_only=True)
    past, logits = out.past_key_values, out.logits[:, -1].float()
    if K > 1:
        past = [(k.repeat_interleave(K, 0), v.repeat_interleave(K, 0)) for k, v in past]
        logits = logits.repeat_interleave(K, 0)
        mask = mask.repeat_interleave(K, 0)
    R = B * K
    rows = [[] for _ in range(R)]
    fin = [False] * R
    scores = torch.zeros(B, K, device=ids.device)
    if K > 1:
        scores[:, 1:] = float("-inf")
    for step in range(max_new):
        logits[:, PAD_ID] = float("-inf")
        if step < min_new:
            logits[:, EOS_ID] = float("-inf")
        _ban(logits, rows, fin, no_repeat_ngram)
        if K == 1:
            nxt = logits.argmax(-1)
            nl = nxt.tolist()
            for b in range(R):
                if not fin[b]:
                    if nl[b] == EOS_ID:
                        fin[b] = True
                    else:
                        rows[b].append(nl[b])
            nxt = torch.where(torch.tensor(fin, device=nxt.device), torch.full_like(nxt, EOS_ID),
                              nxt)
        else:
            logp = logits.log_softmax(-1)
            ft = torch.tensor(fin, device=logp.device)
            if ft.any():
                logp[ft] = float("-inf")
                logp[ft, EOS_ID] = 0.0
            V = logp.shape[-1]
            cand = (scores.view(-1, 1) + logp).view(B, K * V)
            top, idx = cand.topk(K, dim=1)
            src = (torch.arange(B, device=idx.device)[:, None] * K + idx // V).view(-1)
            tok = (idx % V).view(-1)
            scores = top
            sl, tl = src.tolist(), tok.tolist()
            rows = [list(rows[s]) for s in sl]
            fin = [fin[s] for s in sl]
            for r in range(R):
                if not fin[r]:
                    if tl[r] == EOS_ID:
                        fin[r] = True
                    else:
                        rows[r].append(tl[r])
            past = [(k[src], v[src]) for k, v in past]
            mask = mask[src]
            nxt = tok
        if all(fin) or step == max_new - 1:
            break
        mask = torch.cat([mask, mask.new_ones(R, 1)], 1)
        out = model(nxt[:, None], attention_mask=mask, past_key_values=past, use_cache=True)
        past, logits = out.past_key_values, out.logits[:, -1].float()
    if K == 1:
        return rows
    lens = torch.tensor([max(1, len(r)) for r in rows], device=scores.device,
                        dtype=torch.float32).view(B, K)
    best = (scores / lens ** length_penalty).argmax(1).tolist()
    return [rows[b * K + best[b]] for b in range(B)]


def generate_many(model, prompts: List[np.ndarray], device, amp_ctx, max_new: int,
                  min_new: int = 0, num_beams: int = 1, no_repeat_ngram: int = 4,
                  batch: int = 16, log=None) -> List[List[int]]:
    """Length-sorted batches; OOM -> batch halves. Output order == input order."""
    model.eval()
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
    res: List[Optional[List[int]]] = [None] * len(prompts)
    i = 0
    bs = max(1, int(batch))
    while i < len(order):
        g = order[i:i + bs]
        try:
            ids, mask = collate_prompts([prompts[j] for j in g], device)
            with amp_ctx():
                outs = decode_batch(model, ids, mask, max_new, min_new, num_beams,
                                    no_repeat_ngram)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower() or bs == 1:
                raise
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            if log:
                log(f"   [oom] generation batch -> {bs}")
            continue
        for j, o in zip(g, outs):
            res[j] = o
        i += len(g)
    return res


In [ ]:
%%writefile /kaggle/working/phase6/summ_data.py
"""
summ_data.py -- Phase-6 data: schema-tolerant loading, leak guard, held-out dev split, the token
cache and the long-document representations.

Data (PDF 12.3): Train/train.json [{id, document, summary}], Test/test.json [{id, document}].
No dev split is provided -> a deterministic dev split is carved from train by a hash of the
NORMALISED document text (exact duplicates always land on the same side; same split in every
notebook and account). `dev_gen` = the first N dev documents in hash order: the subset that is
actually decoded for ROUGE / BERTScore (generation is the slow part).

Representations (`doc_repr`), every sequence = [EOS] + document part + separator + summary [EOS]
<= 1024 - n_virtual tokens:
  head      first D document tokens, one sequence, summary budget S
  headtail  first 60% + last 40% of D (the judgment's conclusion is at the end)
  dancer    divide-and-conquer (Gidiotis & Tsoumakas 2020): the document is cut into chunks of C
            tokens; every reference sentence is aligned to the chunk with the highest
            content-word recall; chunk i is trained to produce ITS sentences ("" -> EOS only),
            separator "SUMMARY (part i of n):"; at test time the chunk outputs are concatenated
            in order. Lets a 1024-token model write long summaries of long judgments.
  auto      dancer when the median train summary is longer than `auto_long` tokens, else headtail
"""
import hashlib
import json
import os
import pickle
import re
import time
from multiprocessing import get_context
from typing import Dict, List, Optional, Tuple

import numpy as np

from cjpe_common import file_md5, write_json
from cjpe_data import (DataError, _detect_keys, _records, discover_files, leak_guard,
                       length_stats, norm_hash)
from gen_ml import align_sentences, sentence_spans

SUMMARY_KEYS = ("summary", "summaries", "abstract", "headnote", "reference", "target",
                "ref_summary", "gold_summary")
MAX_DOC_CHARS = 250_000
CACHE_VERSION = 1
SEP_TEXT = "\n\nSUMMARY:\n"
PART_TEXT = "\n\nSUMMARY (part {i} of {n}):\n"
_TOK = None


# =============================================================================================
# loading
# =============================================================================================
def load_summ_split(path: str, split: str, log=print) -> Tuple[List[Dict], Dict]:
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    recs, keyed = _records(obj)
    if not recs:
        raise DataError(f"{split}: no records in {path}")
    for r in recs:                                    # keep the summary out of the text guess
        if not isinstance(r, dict):
            raise DataError(f"{split}: records are not JSON objects")
    sk = next((k for k in SUMMARY_KEYS if all(k in r for r in recs[:200])), None)
    view = [{k: v for k, v in r.items() if k != sk} for r in recs[:200]]
    idk, tk, _ = _detect_keys(view, keyed)
    if split == "train" and sk is None:
        raise DataError(f"train: no summary key among {sorted(recs[0].keys())}")
    out = []
    for r in recs:
        t, s = r.get(tk), r.get(sk) if sk else None
        if isinstance(t, list):
            t = "\n".join(str(x) for x in t)
        if isinstance(s, list):
            s = " ".join(str(x) for x in s)
        if not isinstance(t, str):
            raise DataError(f"{split}: document field {tk!r} is {type(t).__name__}")
        out.append({"id": str(r[idk]), "text": t, "summary": s if isinstance(s, str) else None})
    if split == "train":
        out = [d for d in out if d["summary"] and d["summary"].strip() and d["text"].strip()]
    schema = {"container": "dict keyed by id" if keyed else "list of objects", "id_key": idk,
              "text_key": tk, "summary_key": sk, "n": len(out), "fields": sorted(recs[0].keys()),
              "md5": file_md5(path), "path": path}
    return out, schema


def split_dev(train: List[Dict], dev_frac: float, dev_max: int, dev_min: int = 20):
    keyed = sorted(train, key=lambda d: hashlib.md5(norm_hash(d["text"]).encode()).hexdigest())
    n_dev = int(min(dev_max, max(dev_min, round(dev_frac * len(train)))))
    n_dev = min(n_dev, max(1, len(train) // 4))
    dev_h = {norm_hash(d["text"]) for d in keyed[:n_dev]}
    dev = [d for d in keyed if norm_hash(d["text"]) in dev_h]
    tr = [d for d in train if norm_hash(d["text"]) not in dev_h]
    return tr, dev


def load_all(input_roots, dev_frac=0.06, dev_max=300, log=print) -> Tuple[Dict, Dict, Dict]:
    found = discover_files(input_roots, log=log)
    if found["source"] is None:
        raise FileNotFoundError(
            "Phase-6 data not found. Add the Kaggle secret `HF` (token of a Hugging Face account "
            "that accepted the dataset terms) and turn Internet ON -- or attach a dataset with "
            f"Phase_6_Summ/Train/train.json and Test/test.json. Found: {found['paths']}")
    data, schemas = {}, {}
    for s in ("train", "test"):
        data[s], schemas[s] = load_summ_split(found["paths"][s], s, log)
        log(f"   [data] {s}: {schemas[s]['n']} records; keys id={schemas[s]['id_key']!r} "
            f"document={schemas[s]['text_key']!r} summary={schemas[s]['summary_key']!r}")
    data["dev"] = []
    leak = leak_guard(data, log)                         # drops train docs that occur in test
    data["train"], data["dev"] = split_dev(data["train"], dev_frac, dev_max)
    log(f"   [data] held-out dev split: {len(data['dev'])} docs (hash of normalised text); "
        f"train {len(data['train'])}")
    return data, schemas, {"leak_guard": leak, "source": found["source"]}


# =============================================================================================
# token cache
# =============================================================================================
def _init_worker(bpe_dir: str) -> None:
    global _TOK
    from bpe_tokenizer import BPETokenizer
    _TOK = BPETokenizer.from_pretrained(bpe_dir)


def _encode_rec(args):
    text, summary = args
    t = text[:MAX_DOC_CHARS]
    doc = np.asarray(_TOK.encode(t), np.int32)
    n_full = len(doc) if len(t) == len(text) else int(len(doc) * len(text) / max(1, len(t)))
    sents = None
    if summary:
        pieces = sentence_spans(summary.strip())
        sents = [np.asarray(_TOK.encode(p), np.int32) for p in pieces]
    return {"doc": doc, "n_tokens": n_full, "sents": sents,
            "sent_text": pieces if summary else None}


def cache_key(file_md5s: Dict[str, str], tok_md5: str, dev_n: int) -> str:
    s = json.dumps({"v": CACHE_VERSION, "files": file_md5s, "tok": tok_md5, "dev": dev_n,
                    "max_chars": MAX_DOC_CHARS}, sort_keys=True)
    return hashlib.md5(s.encode()).hexdigest()[:16]


def find_cache(key: str, roots: List[str]) -> Optional[str]:
    name = f"summ_cache_{key}.pkl"
    for r in roots:
        if r and os.path.isdir(r):
            for dp, _dn, fns in os.walk(r):
                if name in fns:
                    return os.path.join(dp, name)
    return None


def build_cache(data: Dict[str, List[Dict]], bpe_dir: str, out_dir: str, key: str,
                n_proc: int, log=print) -> str:
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"summ_cache_{key}.pkl")
    t0 = time.time()
    res = {}
    for split, docs in data.items():
        args = [(d["text"], d.get("summary")) for d in docs]
        if n_proc > 1 and len(args) >= 32:
            with get_context("fork").Pool(n_proc, initializer=_init_worker,
                                          initargs=(bpe_dir,)) as p:
                enc = p.map(_encode_rec, args, chunksize=8)
        else:
            _init_worker(bpe_dir)
            enc = [_encode_rec(a) for a in args]
        for d, e in zip(docs, enc):
            e["id"] = d["id"]
            e["summary"] = d.get("summary")
            if split != "train":
                e["text"] = d["text"][:MAX_DOC_CHARS]
        res[split] = enc
        log(f"   [cache] {split}: {len(enc)} docs tokenised ({time.time() - t0:.0f}s so far)")
    with open(path + ".tmp", "wb") as f:
        pickle.dump(res, f, protocol=4)
    os.replace(path + ".tmp", path)
    return path


def load_cache(path: str) -> Dict[str, List[Dict]]:
    with open(path, "rb") as f:
        return pickle.load(f)


def summary_stats(cache: Dict[str, List[Dict]]) -> Dict:
    tr = cache["train"]
    st = [sum(len(s) for s in r["sents"]) for r in tr if r.get("sents")]
    dt = [r["n_tokens"] for r in tr]
    sw = [len((r.get("summary") or "").split()) for r in tr]
    ns = [len(r["sents"]) for r in tr if r.get("sents")]
    return {"summary_tokens": length_stats([float(x) for x in st]),
            "doc_tokens": length_stats([float(x) for x in dt]),
            "summary_words": length_stats([float(x) for x in sw]),
            "summary_sentences": length_stats([float(x) for x in ns]),
            "median_summary_words": float(np.median(sw)) if sw else 200.0}


# =============================================================================================
# representations
# =============================================================================================
class Layout:
    """Token budgets of one run (all sequences <= ctx = 1024 - n_virtual)."""

    def __init__(self, tok, cfg: Dict, stats: Dict, n_virtual: int = 0):
        self.ctx = 1024 - int(n_virtual)
        self.repr = cfg.get("doc_repr", "auto")
        med = stats["summary_tokens"].get("median", 200.0)
        if self.repr == "auto":
            self.repr = "dancer" if med > float(cfg.get("auto_long", 350)) else "headtail"
        self.sep = np.asarray(tok.encode(SEP_TEXT), np.int32)
        self.tok = tok
        self._parts = {}
        if self.repr == "dancer":
            self.C = min(int(cfg.get("chunk_tokens", 640)), self.ctx - 200)
            self.max_chunks = int(cfg.get("max_chunks", 16))
            self.S = self.ctx - 2 - self.C - len(self.part_sep(99, 99))
        else:
            p90 = stats["summary_tokens"].get("p90", 256.0)
            cap = int(cfg.get("summary_frac", 0.5) * self.ctx)
            self.S = int(min(max(64, p90), cap))
            self.D = self.ctx - 2 - len(self.sep) - self.S
        self.median_words = stats.get("median_summary_words", 200.0)

    def part_sep(self, i: int, n: int) -> np.ndarray:
        k = (i, n)
        if k not in self._parts:
            self._parts[k] = np.asarray(self.tok.encode(PART_TEXT.format(i=i, n=n)), np.int32)
        return self._parts[k]

    def describe(self) -> Dict:
        if self.repr == "dancer":
            return {"doc_repr": "dancer", "chunk_tokens": self.C, "max_chunks": self.max_chunks,
                    "target_budget_per_chunk": self.S, "ctx": self.ctx}
        return {"doc_repr": self.repr, "doc_tokens": self.D, "summary_budget": self.S,
                "ctx": self.ctx}

    # ---------------------------------------------------------------- document parts
    def single_prompt(self, doc: np.ndarray) -> np.ndarray:
        D = self.D
        if len(doc) <= D:
            body = doc
        elif self.repr == "head":
            body = doc[:D]
        else:
            h = int(0.6 * D)
            body = np.concatenate([doc[:h], doc[len(doc) - (D - h):]])
        return np.concatenate([[0], body, self.sep]).astype(np.int64)

    def chunk_bounds(self, n_doc: int) -> List[Tuple[int, int]]:
        C = self.C
        n = max(1, int(np.ceil(n_doc / C)))
        b = [(i * C, min(n_doc, (i + 1) * C)) for i in range(n)]
        if len(b) > self.max_chunks:                     # keep first, last, evenly spaced middle
            pick = np.unique(np.round(np.linspace(0, len(b) - 1, self.max_chunks)).astype(int))
            b = [b[i] for i in pick]
        return b

    def chunk_prompts(self, doc: np.ndarray) -> List[np.ndarray]:
        bs = self.chunk_bounds(len(doc))
        return [np.concatenate([[0], doc[s:e], self.part_sep(i + 1, len(bs))]).astype(np.int64)
                for i, (s, e) in enumerate(bs)]

    def prompts(self, rec: Dict) -> List[np.ndarray]:
        if self.repr == "dancer":
            return self.chunk_prompts(rec["doc"])
        return [self.single_prompt(rec["doc"])]

    # ---------------------------------------------------------------- training examples
    def examples(self, rec: Dict) -> List[Tuple[np.ndarray, np.ndarray]]:
        sents = rec.get("sents") or []
        if self.repr != "dancer":
            tgt = np.concatenate(sents) if sents else np.zeros(0, np.int32)
            tgt = tgt[:self.S] if len(tgt) >= self.S else np.concatenate([tgt, [0]])
            return [(self.single_prompt(rec["doc"]), tgt.astype(np.int64))]
        doc = rec["doc"]
        bs = self.chunk_bounds(len(doc))
        texts = [self.tok.decode(doc[s:e].tolist()) for s, e in bs]
        owner = align_sentences(rec.get("sent_text") or [], texts)
        out = []
        for i, (s, e) in enumerate(bs):
            mine = [sents[j] for j, o in enumerate(owner) if o == i]
            tgt = np.concatenate(mine) if mine else np.zeros(0, np.int32)
            tgt = tgt[:self.S] if len(tgt) >= self.S else np.concatenate([tgt, [0]])
            prompt = np.concatenate([[0], doc[s:e], self.part_sep(i + 1, len(bs))])
            out.append((prompt.astype(np.int64), tgt.astype(np.int64)))
        return out

    def max_new(self, train_examples: List[Tuple[np.ndarray, np.ndarray]]) -> int:
        """Generation cap: p95 of the training target lengths (incl. EOS), within the budget."""
        L = [len(t) for _, t in train_examples if len(t) > 1]
        if not L:
            return min(self.S, 64)
        return int(min(self.S, max(16, np.percentile(L, 95))))

    def min_new(self, train_examples, frac: float) -> int:
        """Single-pass layouts: forbid EOS before frac x median target length (an early EOS
        costs recall in both ROUGE and BERTScore). Dancer chunks may legitimately be empty."""
        if self.repr == "dancer":
            return 0
        L = [len(t) for _, t in train_examples if len(t) > 1]
        return int(frac * np.median(L)) if L else 0

    def join(self, pieces: List[str]) -> str:
        return " ".join(p.strip() for p in pieces if p and p.strip())


In [ ]:
%%writefile /kaggle/working/phase6/summ_train.py
"""
summ_train.py -- ONE Phase-6 training job (one Phase-1 model x one technique), run as a
subprocess on ONE GPU by run_experiment.py. Reads RUN/logs/run_config.json.

    python summ_train.py --run-dir RUN [--mode train|overfit]

train: tokenizer + token cache -> layout (head / headtail / dancer) -> model + technique ->
       critical self-checks on the real device -> pre-flight (train tok/s, generation speed) ->
       time plan (epochs / train fraction / dev_gen size) -> training on summary tokens only,
       dev loss 2x per epoch, best.pt by dev loss, early stopping, 30-min resumable last.pt,
       time guard that keeps the generation reserve -> decode dev_gen + test -> dev-tuned length
       cap (ROUGE / BERTScore at 5 caps) -> validated submission.zip -> DONE.json
KD students (technique "kd") load the finished teacher run and distil ONLINE:
  loss = alpha * CE(student, gold) + (1 - alpha) * T^2 * KL(teacher_T || student_T)
  over the summary tokens (teacher logits at exactly the same positions; all Phase-1 models
  share the BPE vocabulary).
Exit codes (as Phases 4/5): 0 ok, 1 error, 2 does not fit, 4 too many NaN steps, 5 watchdog,
6 critical self-check failed, 7 OOM at one sequence per micro-batch.
"""
import argparse
import copy
import json
import math
import os
import random
import sys
import time
import traceback
import zipfile
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F

from cjpe_common import DEFAULT_EPOCHS, read_json, write_json, now_str, write_csv
from cjpe_models import assert_one_device, load_trainable_state, model_size
from cjpe_train import (Ctx, JobExit, amp_setup, is_oom, make_optimizer, make_scaler,
                        make_scheduler, peak_gb, save_best, save_last, set_seed)
from gen_ml import (LENGTH_RATIOS, bertscore, cap_words, diagnostics, free_bertscore, mean_of,
                    primary, rouge_all)
from ner_models import torch_load
from summ_data import Layout, load_cache, summary_stats
from summ_model import (EOS_ID, apply_summ_technique, build_summ_model, collate_train,
                        generate_many, rebuild_summ)

PEFT_ZERO_INIT = ("lorma", "lora", "adalora", "adapters", "pfeiffer")


# =============================================================================================
# data
# =============================================================================================
def build_data(ctx, tok, layout: Layout, cache: Dict) -> Dict:
    cfg = ctx.cfg
    rng = np.random.default_rng(int(cfg["seed"]))
    d = {}
    for split in ("train", "dev"):
        ex, owner = [], []
        for i, r in enumerate(cache[split]):
            for e in layout.examples(r):
                ex.append(e)
                owner.append(i)
        if layout.repr == "dancer" and split == "train":
            empty = [k for k, (_, t) in enumerate(ex) if len(t) <= 1]
            full = [k for k, (_, t) in enumerate(ex) if len(t) > 1]
            cap = int(float(cfg.get("empty_frac", 0.25)) * len(full) / 0.75)
            if len(empty) > cap:
                keep = set(full) | set(rng.choice(empty, cap, replace=False).tolist())
                ex = [e for k, e in enumerate(ex) if k in keep]
                owner = [o for k, o in enumerate(owner) if k in keep]
        cap = int(cfg.get("dev_loss_seqs", 600))
        if split == "dev" and len(ex) > cap:               # dev loss on a fixed random subset
            keep = sorted(rng.choice(len(ex), cap, replace=False).tolist())
            ex, owner = [ex[k] for k in keep], [owner[k] for k in keep]
        d[split] = {"ex": ex, "owner": owner}
    n_gen = min(len(cache["dev"]), int(cfg.get("dev_gen", 64)))
    d["dev_gen"] = list(range(n_gen))
    return d


def assert_integrity(cache: Dict, d: Dict, layout: Layout, vocab: int) -> Dict:
    tr = {r["id"] for r in cache["train"]}
    dv = {r["id"] for r in cache["dev"]}
    te = {r["id"] for r in cache["test"]}
    assert not (tr & te), f"{len(tr & te)} train ids are test ids"
    assert not (tr & dv), f"{len(tr & dv)} train ids are dev ids"
    assert d["train"]["ex"], "no training examples"
    for s in ("train", "dev"):
        for p, t in d[s]["ex"][:5000]:
            assert len(p) + len(t) <= layout.ctx, f"{s}: sequence {len(p) + len(t)} > {layout.ctx}"
            assert int(p.max()) < vocab and (len(t) == 0 or int(t.max()) < vocab), "token id"
    for r in cache["test"][:200]:
        for p in layout.prompts(r):
            assert len(p) < layout.ctx, "test prompt too long"
    return {"train_x_dev": 0, "train_x_test": 0, "n_train_seqs": len(d["train"]["ex"]),
            "n_dev_seqs": len(d["dev"]["ex"])}


def micro_groups(ex, idx: List[int], micro_tokens: int) -> List[List[int]]:
    """Split one effective batch into micro-batches of <= micro_tokens padded tokens."""
    idx = sorted(idx, key=lambda i: len(ex[i][0]) + len(ex[i][1]))
    out, cur, mx = [], [], 0
    for i in idx:
        L = len(ex[i][0]) + len(ex[i][1])
        if cur and max(mx, L) * (len(cur) + 1) > micro_tokens:
            out.append(cur)
            cur, mx = [], 0
        cur.append(i)
        mx = max(mx, L)
    if cur:
        out.append(cur)
    return out


# =============================================================================================
# loss / step
# =============================================================================================
class Stepper:
    def __init__(self, ctx, model, opt, sched, scaler, tech, amp_ctx, cfg, teacher=None):
        self.ctx, self.model, self.opt, self.sched, self.scaler = ctx, model, opt, sched, scaler
        self.tech, self.amp_ctx, self.cfg, self.teacher = tech, amp_ctx, cfg, teacher
        self.micro = int(cfg.get("micro_tokens", 8192))
        self.skipped = self.overflow = 0
        self.params = [p for p in model.parameters() if p.requires_grad]

    def batch_loss(self, b):
        """Sum of per-token losses (fp32) and the number of target tokens."""
        logits, tgt = self.model.lm_logits_at_targets(b["input_ids"], b["attention_mask"],
                                                      b["labels"])
        ce = F.cross_entropy(logits, tgt, reduction="sum")
        if self.teacher is None:
            return ce
        with torch.no_grad():
            tl, _ = self.teacher.lm_logits_at_targets(b["input_ids"], b["attention_mask"],
                                                      b["labels"])
        T = float(self.cfg.get("kd_T", 2.0))
        a = float(self.cfg.get("kd_alpha", 0.5))
        kl = F.kl_div(F.log_softmax(logits / T, -1), F.log_softmax(tl.float() / T, -1),
                      reduction="sum", log_target=True)
        return a * ce + (1 - a) * T * T * kl

    def step(self, ex, idx, gstep, total):
        while True:
            try:
                return self._step(ex, idx, gstep, total)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                self.opt.zero_grad(set_to_none=True)
                torch.cuda.empty_cache()
                one = max(len(ex[i][0]) + len(ex[i][1]) for i in idx)
                if self.micro <= one:
                    raise JobExit(7, "CUDA OOM with one sequence per micro-batch")
                self.micro = max(one, self.micro // 2)
                self.ctx.log(f"   [oom] micro-batch -> {self.micro} tokens")

    def _step(self, ex, idx, gstep, total):
        self.model.train()
        n_t = sum(len(ex[i][1]) for i in idx)
        tot, toks = 0.0, 0
        for g in micro_groups(ex, idx, self.micro):
            b = collate_train([ex[i] for i in g], self.ctx.device)
            with self.amp_ctx():
                loss = self.batch_loss(b) / max(1, n_t)
            extra = self.tech.extra_loss(self.model)
            if extra is not None:
                loss = loss + extra * (b["n_targets"] / max(1, n_t))
            self.scaler.scale(loss).backward()
            tot += float(loss.detach())
            toks += b["n_tokens"]
        self.scaler.unscale_(self.opt)
        gn = torch.nn.utils.clip_grad_norm_(self.params, float(self.cfg.get("grad_clip", 1.0)))
        if not math.isfinite(tot):
            self.skipped += 1
            self.opt.zero_grad(set_to_none=True)
            self.scaler.update()
            return None, toks
        if not torch.isfinite(gn):
            self.overflow += 1
        self.tech.after_backward(self.model, gstep, total)
        self.scaler.step(self.opt)
        self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        if self.sched is not None:
            self.sched.step()
        return tot, toks


@torch.no_grad()
def dev_loss(ctx, model, ex, amp_ctx, micro: int) -> float:
    model.eval()
    tot, n = 0.0, 0
    for g in micro_groups(ex, list(range(len(ex))), micro):
        b = collate_train([ex[i] for i in g], ctx.device)
        with amp_ctx():
            logits, tgt = model.lm_logits_at_targets(b["input_ids"], b["attention_mask"],
                                                     b["labels"])
        tot += float(F.cross_entropy(logits.float(), tgt, reduction="sum"))
        n += int(tgt.numel())
    return tot / max(1, n)


# =============================================================================================
# self-checks
# =============================================================================================
def job_checks(base, model, cfg, d, layout, device, amp_ctx, tech) -> Dict:
    res = {"checks": [], "critical_failed": [], "advisory_failed": []}

    def add(name, kind, fn):
        rec = {"name": name, "kind": kind}
        try:
            v = fn()
            if isinstance(v, dict):
                rec.update(v)
            rec.setdefault("verdict", "pass")
        except Exception as e:
            rec["verdict"] = "fail"
            rec["error"] = f"{type(e).__name__}: {e}"[:1500]
            rec["trace"] = traceback.format_exc()[-1500:]
        if rec["verdict"] != "pass":
            (res["critical_failed"] if kind == "critical" else res["advisory_failed"]).append(name)
        res["checks"].append(rec)

    ex = d["train"]["ex"]
    sel = sorted(range(len(ex)), key=lambda i: len(ex[i][0]) + len(ex[i][1]))[:2]
    b = collate_train([ex[i] for i in sel], device)
    add("all_tensors_on_one_device", "critical",
        lambda: (assert_one_device(model, device), {"device": str(device)})[1])

    def fwd():
        model.train()
        with amp_ctx():
            logits, tgt = model.lm_logits_at_targets(b["input_ids"], b["attention_mask"],
                                                     b["labels"])
            loss = F.cross_entropy(logits.float(), tgt)
        assert logits.shape[0] == tgt.numel() == b["n_targets"], "target positions"
        ex_ = tech.extra_loss(model)
        if ex_ is not None:
            loss = loss + ex_
        assert torch.isfinite(loss), "non-finite loss"
        loss.backward()
        gr = [p.grad for p in model.parameters() if p.requires_grad]
        assert any(g is not None and float(g.abs().sum()) > 0 for g in gr), "no gradient"
        assert all(g is None or bool(torch.isfinite(g).all()) for g in gr), "non-finite grad"
        model.zero_grad(set_to_none=True)
        return {"loss": float(loss)}
    add("forward_backward_finite", "critical", fwd)

    def masked_loss():
        """labels mask: the loss must not depend on document tokens' labels (only targets)."""
        p, t = ex[sel[0]]
        assert len(t) >= 1 and (b["labels"][0] != -100).sum() == len(t)
        return {}
    add("loss_only_on_summary_tokens", "critical", masked_loss)

    def gen():
        p = layout.prompts({"doc": d["_cache_dev0"]})[0]
        out = generate_many(model, [p], device, amp_ctx, 4, 0, int(cfg.get("num_beams", 1)), 4, 1)
        assert len(out) == 1 and len(out[0]) <= 4
        return {"tokens": out[0]}
    add("generation_runs", "critical", gen)

    if base is not None:
        def zero_init():
            model.eval()
            base.eval()
            with torch.no_grad():
                a, _ = model.lm_logits_at_targets(b["input_ids"], b["attention_mask"], b["labels"])
                c, _ = base.lm_logits_at_targets(b["input_ids"], b["attention_mask"], b["labels"])
            diff = float((a - c).abs().max())
            thr = (5e-2 if cfg["technique"] in ("adapters", "pfeiffer") else 1e-3) * max(
                1.0, float(c.abs().max()))
            return {"value": diff, "threshold": thr, "verdict": "pass" if diff <= thr else "warn"}
        add("zero_init_equals_base", "advisory", zero_init)
    model.zero_grad(set_to_none=True)
    res["ok"] = not res["critical_failed"]
    res["selfcheck"] = "fail" if res["critical_failed"] else (
        "warn" if res["advisory_failed"] else "pass")
    return res


# =============================================================================================
# throughput + plan
# =============================================================================================
def measure(ctx, stepper, d, layout, cache, gen_batch: int, n_steps: int = 4):
    model, ex = stepper.model, d["train"]["ex"]
    eff = int(ctx.cfg["effective_batch"])
    sel = np.random.default_rng(1).permutation(len(ex))[:max(1, min(len(ex), eff))].tolist()

    def one():
        toks = 0
        for g in micro_groups(ex, sel, stepper.micro):
            b = collate_train([ex[i] for i in g], ctx.device)
            with stepper.amp_ctx():
                loss = stepper.batch_loss(b) / max(1, b["n_targets"])
            stepper.scaler.scale(loss).backward()
            toks += b["n_tokens"]
        for p in stepper.params:
            p.grad = None
        return toks
    model.train()
    while True:
        try:
            one()
            break
        except RuntimeError as e:
            if not is_oom(e):
                raise
            for p in stepper.params:
                p.grad = None
            torch.cuda.empty_cache()
            big = max(len(ex[i][0]) + len(ex[i][1]) for i in sel)
            if stepper.micro <= big:
                raise JobExit(7, "CUDA OOM with one sequence per micro-batch (pre-flight)")
            stepper.micro = max(big, stepper.micro // 2)
            ctx.log(f"   [oom] pre-flight: micro-batch -> {stepper.micro} tokens")
    sync = (lambda: torch.cuda.synchronize()) if ctx.device.type == "cuda" else (lambda: None)
    sync()
    t = time.time()
    toks = sum(one() for _ in range(n_steps))
    sync()
    r_train = toks / max(1e-6, time.time() - t)
    # generation: seconds for one batch of real prompts x 24 new tokens (min_new forces length)
    prompts = []
    for r in cache["dev"][:max(1, gen_batch)]:
        prompts += layout.prompts(r)
    prompts = sorted(prompts, key=len, reverse=True)[:gen_batch]
    generate_many(model, prompts[:2], ctx.device, stepper.amp_ctx, 4, 4, 1, 4, gen_batch)
    sync()
    t = time.time()
    generate_many(model, prompts, ctx.device, stepper.amp_ctx, 24, 24,
                  int(ctx.cfg.get("num_beams", 1)), 4, gen_batch)
    sync()
    sec_per_step = (time.time() - t) / 24.0
    return r_train, sec_per_step, len(prompts)


def gen_seconds(n_prompts: int, max_new: int, sec_per_step: float, batch: int) -> float:
    return math.ceil(n_prompts / max(1, batch)) * max_new * sec_per_step


def make_plan(ctx, d, layout, cache, r_train, sps, gen_batch, max_new) -> Dict:
    cfg = ctx.cfg
    budget = float(cfg["budget_s"])
    used = time.time() - ctx.t0
    avail = budget - used - max(120.0, 0.04 * budget)
    n_test = sum(len(layout.prompts(r)) for r in cache["test"])
    per_dev = max(1.0, sum(len(layout.prompts(r)) for r in cache["dev"][:50]) /
                  max(1, min(50, len(cache["dev"]))))
    bert_s = 150.0 if cfg.get("bertscore", True) else 0.0
    n_gen = len(d["dev_gen"])
    while True:
        reserve = 1.3 * gen_seconds(n_test + int(n_gen * per_dev), max_new, sps, gen_batch) \
            + bert_s + 60.0
        if reserve <= 0.45 * avail or n_gen <= 16:
            break
        n_gen = max(16, n_gen // 2)
    d["dev_gen"] = d["dev_gen"][:n_gen]
    tok_tr = sum(len(p) + len(t) for p, t in d["train"]["ex"])
    tok_dv = sum(len(p) + len(t) for p, t in d["dev"]["ex"])
    evals = int(cfg.get("evals_per_epoch", 2))
    per_epoch = tok_tr / r_train + evals * tok_dv / (2.5 * r_train)
    e_max = int(cfg.get("epochs_max") or DEFAULT_EPOCHS.get(cfg["technique"], 3))
    e_min = int(cfg.get("min_epochs", 1))
    left = avail - reserve
    e_fit = int(max(0.0, left) // max(1e-6, per_epoch))
    fixed = cfg.get("fixed_plan")
    if fixed:
        ch = {"epochs": int(fixed["epochs"]), "train_fraction": float(fixed.get("frac", 1.0)),
              "reason": "fixed plan"}
    elif e_fit >= e_min:
        ch = {"epochs": min(e_max, e_fit), "train_fraction": 1.0, "reason": "fits"}
    else:
        frac = left / max(1e-6, e_min * per_epoch)
        mf = float(cfg.get("min_train_fraction", 0.1))
        if frac < mf and left < 300:
            raise JobExit(2, f"not even {mf:.0%} of one epoch fits: {left / 60:.1f} min left "
                             f"after the generation reserve of {reserve / 60:.1f} min")
        ch = {"epochs": e_min, "train_fraction": round(max(mf, min(1.0, frac)), 3),
              "reason": "subsample to fit"}
    n_used = int(round(ch["train_fraction"] * len(d["train"]["ex"])))
    proj = used + ch["epochs"] * per_epoch * ch["train_fraction"] + reserve
    return {**ch, "budget_min": budget / 60, "available_min": avail / 60,
            "tokens_per_s_train": r_train, "gen_sec_per_step": sps, "gen_batch": gen_batch,
            "max_new": max_new, "reserve_min": reserve / 60, "dev_gen": n_gen,
            "train_seqs": len(d["train"]["ex"]), "train_seqs_used": n_used,
            "train_docs": len(cache["train"]), "epoch_min": per_epoch / 60, "epochs_fit": e_fit,
            "projected_minutes": proj / 60, "epochs_max": e_max, "test_prompts": n_test,
            "micro_tokens": None, **layout.describe()}


# =============================================================================================
# generation + evaluation + submission
# =============================================================================================
def summarise(ctx, model, tok, layout, recs, amp_ctx, plan) -> List[str]:
    cfg = ctx.cfg
    prompts, owner = [], []
    for i, r in enumerate(recs):
        for p in layout.prompts(r):
            prompts.append(p)
            owner.append(i)
    outs = generate_many(model, prompts, ctx.device, amp_ctx, int(plan["max_new"]),
                         int(plan.get("min_new", 0)),
                         int(cfg.get("num_beams", 1)), int(cfg.get("no_repeat_ngram", 4)),
                         int(plan["gen_batch"]), ctx.log)
    pieces = [[] for _ in recs]
    for o, i in zip(outs, owner):
        pieces[i].append(tok.decode(o, skip_special_tokens=True))
    texts = [layout.join(p) for p in pieces]
    empty = [i for i, t in enumerate(texts) if len(t.split()) < 3]
    if empty:                                            # never submit an empty summary
        first = [layout.prompts(recs[i])[0] for i in empty]
        mn = min(32, int(plan["max_new"]))
        re_ = generate_many(model, first, ctx.device, amp_ctx, max(mn, int(plan["max_new"])), mn,
                            1, int(cfg.get("no_repeat_ngram", 4)), int(plan["gen_batch"]))
        for i, o in zip(empty, re_):
            texts[i] = tok.decode(o, skip_special_tokens=True).strip() or "."
        ctx.log(f"   [gen] {len(empty)} empty outputs regenerated with min_new={mn}")
    return texts, len(empty)


def evaluate_dev(ctx, preds, recs, median_words) -> Dict:
    refs = [r["summary"] for r in recs]
    curve, all_c, all_r = [], [], []
    for ratio in LENGTH_RATIOS:
        cap = None if ratio is None else max(10, int(round(ratio * median_words)))
        c = [cap_words(p, cap) for p in preds]
        curve.append({"ratio": "none" if ratio is None else ratio, "cap_words": cap, "cands": c})
        all_c += c
        all_r += refs
    bs, info = (None, "disabled")
    if ctx.cfg.get("bertscore", True):
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        bs, info = bertscore(all_c, all_r, dev, log=ctx.log)
        free_bertscore()
    n = len(preds)
    rows = []
    for k, c in enumerate(curve):
        rg = rouge_all(c["cands"], refs)
        rec = {"ratio": c["ratio"], "cap_words": c["cap_words"],
               "mean_words": float(np.mean([len(x.split()) for x in c["cands"]])) if n else 0,
               "rouge1": rg["rouge1"], "rouge2": rg["rouge2"], "rougeL": rg["rougeL"],
               "bertscore_f1": float(np.mean(bs[k * n:(k + 1) * n])) if bs else None,
               "_r1": rg.get("rouge1_each"), "_rl": rg.get("rougeL_each"),
               "_bs": bs[k * n:(k + 1) * n] if bs else None}
        rec["score"], rec["metric"] = primary(rec)
        rows.append(rec)
    best = max(rows, key=lambda r: -1 if r["score"] is None else r["score"])
    return {"curve": rows, "best": best, "bertscore_info": info,
            "cands": curve[rows.index(best)]["cands"]}


def write_outputs(ctx, cache, dev_idx, ev, dev_preds_raw, test_texts, n_empty) -> Dict:
    run = ctx.run_dir
    os.makedirs(os.path.join(run, "dev"), exist_ok=True)
    os.makedirs(os.path.join(run, "submission"), exist_ok=True)
    recs = [cache["dev"][i] for i in dev_idx]
    best = ev["best"]
    diag = []
    with open(os.path.join(run, "dev", "dev_predictions.jsonl"), "w") as f:
        for k, (r, c) in enumerate(zip(recs, ev["cands"])):
            dg = diagnostics(c, r.get("text"), r["summary"])
            row = {"id": r["id"], "n_doc_tokens": r["n_tokens"], "rouge1": best["_r1"][k],
                   "rougeL": best["_rl"][k],
                   "bertscore_f1": best["_bs"][k] if best["_bs"] else None, **dg}
            diag.append(row)
            f.write(json.dumps({**row, "prediction": c, "prediction_uncapped":
                                dev_preds_raw[k], "reference": r["summary"]}) + "\n")
    write_csv(os.path.join(run, "dev", "length_curve.csv"),
              [{k: v for k, v in c.items() if not k.startswith("_")} for c in ev["curve"]],
              ["ratio", "cap_words", "mean_words", "rouge1", "rouge2", "rougeL", "bertscore_f1",
               "score", "metric"])
    buckets = []
    for name, lo, hi in (("<2k", 0, 2000), ("2k-5k", 2000, 5000), ("5k-10k", 5000, 10000),
                         (">10k", 10000, 10 ** 12)):
        g = [x for x in diag if lo <= x["n_doc_tokens"] < hi]
        if g:
            buckets.append({"doc_tokens": name, "n": len(g), "rouge1": mean_of(g, "rouge1"),
                            "rougeL": mean_of(g, "rougeL"), "bertscore_f1": mean_of(g, "bertscore_f1"),
                            "ref_content_recall": mean_of(g, "ref_content_recall"),
                            "words": mean_of(g, "words")})
    write_csv(os.path.join(run, "dev", "length_buckets.csv"), buckets,
              ["doc_tokens", "n", "rouge1", "rougeL", "bertscore_f1", "ref_content_recall", "words"])
    keys = ("words", "ref_words", "compression", "repeated_trigram_rate", "novel_bigram_rate",
            "content_support", "number_precision", "citation_precision", "ref_content_recall")
    rep = {"scores": {k: best[k] for k in ("rouge1", "rouge2", "rougeL", "bertscore_f1", "score",
                                           "metric", "ratio", "cap_words")},
           "diagnostics": {k: mean_of(diag, k) for k in keys}, "length_buckets": buckets,
           "bertscore_info": ev["bertscore_info"], "n_dev_gen": len(recs),
           "empty_regenerated_dev": n_empty}
    write_json(os.path.join(run, "dev", "dev_report.json"), rep)
    # ---- submission (list format, PDF 12.3.5), length cap chosen on dev -----------------------
    cap = best["cap_words"]
    test_final = [cap_words(t, cap) for t in test_texts]
    items = [{"id": r["id"], "summary": s} for r, s in zip(cache["test"], test_final)]
    from validate_submission import validate
    with open(os.path.join(run, "submission", "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False)
    zp = os.path.join(run, "submission", "submission.zip")
    with zipfile.ZipFile(zp + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(run, "submission", "prediction.json"), "prediction.json")
    os.replace(zp + ".tmp", zp)
    v = validate(zp, [r["id"] for r in cache["test"]])
    rep["validation"] = v
    rep["test_mean_words"] = v.get("mean_words")
    return rep


# =============================================================================================
# the job
# =============================================================================================
def run_train(ctx: Ctx, mode: str) -> int:
    cfg = ctx.cfg
    set_seed(int(cfg["seed"]))
    dev = ctx.device
    amp_dtype, amp_ctx = amp_setup(dev)
    ctx.status("preflight")
    ctx.log(f"start {cfg['experiment']}/{cfg['technique']}/{cfg['model']} on {dev}"
            f"{' (' + torch.cuda.get_device_name(0) + ')' if dev.type == 'cuda' else ''}, "
            f"budget {cfg['budget_s'] / 60:.0f} min, attempt {cfg.get('attempt', 1)}")
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    inj = os.environ.get("PHASE6_INJECT_FAIL", "")
    if inj and inj.split(":")[0] == cfg["job"] and int(cfg.get("attempt", 1)) == 1:
        raise JobExit(int(inj.split(":")[1]), "injected failure (test)")

    from bpe_tokenizer import BPETokenizer
    tok = BPETokenizer.from_pretrained(cfg["bpe_dir"])
    cache = load_cache(cfg["cache_path"])
    stats = summary_stats(cache)

    # ---- model + technique ----------------------------------------------------------------
    model, p1cfg = build_summ_model(cfg["phase1_dir"], cfg.get("layer_ids"))
    vocab = int(p1cfg["config"]["vocab_size"])
    base = copy.deepcopy(model) if (cfg["technique"] in PEFT_ZERO_INIT and
                                    not cfg.get("skip_job_checks")) else None
    sample = cache["train"][0]["doc"][:64]
    groups, tech = apply_summ_technique(model, cfg["technique"], cfg, ctx.log, sample)
    model.to(dev)
    assert_one_device(model, dev)
    size = model_size(model)
    write_json(os.path.join(ctx.run_dir, "logs", "model_size.json"), size)

    layout = Layout(tok, cfg, stats, model.n_virtual)
    d = build_data(ctx, tok, layout, cache)
    d["_cache_dev0"] = cache["dev"][0]["doc"]
    integ = assert_integrity(cache, d, layout, vocab)
    write_json(os.path.join(ctx.run_dir, "logs", "integrity.json"), integ)
    max_new = layout.max_new(d["train"]["ex"])
    min_new = min(max_new, layout.min_new(d["train"]["ex"], float(cfg.get("min_new_frac", 0.3))))
    ctx.log(f"   data: train {len(cache['train'])} docs -> {len(d['train']['ex'])} sequences, dev "
            f"{len(cache['dev'])} ({len(d['dev']['ex'])} seqs), test {len(cache['test'])}; "
            f"layout {layout.describe()}; max_new {max_new}; summary tokens median "
            f"{stats['summary_tokens'].get('median')} p90 {stats['summary_tokens'].get('p90')}")

    checks = {"selfcheck": "skipped"}
    if not cfg.get("skip_job_checks"):
        if base is not None:
            base.to(dev)
        checks = job_checks(base, model, cfg, d, layout, dev, amp_ctx, tech)
    del base
    if dev.type == "cuda":
        torch.cuda.empty_cache()
    write_json(os.path.join(ctx.run_dir, "logs", "job_checks.json"), checks)
    for c in checks.get("checks", []):
        v = c.get("value")
        ctx.log(f"   [check] {c['name']:<28} {c['kind']:<9} {c['verdict']}"
                + (f" value={v:.3g} thr={c.get('threshold'):.3g}" if isinstance(v, float) else "")
                + (f"  {c.get('error', '')[:200]}" if c["verdict"] == "fail" else ""))
    if checks.get("critical_failed"):
        raise JobExit(6, f"critical self-check failed: {checks['critical_failed']}")

    teacher = None
    if cfg["technique"] == "kd":
        tdir = cfg["teacher_dir"]
        trc = read_json(os.path.join(tdir, "logs", "run_config.json"))
        teacher = rebuild_summ(trc, os.path.join(tdir, "checkpoints", "best.pt"), None, dev)
        for p in teacher.parameters():
            p.requires_grad = False
        ctx.log(f"   [kd] teacher {trc['model']} ({trc['technique']}) loaded from {tdir}; "
                f"alpha {cfg.get('kd_alpha')} T {cfg.get('kd_T')}")

    opt = make_optimizer(groups)
    scaler = make_scaler(amp_dtype == torch.float16)
    stepper = Stepper(ctx, model, opt, None, scaler, tech, amp_ctx, cfg, teacher)
    if mode == "overfit":
        return run_overfit(ctx, stepper, d)

    # ---- pre-flight + plan ----------------------------------------------------------------
    plan_path = os.path.join(ctx.run_dir, "logs", "plan.json")
    last_pt = os.path.join(ctx.run_dir, "checkpoints", "last.pt")
    resume = None
    if os.path.exists(last_pt) and os.path.exists(plan_path) and cfg.get("resume", True):
        try:
            resume = torch_load(last_pt)
            plan = read_json(plan_path)
            d["dev_gen"] = d["dev_gen"][:int(plan["dev_gen"])]
            ctx.log(f"   [resume] from step {resume['st']['step']} (plan reused)")
        except Exception as e:
            ctx.log(f"   [resume] unreadable last.pt ({e}); starting fresh")
            resume = None
    if resume is None:
        gb = int(cfg.get("gen_batch", 16))
        r_train, sps, _n = measure(ctx, stepper, d, layout, cache, gb)
        ctx.log(f"   [preflight] train {r_train:,.0f} tok/s, generation {sps * 1000:.0f} ms/step "
                f"(batch {gb}), micro-batch {stepper.micro} tokens, peak {peak_gb():.2f} GB")
        plan = make_plan(ctx, d, layout, cache, r_train, sps, gb, max_new)
        plan["micro_tokens"] = stepper.micro
        plan["min_new"] = min_new
        plan["selfcheck"] = checks.get("selfcheck")
        write_json(plan_path, plan)
    stepper.micro = int(plan.get("micro_tokens") or stepper.micro)
    ctx.log(f"   [plan] CHOSEN: epochs={plan['epochs']} train_fraction={plan['train_fraction']} "
            f"({plan['train_seqs_used']}/{plan['train_seqs']} seqs), dev_gen {plan['dev_gen']}, "
            f"max_new {plan['max_new']}, generation reserve {plan['reserve_min']:.0f} min, "
            f"projected {plan['projected_minutes']:.0f} of {plan['budget_min']:.0f} min "
            f"({plan['reason']})")

    ex = d["train"]["ex"]
    rng0 = np.random.default_rng(int(cfg["seed"]))
    keep = sorted(rng0.permutation(len(ex))[:max(1, plan["train_seqs_used"])].tolist())
    eff = int(cfg["effective_batch"])
    spe = max(1, math.ceil(len(keep) / eff))
    total = spe * int(plan["epochs"])
    sched = make_scheduler(opt, total, float(cfg.get("warmup_frac", 0.06)))
    stepper.sched = sched
    evals = int(cfg.get("evals_per_epoch", 2))
    eval_at = sorted({max(1, round(spe * (k + 1) / evals)) for k in range(evals)})
    patience = int(cfg.get("patience", 2)) * evals
    st = {"step": 0, "epoch": 0, "pos": 0, "best": float("inf"), "best_epoch": None, "bad": 0,
          "evals": [], "skipped": 0, "train_seconds": 0.0}
    if resume is not None:
        load_trainable_state(model, resume["model"])
        opt.load_state_dict(resume["opt"])
        sched.load_state_dict(resume["sched"])
        scaler.load_state_dict(resume["scaler"])
        st.update(resume["st"])
        try:
            random.setstate(resume["rng"]["py"])
            np.random.set_state(resume["rng"]["np"])
            torch.set_rng_state(resume["rng"]["torch"])
        except Exception:
            pass
        write_json(os.path.join(ctx.run_dir, "logs", "resume.json"), {"resumed_at_step": st["step"]})
        del resume
    deadline = ctx.t0 + float(cfg["budget_s"]) - 60 * float(plan["reserve_min"])
    ctx.status("training", plan=plan["reason"])
    ctx.log(f"   [train] {len(keep)} seqs/epoch, {spe} steps/epoch x {plan['epochs']} = {total} "
            f"steps; training deadline in {(deadline - time.time()) / 60:.0f} min")
    last_ck, t_train, stop = time.time(), time.time(), "completed"
    kill_at = int(cfg.get("kill_at_step", -1))
    ema = None
    while st["epoch"] < int(plan["epochs"]):
        order = [keep[i] for i in np.random.default_rng(int(cfg["seed"]) + 1000 * st["epoch"])
                 .permutation(len(keep))]
        while st["pos"] < spe:
            if time.time() > deadline:
                stop = "time guard"
                break
            batch = order[st["pos"] * eff:(st["pos"] + 1) * eff]
            if not batch:
                st["pos"] = spe
                break
            loss, _t = stepper.step(ex, batch, st["step"], total)
            st["step"] += 1
            st["pos"] += 1
            if loss is None:
                st["skipped"] += 1
                if st["step"] >= 20 and st["skipped"] > 0.05 * st["step"]:
                    raise JobExit(4, f"{st['skipped']} of {st['step']} steps non-finite")
            else:
                ema = loss if ema is None else 0.98 * ema + 0.02 * loss
            if st["step"] % 50 == 0:
                ctx.log(f"   step {st['step']}/{total} ep {st['epoch'] + 1} loss {ema:.4f} lr "
                        f"{sched.get_last_lr()[0]:.2e} peak {peak_gb():.1f} GB skipped "
                        f"{st['skipped']} fp16-overflow {stepper.overflow}")
            if st["step"] == kill_at:
                save_last(ctx, model, opt, sched, scaler, st)
                os._exit(137)
            if st["pos"] in eval_at:
                dl = dev_loss(ctx, model, d["dev"]["ex"], amp_ctx, stepper.micro)
                fe = st["epoch"] + st["pos"] / spe
                rec = {"type": "eval", "step": st["step"], "epoch": round(fe, 3),
                       "train_loss": ema, "dev_loss": dl, "lr": sched.get_last_lr()[0],
                       "minutes": (time.time() - ctx.t0) / 60, "skipped_steps": st["skipped"]}
                ctx.jlog(rec)
                st["evals"].append(rec)
                sel = tech.selectable(st["step"], total)
                ctx.log(f"   [eval] epoch {fe:.2f}: dev loss {dl:.4f} (ppl {math.exp(min(dl, 30)):.1f})"
                        f"{'' if sel else ' (not selectable yet)'}")
                if not math.isfinite(dl):
                    raise JobExit(5, "watchdog: non-finite dev loss")
                if sel and dl < st["best"] - 1e-6:
                    st["best"], st["best_epoch"], st["bad"] = dl, round(fe, 3), 0
                    save_best(ctx, model)
                elif sel:
                    st["bad"] += 1
                write_json(os.path.join(ctx.run_dir, "logs", "progress.json"),
                           {"best_dev_loss": st["best"], "best_epoch": st["best_epoch"],
                            "step": st["step"], "total_steps": total, "epochs_run": round(fe, 3)})
                if sel and st["bad"] >= patience:
                    stop = "early stopping"
                    break
            if time.time() - last_ck > 60 * float(cfg.get("ckpt_every_min", 30)):
                save_last(ctx, model, opt, sched, scaler, st)
                last_ck = time.time()
        if stop != "completed":
            break
        st["epoch"] += 1
        st["pos"] = 0
    st["train_seconds"] = time.time() - t_train
    ctx.log(f"   [train] stopped: {stop} at step {st['step']}/{total}")
    del teacher
    stepper.teacher = None
    best_pt = os.path.join(ctx.run_dir, "checkpoints", "best.pt")
    tech.force_final(model)
    if not os.path.exists(best_pt) or (cfg["technique"] == "adalora" and
                                       not math.isfinite(st["best"])):
        st["best"] = dev_loss(ctx, model, d["dev"]["ex"], amp_ctx, stepper.micro)
        st["best_epoch"] = st["epoch"] + st["pos"] / spe
        save_best(ctx, model)
    load_trainable_state(model, torch_load(best_pt))
    if os.path.exists(last_pt):
        os.remove(last_pt)
    return finish(ctx, model, tok, layout, cache, d, amp_ctx, st, plan, size, checks, stop, tech)


def finish(ctx, model, tok, layout, cache, d, amp_ctx, st, plan, size, checks, stop, tech) -> int:
    cfg = ctx.cfg
    if ctx.device.type == "cuda":
        torch.cuda.empty_cache()
    ctx.status("evaluating")
    t = time.time()
    recs = [cache["dev"][i] for i in d["dev_gen"]]
    dev_raw, n_e1 = summarise(ctx, model, tok, layout, recs, amp_ctx, plan)
    ctx.status("predicting")
    test_texts, n_e2 = summarise(ctx, model, tok, layout, cache["test"], amp_ctx, plan)
    dt = time.time() - t
    model.cpu()
    if ctx.device.type == "cuda":
        torch.cuda.empty_cache()
    ev = evaluate_dev(ctx, dev_raw, recs, layout.median_words)
    rep = write_outputs(ctx, cache, d["dev_gen"], ev, dev_raw, test_texts, n_e1)
    b = ev["best"]
    v = rep["validation"]
    metrics = {
        "dev": rep["scores"], "dev_score": b["score"], "score_metric": b["metric"],
        "dev_bertscore_f1": b["bertscore_f1"], "dev_rouge1": b["rouge1"],
        "dev_rouge2": b["rouge2"], "dev_rougeL": b["rougeL"], "length_ratio": b["ratio"],
        "cap_words": b["cap_words"], "dev_loss": st["best"],
        "dev_ppl": math.exp(min(30.0, st["best"])) if math.isfinite(st["best"]) else None,
        "bertscore_info": ev["bertscore_info"], **rep["diagnostics"],
        "empty_outputs": n_e1 + n_e2, "test_mean_words": rep.get("test_mean_words"),
        "doc_repr": layout.repr, "chunk_tokens": getattr(layout, "C", None),
        "summary_budget": layout.S, "max_new": plan["max_new"], "min_new": plan.get("min_new"),
        "num_beams": int(cfg.get("num_beams", 1)), "dev_gen_docs": len(recs),
        "best_epoch": st["best_epoch"],
        "epochs_run": st["evals"][-1]["epoch"] if st["evals"] else 0, "steps": st["step"],
        "skipped_steps": st["skipped"], "stop_reason": stop,
        "train_minutes": st["train_seconds"] / 60, "total_minutes": (time.time() - ctx.t0) / 60,
        "ms_per_doc_infer": 1000 * dt / max(1, len(recs) + len(cache["test"])),
        "peak_mem_gb": peak_gb(), "train_fraction": plan["train_fraction"],
        "epochs_planned": plan["epochs"], "train_seqs": plan["train_seqs"],
        "train_seqs_used": plan["train_seqs_used"], "tokens_per_s": plan["tokens_per_s_train"],
        "gen_tokens_per_s": plan["gen_batch"] / max(1e-9, plan["gen_sec_per_step"]),
        "projected_minutes": plan["projected_minutes"], "selfcheck": checks.get("selfcheck"),
        "validation_ok": v["ok"], "degenerate": v.get("degenerate"),
        "technique_report": tech.final_report(model), **size,
    }
    write_json(os.path.join(ctx.run_dir, "metrics.json"), metrics)
    if not v["ok"]:
        ctx.status("failed", f"submission validation failed: {v['errors']}")
        return 4
    if v.get("degenerate"):
        ctx.log("   !!! WARNING: every test document got the same summary (degenerate)")
    write_json(os.path.join(ctx.run_dir, "DONE.json"),
               {"finished": now_str(), "dev_score": b["score"], "score_metric": b["metric"],
                "stop_reason": stop, "submission": os.path.join(ctx.run_dir, "submission",
                                                                 "submission.zip")})
    ctx.status("done", f"dev {b['metric']} {b['score']}")
    f = lambda x: "n/a" if x is None else f"{x:.4f}"  # noqa: E731
    ctx.log(f"   DONE dev BERTScore-F1 {f(b['bertscore_f1'])}, ROUGE-1 {f(b['rouge1'])}, ROUGE-2 "
            f"{f(b['rouge2'])}, ROUGE-L {f(b['rougeL'])} (length cap {b['ratio']} x median = "
            f"{b['cap_words']} words; {ev['bertscore_info']}); dev loss {f(st['best'])}; test mean "
            f"{rep.get('test_mean_words') or 0:.0f} words; submission validated")
    return 0


def run_overfit(ctx, stepper, d) -> int:
    """8 training sequences, up to 80 steps: the summary-token loss must fall below 25% of its
    start (or below 0.5)."""
    ex = d["train"]["ex"]
    idx = list(range(min(8, len(ex))))
    for g in stepper.opt.param_groups:
        g["lr"] = g["initial_lr"] = max(g["lr"], float(ctx.cfg.get("overfit_lr", 3e-3)))
    stepper.sched = None
    first = last = None
    n = 0
    for n in range(1, 81):
        l, _ = stepper.step(ex, idx, n, 80)
        if l is not None:
            first = l if first is None else first
            last = l
        if last is not None and (last < 0.5 or last < 0.25 * first):
            break
    ok = last is not None and first is not None and (last < 0.5 or last < 0.25 * first)
    res = {"passed": bool(ok), "first_loss": first, "final_loss": last, "steps": n}
    write_json(os.path.join(ctx.run_dir, "logs", "overfit.json"), res)
    ctx.status("done", f"overfit {res}")
    return 0 if ok else 1


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-dir", required=True)
    ap.add_argument("--mode", default="train", choices=["train", "overfit"])
    a = ap.parse_args()
    cfg = read_json(os.path.join(a.run_dir, "logs", "run_config.json"))
    ctx = Ctx(a.run_dir, cfg)
    code, reason = 1, "unknown"
    try:
        code = run_train(ctx, a.mode)
        reason = "ok" if code == 0 else f"exit {code}"
    except JobExit as e:
        code, reason = e.code, e.reason
        ctx.log(f"   JOB EXIT {code}: {reason}")
    except Exception as e:
        code = 7 if is_oom(e) else 1
        reason = f"{type(e).__name__}: {e}"
        ctx.log(f"   JOB ERROR: {reason}\n{traceback.format_exc()[-3000:]}")
    finally:
        st = read_json(os.path.join(a.run_dir, "status.json"), {}) or {}
        if code != 0 and st.get("state") not in ("failed",):
            ctx.status("failed", reason, exit_code=code)
        elif code == 0 and st.get("state") not in ("done", "degenerate"):
            ctx.status("done", reason)
    sys.stdout.flush()
    sys.exit(code)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase6/cjpe_predict.py
"""
cjpe_predict.py -- batched inference, the dev/test output files, the submission zip and its
validation (Phase 5, multi-label).

Every finished model writes:
  dev/dev_probs.npz       ids, gold (uint8), probs (float16)      dev/dev_report.json
  dev/per_label_dev.csv   label, supports, P/R/F1, predicted count, tau
  dev/errors.csv          dev docs whose label set is wrong (missed / spurious ids, snippet)
  dev/dev_metrics.json    scores of the chosen rule (+ global, 0.5)
  submission/test_probs.npz, submission/prediction.json, submission/submission.zip,
  submission/validation_report.json
"""
import csv
import json
import os
import zipfile
from typing import Dict, List

import numpy as np
import torch

from cjpe_common import write_json
from lsi_ml import apply_rule, full_report, rule_taus
from cjpe_models import collate_docs, forward_batch
from validate_submission import validate


def batches_by_chunks(chunk_lists: List[List[np.ndarray]], max_chunks: int,
                      order=None) -> List[List[int]]:
    """Group document indices so that each group has <= max_chunks chunks (>= 1 doc)."""
    order = list(range(len(chunk_lists))) if order is None else list(order)
    out, cur, n = [], [], 0
    for i in order:
        c = len(chunk_lists[i])
        if cur and n + c > max_chunks:
            out.append(cur)
            cur, n = [], 0
        cur.append(i)
        n += c
    if cur:
        out.append(cur)
    return out


@torch.no_grad()
def predict_logits(model, chunk_lists, device, amp_ctx, max_chunks: int = 32) -> np.ndarray:
    """(n_docs, L) fp32 logits; documents sorted by length for less padding, then unsorted."""
    model.eval()
    n = len(chunk_lists)
    out = np.zeros((n, model.classifier.out.out_features), np.float32)
    order = sorted(range(n), key=lambda i: (len(chunk_lists[i]),
                                            max(len(c) for c in chunk_lists[i])))
    mc = max_chunks
    groups = batches_by_chunks(chunk_lists, mc, order)
    gi = 0
    while gi < len(groups):
        g = groups[gi]
        try:
            b = collate_docs([chunk_lists[i] for i in g], device)
            with amp_ctx():
                logits, _ = forward_batch(model, b)
            out[g] = logits.float().cpu().numpy()
            gi += 1
        except RuntimeError as e:                      # OOM: smaller groups, same order
            if "out of memory" not in str(e).lower() or mc <= 1:
                raise
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            mc = max(1, mc // 2)
            rest = [i for gg in groups[gi:] for i in gg]
            groups = groups[:gi] + batches_by_chunks(chunk_lists, mc, rest)
    return out


def softmax1(logits: np.ndarray) -> np.ndarray:
    z = logits - logits.max(1, keepdims=True)
    e = np.exp(z)
    return (e[:, 1] / e.sum(1)).astype(np.float64)


def _write_rows(path: str, header: List[str], rows: List[List]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    os.replace(tmp, path)


def _strip(rep: Dict) -> Dict:
    return {k: v for k, v in rep.items() if not k.startswith("_")}


def write_dev_outputs(run_dir: str, ids, G, Pr, rule, n_tokens, texts_snip: Dict[str, str],
                      label_ids, train_support, n_dev_only_pos: int = 0) -> Dict:
    rep = full_report(G, Pr, rule, n_tokens, label_ids, train_support, n_dev_only_pos)
    P = rep["_P"]
    dd = os.path.join(run_dir, "dev")
    os.makedirs(dd, exist_ok=True)
    np.savez_compressed(os.path.join(dd, "dev_probs.npz"), ids=np.asarray(ids),
                        gold=G.astype(np.uint8), probs=Pr.astype(np.float16),
                        label_ids=np.asarray(label_ids))
    pl = rep["_per_label"]
    _write_rows(os.path.join(dd, "per_label_dev.csv"), list(pl[0].keys()),
                [list(r.values()) for r in pl])
    lab = np.asarray(label_ids)
    err = []
    for k, i in enumerate(ids):
        g, p = G[k].astype(bool), P[k]
        if (g != p).any():
            err.append([i, " ".join(map(str, lab[g & ~p])), " ".join(map(str, lab[p & ~g])),
                        int(g.sum()), int(p.sum()), int(n_tokens[k]), texts_snip.get(i, "")])
    if not err:
        err.append(["", "", "", "", "", "", "no dev document with a wrong label set"])
    _write_rows(os.path.join(dd, "errors.csv"),
                ["id", "missed", "spurious", "n_gold", "n_pred", "n_tokens", "snippet"], err)
    out = _strip(rep)
    write_json(os.path.join(dd, "dev_report.json"), out)
    write_json(os.path.join(dd, "dev_metrics.json"),
               {"rule": out["rule"], "scores": out["scores"], "scores_global": out["scores_global"],
                "scores_at_0p5": out["scores_at_0p5"]})
    return rep


def write_submission(run_dir: str, test_ids: List[str], Pr, rule: Dict, label_ids) -> Dict:
    L = Pr.shape[1]
    P = apply_rule(Pr, rule_taus(rule, L), int(rule["kmax"]))
    lab = np.asarray(label_ids, dtype=np.int64)
    sub = os.path.join(run_dir, "submission")
    os.makedirs(sub, exist_ok=True)
    np.savez_compressed(os.path.join(sub, "test_probs.npz"), ids=np.asarray(test_ids),
                        probs=Pr.astype(np.float16), label_ids=lab)
    items = [{"id": str(i), "pred": sorted(int(x) for x in set(lab[P[k]].tolist()))}
             for k, i in enumerate(test_ids)]
    pj = os.path.join(sub, "prediction.json")
    with open(pj + ".tmp", "w", encoding="utf-8") as f:
        json.dump(items, f)
    os.replace(pj + ".tmp", pj)
    zp = os.path.join(sub, "submission.zip")
    with zipfile.ZipFile(zp + ".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")
    os.replace(zp + ".tmp", zp)
    rep = validate(zp, test_ids, [int(x) for x in lab])
    rep["rule"] = rule.get("name")
    for f in ("dev_probs.npz", "per_label_dev.csv", "errors.csv"):
        p = os.path.join(run_dir, "dev", f)
        rep[f"{f}_bytes"] = os.path.getsize(p) if os.path.exists(p) else 0
    rep["test_probs.npz_bytes"] = os.path.getsize(os.path.join(sub, "test_probs.npz"))
    empty = [k for k in rep if k.endswith("_bytes") and not rep[k]]
    if empty:
        rep["ok"] = False
        rep["errors"].append(f"empty output files: {empty}")
    write_json(os.path.join(sub, "validation_report.json"), rep)
    return rep


In [ ]:
%%writefile /kaggle/working/phase6/validate_submission.py
"""
validate_submission.py -- block any invalid Phase-6 (summarisation) submission.

Checks: the archive is a zip whose only entry is `prediction.json` at its ROOT; it is a JSON list
of {"id": str, "summary": str} (PDF 12.3.5, list format); ids are exactly the test.json ids (no
missing, extra or duplicate); every summary is a non-empty string; plus a degenerate check
(every document got the same summary).

    python validate_submission.py submission.zip --test-ids test_ids.json
"""
import argparse
import json
import sys
import zipfile
from typing import Dict, Iterable, Optional


def validate(zip_path: str, test_ids: Iterable[str], valid_ids: Optional[Iterable] = None) -> Dict:
    test_ids = [str(i) for i in test_ids]
    rep = {"zip": zip_path, "ok": False, "errors": [], "degenerate": False}
    try:
        with zipfile.ZipFile(zip_path) as z:
            names = z.namelist()
            rep["entries"] = names
            if names != ["prediction.json"]:
                rep["errors"].append(f"zip must hold exactly prediction.json at its root: {names[:5]}")
                return rep
            items = json.loads(z.read("prediction.json").decode("utf-8"))
    except Exception as e:
        rep["errors"].append(f"unreadable: {type(e).__name__}: {e}")
        return rep
    if not isinstance(items, list) or not items:
        rep["errors"].append("prediction.json must be a non-empty JSON list")
        return rep
    seen, bad = [], 0
    for it in items:
        if not (isinstance(it, dict) and isinstance(it.get("id"), str)
                and isinstance(it.get("summary"), str) and it["summary"].strip()):
            bad += 1
            continue
        seen.append(it["id"])
    if bad:
        rep["errors"].append(f"{bad} items are not {{'id': str, 'summary': non-empty str}}")
    if len(seen) != len(set(seen)):
        rep["errors"].append(f"{len(seen) - len(set(seen))} duplicate ids")
    miss, extra = set(test_ids) - set(seen), set(seen) - set(test_ids)
    if miss:
        rep["errors"].append(f"{len(miss)} test ids missing, e.g. {sorted(miss)[:3]}")
    if extra:
        rep["errors"].append(f"{len(extra)} unknown ids, e.g. {sorted(extra)[:3]}")
    summ = [it.get("summary", "") for it in items if isinstance(it, dict)]
    rep["n"] = len(items)
    rep["mean_words"] = sum(len(str(s).split()) for s in summ) / max(1, len(summ))
    rep["degenerate"] = len(summ) > 1 and len(set(summ)) == 1
    rep["ok"] = not rep["errors"]
    return rep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("zip")
    ap.add_argument("--test-ids", required=True, help="JSON list of test ids")
    a = ap.parse_args()
    with open(a.test_ids) as f:
        ids = json.load(f)
    r = validate(a.zip, ids)
    print(json.dumps({k: v for k, v in r.items() if k != "entries"}, indent=1))
    sys.exit(0 if r["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase6/select_best.py
"""
select_best.py -- per-experiment BEST selection and the six-upload plan (sections 11.2, 11.3).

    python select_best.py --exp-dir /kaggle/working/phase5_out/E2_lorma      # BEST/ for one exp
    python select_best.py --plan [--out-root ...]                             # UPLOAD_PLAN.csv

BEST: highest dev score among DONE jobs (not DEGENERATE, not the E6 teacher); ties -> fewer
trainable parameters.

Plan: exactly one row per counted experiment E1..E6, every Phase-1 model at least once (relaxed
only when impossible, and then reported), FAILED / DEGENERATE / missing-zip rows excluded,
maximise the mean dev score, and the six prediction.json files must be pairwise different
(identical submissions do not count on Codabench). Exhaustive search (<= 4^6 combinations).
"""
import argparse
import hashlib
import itertools
import json
import os
import shutil
import zipfile
from typing import Dict, List, Optional

from cjpe_common import (COUNTED, EXPERIMENTS, MODELS, OUT_ROOT, default_input_roots,
                         read_csv, read_json, write_csv, write_json)


def _f(x, default=-1.0) -> float:
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def _unify(rows: List[Dict]) -> List[Dict]:
    """dev_score := one metric for all rows (BERTScore only when every DONE row has it)."""
    from aggregate import common_metric
    m = common_metric(rows)
    for r in rows:
        r["dev_score"] = r.get(m)
    return rows


def prediction_hash(zip_path: str) -> Optional[str]:
    try:
        with zipfile.ZipFile(zip_path) as z:
            items = json.loads(z.read("prediction.json"))
        m = sorted((str(r["id"]), str(r["summary"])) for r in items)
        return hashlib.md5(json.dumps(m).encode()).hexdigest()
    except Exception:
        return None


def is_candidate(r: Dict, require_technique: Optional[str] = None) -> bool:
    if r.get("status") != "DONE":
        return False
    if r.get("technique", "").endswith("(teacher)"):
        return False
    if require_technique and r.get("technique") != require_technique:
        return False
    z = r.get("submission_zip")
    return bool(z) and os.path.exists(z)


def select_exp(exp_dir: str, rows: List[Dict], log=print) -> Optional[Dict]:
    exp = os.path.basename(exp_dir.rstrip("/"))
    rows = _unify(rows)
    cands = [r for r in rows if is_candidate(r)]
    best_dir = os.path.join(exp_dir, "BEST")
    if not cands:
        log(f"   [select] {exp}: no finished, non-degenerate model -> no BEST/")
        degen = [r for r in rows if r.get("status") == "DEGENERATE" and r.get("submission_zip")
                 and os.path.exists(r["submission_zip"])]
        if not degen:
            return None
        log(f"   [select] {exp}: only DEGENERATE models exist; keeping the best of them in BEST/ "
            f"(flagged)")
        cands = degen
    cands.sort(key=lambda r: (-_f(r.get("dev_score")), _f(r.get("trainable_params"), 1e18)))
    w = cands[0]
    os.makedirs(best_dir, exist_ok=True)
    shutil.copy2(w["submission_zip"], os.path.join(best_dir, "submission.zip"))
    run_dir = os.path.dirname(os.path.dirname(w["submission_zip"]))
    rc = os.path.join(run_dir, "logs", "run_config.json")
    if os.path.exists(rc):
        shutil.copy2(rc, os.path.join(best_dir, "run_config.json"))
    sel = {"experiment": w.get("experiment"), "job": w.get("job"), "model_id": w.get("model_id"),
           "technique": w.get("technique"), "dev_score": _f(w.get("dev_score")),
           "status": w.get("status"), "run_dir": run_dir,
           "ranking": [{"job": r.get("job"), "model_id": r.get("model_id"),
                        "dev_score": _f(r.get("dev_score")),
                        "trainable_params": r.get("trainable_params")} for r in cands]}
    write_json(os.path.join(best_dir, "SELECTION.json"), sel)
    log(f"   [select] {exp}: BEST = {w.get('job')} (dev score {_f(w.get('dev_score')):.4f})")
    return sel


def plan_uploads(rows: List[Dict], out_path: str, log=print) -> Dict:
    by_exp: Dict[str, List[Dict]] = {}
    rows = _unify(rows)
    for e in COUNTED:
        tech = EXPERIMENTS[e]["technique"]
        c = [r for r in rows if r.get("experiment") == e and is_candidate(r, tech)]
        for r in c:
            r["_hash"] = prediction_hash(r["submission_zip"])
        by_exp[e] = sorted([r for r in c if r["_hash"]],
                           key=lambda r: -_f(r.get("dev_score")))
    missing = [e for e in COUNTED if not by_exp[e]]
    exps = [e for e in COUNTED if by_exp[e]]
    best, best_key, relaxed = None, None, False
    # last resort (Phase 5): if no combination has pairwise different predictions (e.g. weak
    # models all predicting the most frequent statute), still plan one upload per experiment
    for require_all, distinct in ((True, True), (False, True), (True, False), (False, False)):
        for combo in itertools.product(*[by_exp[e] for e in exps]):
            hashes = [r["_hash"] for r in combo]
            if distinct and len(set(hashes)) != len(hashes):
                continue
            models = {r.get("model_id") for r in combo}
            if require_all and len(models) < min(4, len(exps)):
                continue
            key = (len(models), sum(_f(r.get("dev_score")) for r in combo))
            if best is None or key > best_key:
                best, best_key = combo, key
        if best is not None:
            relaxed = not (require_all and distinct)
            break
    plan_rows = []
    if best:
        first = [r for r in best if r.get("experiment") in ("E1", "E2")]
        ordered = first[:1] + [r for r in best if r not in first[:1]]
        for k, r in enumerate(ordered, 1):
            plan_rows.append({"order": k, "experiment": r.get("experiment"),
                              "technique": r.get("technique"), "model_id": r.get("model_id"),
                              "job": r.get("job"), "dev_score": r.get("dev_score"),
                              "path": r.get("submission_zip"), "role": "required"})
        used = {r.get("submission_zip") for r in best}
        used_h = {r["_hash"] for r in best}
        extra = 0
        for m in MODELS:
            pool = [r for e in exps for r in by_exp[e] if r.get("model_id") == m
                    and r.get("submission_zip") not in used and r["_hash"] not in used_h]
            if pool and extra < 4:
                r = max(pool, key=lambda r: _f(r.get("dev_score")))
                extra += 1
                used_h.add(r["_hash"])
                plan_rows.append({"order": len(plan_rows) + 1, "experiment": r.get("experiment"),
                                  "technique": r.get("technique"), "model_id": m,
                                  "job": r.get("job"), "dev_score": r.get("dev_score"),
                                  "path": r.get("submission_zip"), "role": "optional extra"})
    write_csv(out_path, plan_rows, ["order", "experiment", "technique", "model_id", "job",
                                    "dev_score", "role", "path"])
    models = sorted({r["model_id"] for r in plan_rows if r["role"] == "required"})
    rep = {"n_required": sum(1 for r in plan_rows if r["role"] == "required"),
           "missing_experiments": missing, "models_covered": models,
           "all_models_covered": len(models) == 4, "relaxed_model_constraint": relaxed}
    log("   [plan] upload order (Codabench, <= 10/day):")
    for r in plan_rows:
        log(f"   [plan] {r['order']:>2}. {r['experiment']} {r['technique']:<11} {r['model_id']} "
            f"dev score {_f(r['dev_score']):.4f} [{r['role']}]  {r['path']}")
    if missing:
        log(f"   [plan] MISSING experiments (no valid submission yet): {missing}")
    if plan_rows and not rep["all_models_covered"]:
        log(f"   [plan] WARNING: only models {models} appear among the six uploads")
    return rep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp-dir")
    ap.add_argument("--plan", action="store_true")
    ap.add_argument("--out-root", default=OUT_ROOT)
    a = ap.parse_args()
    if a.exp_dir:
        f = [p for p in os.listdir(a.exp_dir) if p.startswith("results_") and p.endswith(".csv")]
        rows = read_csv(os.path.join(a.exp_dir, f[0])) if f else []
        select_exp(a.exp_dir, rows)
    if a.plan:
        from aggregate import aggregate
        rows = aggregate(a.out_root, default_input_roots())
        plan_uploads(rows, os.path.join(a.out_root, "UPLOAD_PLAN.csv"))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase6/aggregate.py
"""
aggregate.py -- one CSV row per (experiment, job) from the files a job leaves behind, the
per-experiment CSVs, ALL_RESULTS.csv + pivots, and the E8 report (ensemble, error analysis).

    python aggregate.py [--out-root /kaggle/working/phase5_out] [--report]

`build_row(run_dir)` reads run_config.json, status.json, plan.json, progress.json, metrics.json,
model_size.json and job_checks.json; it never fails (missing files -> empty cells), so the CSV
can be rewritten at any moment (rule R2: a row exists for every planned job from minute one).
"""
import argparse
import glob
import json
import os
import shutil
from typing import Dict, List, Optional

import numpy as np

from cjpe_common import (COUNTED, CSV_COLUMNS, EPOCH_COLUMNS, EXPERIMENTS, MODELS, OUT_ROOT,
                         default_input_roots, iter_out_dirs, read_csv, read_json, write_csv,
                         write_json, log_to)

STATE_TO_STATUS = {"done": "DONE", "degenerate": "DEGENERATE", "failed": "FAILED",
                   "skipped_time": "SKIPPED_TIME", "preflight": "RUNNING",
                   "training": "RUNNING", "evaluating": "RUNNING", "predicting": "RUNNING",
                   "pending": "PENDING", "queued": "PENDING", "smoke_ok": "SMOKE_OK"}


def build_row(run_dir: str, teacher_total: Optional[float] = None) -> Dict:
    rc = read_json(os.path.join(run_dir, "logs", "run_config.json"), {}) or {}
    st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
    plan = read_json(os.path.join(run_dir, "logs", "plan.json"), {}) or {}
    prog = read_json(os.path.join(run_dir, "logs", "progress.json"), {}) or {}
    met = read_json(os.path.join(run_dir, "metrics.json"), {}) or {}
    size = read_json(os.path.join(run_dir, "logs", "model_size.json"), {}) or {}
    chk = read_json(os.path.join(run_dir, "logs", "job_checks.json"), {}) or {}
    status = STATE_TO_STATUS.get(st.get("state", "pending"), "PENDING")
    if status == "FAILED" and st.get("exit_code") == 2:
        status = "SKIPPED_TIME"
    if status == "DONE" and not os.path.exists(os.path.join(run_dir, "DONE.json")):
        status = "FAILED" if met else status
    zp = os.path.join(run_dir, "submission", "submission.zip")
    tech = rc.get("technique", "")
    if rc.get("is_teacher"):
        tech = f"{tech} (teacher)"
    tot = size.get("total_params") or met.get("total_params")
    row = {
        "experiment": rc.get("experiment"), "technique": tech, "model_id": rc.get("model"),
        "arch": rc.get("arch"), "doc_repr": met.get("doc_repr", plan.get("doc_repr",
                                                                         rc.get("doc_repr"))),
        "chunk_tokens": met.get("chunk_tokens", plan.get("chunk_tokens")),
        "summary_budget": met.get("summary_budget"), "max_new": met.get("max_new",
                                                                         plan.get("max_new")),
        "num_beams": rc.get("num_beams"), "status": status,
        "fail_reason": "" if status in ("DONE", "PENDING", "RUNNING") else st.get("reason", ""),
        "selfcheck": (met.get("selfcheck", chk.get("selfcheck", "")) or "") + (
            " DEGENERATE" if met.get("degenerate") else ""),
        "train_docs": plan.get("train_docs"), "train_seqs": plan.get("train_seqs"),
        "train_fraction": met.get("train_fraction", plan.get("train_fraction")),
        "dev_docs": rc.get("n_dev"), "dev_gen_docs": met.get("dev_gen_docs", plan.get("dev_gen")),
        "test_docs": rc.get("n_test"),
        "epochs_planned": met.get("epochs_planned", plan.get("epochs")),
        "epochs_run": met.get("epochs_run", prog.get("epochs_run")),
        "best_epoch": met.get("best_epoch", prog.get("best_epoch")), "lr": rc.get("lr"),
        "batch_effective": rc.get("effective_batch"),
        "dev_loss": met.get("dev_loss", prog.get("best_dev_loss")),
        "trainable_params": size.get("trainable_params"), "total_params": tot,
        "trainable_pct": size.get("trainable_pct"),
        "params_ratio_vs_teacher": (tot / teacher_total) if (tot and teacher_total and rc.get(
            "technique") in ("kd", "student_hard")) else None,
        "projected_minutes": plan.get("projected_minutes"),
        "tokens_per_s": plan.get("tokens_per_s_train"), "gpu_slot": rc.get("gpu_slot"),
        "attempts": rc.get("attempt"), "train_files": "train.json",
        "leak_guard": rc.get("leak_guard", "passed"), "seed": rc.get("seed"),
        "job": rc.get("job", os.path.basename(run_dir)),
        "submission_zip": zp if os.path.exists(zp) else "",
        "started_at": rc.get("started_at"), "finished_at": st.get("time") if status in (
            "DONE", "FAILED", "DEGENERATE", "SKIPPED_TIME") else "",
    }
    for k in ("score_metric", "dev_score", "dev_bertscore_f1", "dev_rouge1", "dev_rouge2",
              "dev_rougeL", "dev_ppl", "length_ratio", "cap_words", "words", "ref_words",
              "compression", "repeated_trigram_rate", "novel_bigram_rate", "content_support",
              "number_precision", "citation_precision", "ref_content_recall", "empty_outputs",
              "test_mean_words", "train_minutes", "gen_tokens_per_s", "ms_per_doc_infer",
              "peak_mem_gb"):
        row[k] = met.get(k)
    return row


def epoch_rows(run_dir: str) -> List[Dict]:
    rc = read_json(os.path.join(run_dir, "logs", "run_config.json"), {}) or {}
    p = os.path.join(run_dir, "logs", "train_log.jsonl")
    out = []
    if os.path.exists(p):
        with open(p) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except Exception:
                    continue
                if r.get("type") == "eval":
                    out.append({"experiment": rc.get("experiment"), "model_id": rc.get("model"),
                                "job": rc.get("job"), **r})
    return out


def exp_rows(exp_dir: str, planned: Optional[List[Dict]] = None) -> List[Dict]:
    """Rows for every job folder of an experiment (+ PENDING rows for planned jobs)."""
    jobs = sorted(d for d in os.listdir(exp_dir) if os.path.isdir(os.path.join(exp_dir, d))
                  and d != "BEST") if os.path.isdir(exp_dir) else []
    teacher_total = None
    for j in jobs:
        rc = read_json(os.path.join(exp_dir, j, "logs", "run_config.json"), {}) or {}
        if rc.get("is_teacher"):
            teacher_total = (read_json(os.path.join(exp_dir, j, "logs", "model_size.json"), {})
                             or {}).get("total_params")
    rows = {}
    for p in planned or []:
        rows[p["job"]] = dict(p)
    for j in jobs:
        if not os.path.exists(os.path.join(exp_dir, j, "logs", "run_config.json")):
            st = read_json(os.path.join(exp_dir, j, "status.json"), {}) or {}
            if j in rows and st.get("state") in STATE_TO_STATUS:
                rows[j]["status"] = STATE_TO_STATUS[st["state"]]
                if rows[j]["status"] not in ("PENDING", "RUNNING"):
                    rows[j]["fail_reason"] = st.get("reason", "")
            continue
        r = build_row(os.path.join(exp_dir, j), teacher_total)
        if j in rows:
            r = {**rows[j], **{k: v for k, v in r.items() if v not in (None, "")}}
            if r.get("status") == "PENDING" and rows[j].get("status") not in (None, "PENDING"):
                r["status"] = rows[j]["status"]
                r["fail_reason"] = rows[j].get("fail_reason", "")
        rows[j] = r
    return list(rows.values())


def write_exp_csvs(exp_dir: str, planned: Optional[List[Dict]] = None) -> List[Dict]:
    folder = os.path.basename(exp_dir.rstrip("/"))
    rows = exp_rows(exp_dir, planned)
    write_csv(os.path.join(exp_dir, f"results_{folder}.csv"), rows, CSV_COLUMNS)
    ep = []
    for j in sorted(os.listdir(exp_dir)) if os.path.isdir(exp_dir) else []:
        if os.path.isdir(os.path.join(exp_dir, j)) and j != "BEST":
            ep += epoch_rows(os.path.join(exp_dir, j))
    write_csv(os.path.join(exp_dir, f"epoch_log_{folder}.csv"), ep, EPOCH_COLUMNS)
    return rows


def _status_rank(s: str) -> int:
    return {"DONE": 5, "DEGENERATE": 4, "RUNNING": 3, "FAILED": 2, "SKIPPED_TIME": 2,
            "PENDING": 1}.get(s, 0)


def aggregate(out_root: str = OUT_ROOT, input_roots=None, log=print) -> List[Dict]:
    """Merge every results_*.csv (working dir first, then attached inputs) -> ALL_RESULTS.csv.
    submission_zip paths are re-resolved where the CSV was found."""
    input_roots = default_input_roots() if input_roots is None else input_roots
    best: Dict = {}
    for od in iter_out_dirs(out_root, input_roots):
        for csvp in sorted(glob.glob(os.path.join(od, "*", "results_*.csv"))):
            exp_dir = os.path.dirname(csvp)
            for r in read_csv(csvp):
                job = r.get("job") or ""
                z = os.path.join(exp_dir, job, "submission", "submission.zip")
                r["submission_zip"] = z if (job and os.path.exists(z)) else ""
                r["_run_dir"] = os.path.join(exp_dir, job)
                key = (r.get("experiment"), job)
                old = best.get(key)
                if old is None or _status_rank(r.get("status", "")) > _status_rank(
                        old.get("status", "")):
                    best[key] = r
    rows = sorted(best.values(), key=lambda r: (str(r.get("experiment")), str(r.get("job"))))
    if out_root:
        os.makedirs(out_root, exist_ok=True)
        write_csv(os.path.join(out_root, "ALL_RESULTS.csv"), rows, CSV_COLUMNS)
        write_pivots(out_root, rows)
    return rows


def write_pivots(out_root: str, rows: List[Dict]) -> None:
    for metric, name in (("dev_bertscore_f1", "ALL_RESULTS_pivot.csv"),
                         ("dev_rouge1", "ALL_RESULTS_pivot_rouge1.csv"),
                         ("dev_rougeL", "ALL_RESULTS_pivot_rougeL.csv"),
                         ("trainable_pct", "ALL_RESULTS_pivot_trainable_pct.csv"),
                         ("train_minutes", "ALL_RESULTS_pivot_train_minutes.csv")):
        table: Dict[str, Dict] = {}
        for r in rows:
            if r.get("status") not in ("DONE", "DEGENERATE"):
                continue
            k = f"{r.get('experiment')}:{r.get('technique')}"
            if r.get("experiment") == "E7":
                k = f"E7:{r.get('job', '').rsplit('_M', 1)[0]}"
            table.setdefault(k, {"experiment_technique": k})
            table[k][r.get("model_id")] = r.get(metric)
        write_csv(os.path.join(out_root, name), list(table.values()),
                  ["experiment_technique", "M1", "M2", "M3", "M4"])


# =============================================================================================
# E8 report: consensus ensemble of the BEST runs + summary-quality analysis + figures
# =============================================================================================
def _score(r: Dict, metric: str) -> float:
    try:
        return float(r.get(metric))
    except (TypeError, ValueError):
        return -1.0


def common_metric(rows: List[Dict]) -> str:
    """BERTScore when EVERY finished run has it, else the ROUGE mean (never mix the two)."""
    done = [r for r in rows if r.get("status") == "DONE"]
    if done and all(r.get("dev_bertscore_f1") not in (None, "") for r in done):
        return "dev_bertscore_f1"
    for r in rows:
        try:
            r["dev_rouge_mean"] = 0.5 * (float(r["dev_rouge1"]) + float(r["dev_rougeL"]))
        except (TypeError, ValueError, KeyError):
            r["dev_rouge_mean"] = None
    return "dev_rouge_mean"


def _best_per_exp(rows: List[Dict]) -> List[Dict]:
    m = common_metric(rows)
    out = []
    for e in COUNTED:
        c = [r for r in rows if r.get("experiment") == e and r.get("status") == "DONE"
             and not r.get("technique", "").endswith("(teacher)")]
        if c:
            out.append(max(c, key=lambda r: _score(r, m)))
    return out


def _jsonl(path: str) -> List[Dict]:
    out = []
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                try:
                    out.append(json.loads(line))
                except Exception:
                    pass
    return out


def ensemble(rows: List[Dict], out_root: str, log=print) -> Optional[Dict]:
    """Consensus (MBR) ensemble: per document, the member summary with the highest mean ROUGE-L
    to the other members' summaries. Scored on the dev documents every member decoded (ROUGE;
    optimistic because dev chose the members); written as a ready submission."""
    import zipfile
    from gen_ml import consensus_pick, rouge_all
    from validate_submission import validate
    mem = []
    for m in _best_per_exp(rows):
        dv = {x["id"]: x for x in _jsonl(os.path.join(m["_run_dir"], "dev", "dev_predictions.jsonl"))}
        tp = os.path.join(m["_run_dir"], "submission", "prediction.json")
        if dv and os.path.exists(tp):
            mem.append((m, dv, {x["id"]: x["summary"] for x in json.load(open(tp, encoding="utf-8"))}))
    if len(mem) < 3:
        log("   [ensemble] fewer than 3 members with saved predictions -> skipped")
        return None
    ids = sorted(set.intersection(*[set(dv) for _, dv, _ in mem]))
    refs = [mem[0][1][i]["reference"] for i in ids]
    ens = [mem[0][1][i]["prediction"] for i in ids]
    for k, i in enumerate(ids):
        c = [dv[i]["prediction"] for _, dv, _ in mem]
        ens[k] = c[consensus_pick(c)]
    r_ens = rouge_all(ens, refs)
    singles = {f"{m.get('experiment')}:{m.get('model_id')}":
               rouge_all([dv[i]["prediction"] for i in ids], refs) for m, dv, _ in mem}
    mean = lambda r: 0.5 * (r["rouge1"] + r["rougeL"])  # noqa: E731
    best_single = max(mean(v) for v in singles.values())
    test_ids = read_json(os.path.join(out_root, "test_ids.json"), None) or sorted(mem[0][2])
    items = []
    for i in test_ids:
        c = [tp.get(i, "") for _, _, tp in mem]
        items.append({"id": i, "summary": c[consensus_pick(c)] or "."})
    d = os.path.join(out_root, "E8_report", "ensemble")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(items, f, ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(d, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(d, "prediction.json"), "prediction.json")
    v = validate(os.path.join(d, "submission.zip"), test_ids)
    rep = {"members": list(singles), "dev_docs": len(ids),
           "ensemble_rouge1": r_ens["rouge1"], "ensemble_rougeL": r_ens["rougeL"],
           "best_single_rouge_mean": best_single, "validation_ok": v["ok"],
           "use_it": bool(v["ok"] and mean(r_ens) > best_single + 0.002),
           "note": "ROUGE only (BERTScore needs the GPU); dev chose the members -> optimistic"}
    write_json(os.path.join(d, "ensemble_report.json"), rep)
    log(f"   [ensemble] {len(mem)} members on {len(ids)} dev docs: ROUGE mean "
        f"{mean(r_ens):.4f} vs best single {best_single:.4f} -> use_it={rep['use_it']}")
    return rep


def error_analysis(rows: List[Dict], out_root: str, dev_texts=None, log=print) -> None:
    """Per-experiment summary-quality tables: length curves, document-length buckets,
    redundancy / abstractiveness / factuality proxies, the worst dev documents, and plots."""
    rep_dir = os.path.join(out_root, "E8_report")
    os.makedirs(rep_dir, exist_ok=True)
    curves, buckets, worst = [], [], []
    done = [r for r in rows if r.get("status") == "DONE" and r.get("_run_dir")]
    for r in done:
        base = {"experiment": r.get("experiment"), "technique": r.get("technique"),
                "model_id": r.get("model_id"), "job": r.get("job"), "doc_repr": r.get("doc_repr")}
        for x in read_csv(os.path.join(r["_run_dir"], "dev", "length_curve.csv")):
            curves.append({**base, **x})
        for x in read_csv(os.path.join(r["_run_dir"], "dev", "length_buckets.csv")):
            buckets.append({**base, **x})
        preds = _jsonl(os.path.join(r["_run_dir"], "dev", "dev_predictions.jsonl"))
        for x in sorted(preds, key=lambda x: x.get("rougeL") or 0)[:5]:
            worst.append({**base, "id": x["id"], "rougeL": x.get("rougeL"),
                          "bertscore_f1": x.get("bertscore_f1"), "words": x.get("words"),
                          "ref_words": x.get("ref_words"), "n_doc_tokens": x.get("n_doc_tokens"),
                          "prediction_start": (x.get("prediction") or "")[:300]})
    if curves:
        write_csv(os.path.join(rep_dir, "length_curves.csv"), curves, list(curves[0].keys()))
    if buckets:
        write_csv(os.path.join(rep_dir, "doc_length_buckets.csv"), buckets, list(buckets[0].keys()))
    if worst:
        write_csv(os.path.join(rep_dir, "worst_dev_docs.csv"), worst, list(worst[0].keys()))
    qual_cols = ["experiment", "technique", "model_id", "doc_repr", "dev_bertscore_f1",
                 "dev_rouge1", "dev_rouge2", "dev_rougeL", "words", "ref_words", "compression",
                 "repeated_trigram_rate", "novel_bigram_rate", "content_support",
                 "number_precision", "citation_precision", "ref_content_recall", "trainable_pct",
                 "total_params", "params_ratio_vs_teacher", "train_minutes", "ms_per_doc_infer",
                 "peak_mem_gb"]
    write_csv(os.path.join(rep_dir, "quality_table.csv"), done, qual_cols)
    log(f"   [report] {len(done)} finished runs: quality_table, length_curves ({len(curves)}), "
        f"doc_length_buckets ({len(buckets)}), worst_dev_docs ({len(worst)})")
    m = common_metric(rows)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ok = [r for r in done if _score(r, m) >= 0]
        if ok:
            fig, ax = plt.subplots(figsize=(max(6, 0.45 * len(ok)), 4))
            ax.bar(range(len(ok)), [_score(r, m) for r in ok], color="#4C72B0")
            ax.set_xticks(range(len(ok)))
            ax.set_xticklabels([f"{r['experiment']}\n{r['model_id']}" for r in ok], fontsize=7)
            ax.set_ylabel(m)
            ax.set_title("Phase 6 summarisation: dev score per experiment and model")
            fig.tight_layout()
            fig.savefig(os.path.join(rep_dir, "dev_score.png"), dpi=130)
            plt.close(fig)
            fig, ax = plt.subplots(figsize=(6, 4))
            for r in ok:
                try:
                    ax.scatter(float(r["words"]), _score(r, m), s=25)
                    ax.annotate(f"{r['experiment']}/{r['model_id']}", (float(r["words"]),
                                                                       _score(r, m)), fontsize=6)
                except (TypeError, ValueError, KeyError):
                    pass
            ax.set_xlabel("mean generated words (dev)")
            ax.set_ylabel(m)
            fig.tight_layout()
            fig.savefig(os.path.join(rep_dir, "score_vs_length.png"), dpi=130)
            plt.close(fig)
    except Exception as e:
        log(f"   [report] plots skipped ({type(e).__name__}: {e})")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-root", default=OUT_ROOT)
    ap.add_argument("--report", action="store_true")
    a = ap.parse_args()
    rows = aggregate(a.out_root)
    print(f"ALL_RESULTS.csv: {len(rows)} rows")
    if a.report:
        ensemble(rows, a.out_root)
        error_analysis(rows, a.out_root)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/phase6/run_experiment.py
"""
run_experiment.py -- the Phase-6 orchestrator (one notebook = one call).

    python run_experiment.py --exp E2 --models M4,M2,M3,M1 --gpu-mode jobs --resume auto \
                             --time-budget-h 2.0

Order of work (sections 3, 4, 8, 9 of the brief):
  1. Phase-1 discovery (by the `arch` field), the job list, results_<exp>.csv with one PENDING
     row per job + INDEX.md                                   <- outputs exist from minute one
  2. data discovery (attached dataset, else the HF download with the Kaggle secret `HF`),
     schema-tolerant loading, data audit, LEAK GUARD, PROVENANCE.json
     + the held-out dev split carved from train (no public dev split in Phase 6)
  3. token cache (reused when an attached cache has the same key)
  4. job queue: one slot per visible GPU (no DDP); each job is a `summ_train.py` subprocess
     with CUDA_VISIBLE_DEVICES=<slot>; per-job time budget from the notebook budget;
     retry ladder on failure; the CSV and INDEX.md are refreshed every minute
  5. BEST/ per experiment, ALL_RESULTS.csv (+ pivots), UPLOAD_PLAN.csv, final INDEX.md
Exit code: 0 if at least one job finished (DONE), else 1 (2 = data/weights missing).
The notebook prints the code and never raises (rule R1).
"""
import argparse
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
from typing import Dict, List, Optional

from cjpe_common import (CACHE_DIR, CODE_DIR, COUNTED, CSV_COLUMNS, DEFAULT_LR,
                         DEFAULT_MODEL_ORDER, EXPERIMENTS, JOB_EXIT, MODELS, OUT_ROOT, SPEC,
                         default_input_roots, dir_size_gb, find_phase1_models, free_gb, log_to,
                         module_md5s, now_str, read_json, student_layers, write_csv, write_json)

LOG = None


def log(msg: str) -> None:
    log_to(LOG, msg)


def build_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--exp", required=True, choices=sorted(EXPERIMENTS))
    ap.add_argument("--models", default=",".join(DEFAULT_MODEL_ORDER))
    ap.add_argument("--gpu-mode", default="jobs", choices=["jobs"])
    ap.add_argument("--resume", default="auto", choices=["auto", "off"])
    ap.add_argument("--time-budget-h", type=float, default=2.0)
    ap.add_argument("--out-root", default=OUT_ROOT)
    ap.add_argument("--cache-dir", default=CACHE_DIR)
    ap.add_argument("--doc-repr", default="auto", choices=["auto", "head", "headtail", "dancer"])
    ap.add_argument("--auto-long", type=float, default=350,
                    help="auto: dancer when the median train summary has more tokens than this")
    ap.add_argument("--chunk-tokens", type=int, default=640)
    ap.add_argument("--max-chunks", type=int, default=16)
    ap.add_argument("--summary-frac", type=float, default=0.5)
    ap.add_argument("--effective-batch", type=int, default=16)
    ap.add_argument("--micro-tokens", type=int, default=8192)
    ap.add_argument("--gen-batch", type=int, default=16)
    ap.add_argument("--dev-gen", type=int, default=64, help="dev documents decoded for scores")
    ap.add_argument("--dev-frac", type=float, default=0.06)
    ap.add_argument("--dev-max", type=int, default=300)
    ap.add_argument("--num-beams", type=int, default=1)
    ap.add_argument("--no-repeat-ngram", type=int, default=4)
    ap.add_argument("--min-new-frac", type=float, default=0.3,
                    help="single-pass layouts: no EOS before this x median summary length")
    ap.add_argument("--no-bertscore", action="store_true")
    ap.add_argument("--n-virtual", type=int, default=20, help="prefix / prompt length")
    ap.add_argument("--epochs-max", type=int, default=None)
    ap.add_argument("--min-epochs", type=int, default=1)
    ap.add_argument("--evals-per-epoch", type=int, default=2)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--teacher-model", default="M4")
    ap.add_argument("--teacher-frac", type=float, default=0.38)
    ap.add_argument("--kd-alpha", type=float, default=0.5)
    ap.add_argument("--kd-T", type=float, default=2.0)
    ap.add_argument("--ablation-model", default="M4")
    ap.add_argument("--ablation-technique", default="lorma")
    ap.add_argument("--e7-jobs", default="qlora,prefix,prompt,repr_headtail,repr_dancer,beam3")
    ap.add_argument("--e7-fixed-frac", type=float, default=0.5)
    ap.add_argument("--e7-fixed-epochs", type=int, default=1)
    ap.add_argument("--skip-job-checks", action="store_true")
    ap.add_argument("--max-retries", type=int, default=2)
    ap.add_argument("--n-proc", type=int, default=max(1, min(4, os.cpu_count() or 1)))
    ap.add_argument("--reserve-min", type=float, default=6.0)
    ap.add_argument("--lr-mult", type=float, default=1.0)
    return ap


# =============================================================================================
# jobs
# =============================================================================================
def make_jobs(a, p1: Dict[str, str]) -> List[Dict]:
    models = [m.strip() for m in a.models.split(",") if m.strip()]
    e = a.exp
    jobs = []
    if e in COUNTED and e != "E6":
        tech = EXPERIMENTS[e]["technique"]
        for m in models:
            jobs.append({"job": MODELS[m]["folder"], "model": m, "technique": tech})
    elif e == "E6":
        t = a.teacher_model
        jobs.append({"job": f"teacher_{MODELS[t]['folder']}", "model": t,
                     "technique": "full_ft", "is_teacher": True})
        for m in models:
            jobs.append({"job": f"student_{MODELS[m]['folder']}", "model": m, "technique": "kd",
                         "student": True, "depends": [f"teacher_{MODELS[t]['folder']}"]})
    elif e == "E7":
        am = a.ablation_model
        fixed = {"epochs": a.e7_fixed_epochs, "frac": a.e7_fixed_frac}
        for spec in [s.strip() for s in a.e7_jobs.split(",") if s.strip()]:
            f = MODELS[am]["folder"]
            if spec in ("qlora", "prefix", "prompt", "lora", "lorma", "adapters", "pfeiffer"):
                jobs.append({"job": f"{spec}_{f}", "model": am, "technique": spec})
            elif spec.startswith("repr_"):
                jobs.append({"job": f"{spec}_{f}", "model": am, "technique": a.ablation_technique,
                             "doc_repr": spec[5:], "fixed_plan": fixed})
            elif spec.startswith("beam"):
                jobs.append({"job": f"{spec}_{f}", "model": am, "technique": a.ablation_technique,
                             "num_beams": int(spec[4:] or 3), "fixed_plan": fixed})
    for j in jobs:
        j.setdefault("depends", [])
        j["state"] = "pending"
        j["attempt"] = 0
        j["overrides"] = {}
    return jobs


def planned_row(a, j: Dict) -> Dict:
    return {"experiment": a.exp, "technique": j["technique"] + (" (teacher)" if j.get(
        "is_teacher") else ""), "model_id": j["model"], "arch": MODELS[j["model"]]["arch"],
            "job": j["job"], "status": "PENDING", "doc_repr": j.get("doc_repr", a.doc_repr),
            "train_files": "train.json", "seed": a.seed}


# =============================================================================================
# INDEX.md
# =============================================================================================
def write_index(a, out_root, exp_dir, jobs, extra: Optional[List[str]] = None) -> None:
    from aggregate import write_exp_csvs
    rows = write_exp_csvs(exp_dir, [planned_row(a, j) for j in jobs]) if jobs else []
    lines = [f"# Phase 6 (summarisation) -- {a.exp}: {EXPERIMENTS[a.exp]['title']}", "",
             f"Updated {now_str()}. Deadline {SPEC['deadline']}.", "",
             "| job | model | technique | status | BERTScore-F1 | ROUGE-1 | ROUGE-L | repr | epochs "
             "| train frac | minutes | reason |", "|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for r in rows:
        f = lambda k, n=4: (f"{float(r[k]):.{n}f}" if r.get(k) not in (None, "") else "")  # noqa
        lines.append(f"| {r.get('job')} | {r.get('model_id')} | {r.get('technique')} | "
                     f"{r.get('status')} | {f('dev_bertscore_f1')} | {f('dev_rouge1')} | "
                     f"{f('dev_rougeL')} | {r.get('doc_repr') or ''} | {f('epochs_run', 2)} | "
                     f"{f('train_fraction', 2)} | {f('train_minutes', 1)} | "
                     f"{str(r.get('fail_reason') or '').replace('|', '/')[:120]} |")
    todo = [r for r in rows if r.get("status") in ("FAILED", "SKIPPED_TIME", "PENDING")]
    if todo:
        ms = sorted({r["model_id"] for r in todo if r.get("model_id")})
        lines += ["", "## Next command for the unfinished models (here or on another account)",
                  "", "```", f"!cd /kaggle/working/phase6 && python run_experiment.py --exp "
                             f"{a.exp} --models {','.join(ms)} --resume auto --time-budget-h "
                             f"{a.time_budget_h}", "```",
                  "Attach this notebook's output as an input so finished models are skipped."]
    if extra:
        lines += [""] + extra
    p = os.path.join(out_root, "INDEX.md")
    os.makedirs(out_root, exist_ok=True)
    with open(p + ".tmp", "w") as f:
        f.write("\n".join(lines) + "\n")
    os.replace(p + ".tmp", p)


# =============================================================================================
# data
# =============================================================================================
def prepare_data(a, out_root: str, bpe_dir: str) -> Dict:
    from ner_common import file_md5
    from summ_data import build_cache, cache_key, find_cache, load_all, load_cache, summary_stats
    from cjpe_data import provenance
    roots = default_input_roots()
    data, schemas, info = load_all(roots, a.dev_frac, a.dev_max, log)
    if info["leak_guard"]["train_test_overlap_pct"] > 2.0:
        log("   !!! [leak-guard] more than 2% of train overlaps test. Those documents were DROPPED.")
    prov = provenance(schemas, info["leak_guard"], len(data["train"]), info["source"])
    prov["dev_split"] = {"n": len(data["dev"]), "rule": "md5(normalised text) order, first "
                         f"{len(data['dev'])} documents; never trained on"}
    write_json(os.path.join(out_root, "PROVENANCE.json"), prov)
    write_json(os.path.join(out_root, "test_ids.json"), [d["id"] for d in data["test"]])
    key = cache_key({s: schemas[s]["md5"] for s in schemas},
                    file_md5(os.path.join(bpe_dir, "bpe_model.json")), len(data["dev"]))
    path = find_cache(key, [a.cache_dir] + roots)
    if path:
        log(f"   [cache] reusing {path}")
    else:
        t0 = time.time()
        log(f"   [cache] tokenising every document ({a.n_proc} processes) ...")
        path = build_cache(data, bpe_dir, a.cache_dir, key, a.n_proc, log)
        log(f"   [cache] built in {time.time() - t0:.0f}s")
    stats = summary_stats(load_cache(path))
    write_json(os.path.join(out_root, "data_audit.json"), {"schemas": schemas, **stats,
                                                           "leak_guard": info["leak_guard"]})
    st, dt = stats["summary_tokens"], stats["doc_tokens"]
    log(f"   [audit] document tokens median {dt.get('median', 0):.0f} p90 {dt.get('p90', 0):.0f}; "
        f"summary tokens median {st.get('median', 0):.0f} p90 {st.get('p90', 0):.0f}; summary words "
        f"median {stats['median_summary_words']:.0f}")
    repr_ = a.doc_repr if a.doc_repr != "auto" else (
        "dancer" if st.get("median", 0) > a.auto_long else "headtail")
    log(f"   [repr] doc_repr={a.doc_repr} -> {repr_}")
    return {"cache_path": path, "cache_key": key, "n": {s: len(v) for s, v in data.items()},
            "data_md5": {s: schemas[s]["md5"] for s in schemas}}


# =============================================================================================
# scheduler
# =============================================================================================
def n_gpus() -> int:
    try:
        import torch
        return torch.cuda.device_count()
    except Exception:
        return 0


def job_config(a, j: Dict, p1: Dict[str, str], data: Dict, exp_dir: str, slot: int,
               budget_s: float, attn: str, bpe_dir: str, jobs_by_name: Dict) -> Dict:
    tech = j["technique"]
    lr = DEFAULT_LR.get(tech, 5e-4) * a.lr_mult * j["overrides"].get("lr_mult", 1.0)
    cfg = {
        "experiment": a.exp, "exp_folder": EXPERIMENTS[a.exp]["folder"], "job": j["job"],
        "model": j["model"], "model_folder": MODELS[j["model"]]["folder"],
        "arch": MODELS[j["model"]]["arch"], "phase1_dir": p1[j["model"]], "bpe_dir": bpe_dir,
        "technique": tech, "doc_repr": j.get("doc_repr", a.doc_repr), "auto_long": a.auto_long,
        "chunk_tokens": a.chunk_tokens, "max_chunks": a.max_chunks,
        "summary_frac": a.summary_frac, "lr": lr, "head_lr": lr, "weight_decay": 0.01,
        "grad_clip": 1.0, "warmup_frac": 0.06, "epochs_max": a.epochs_max,
        "min_epochs": a.min_epochs, "evals_per_epoch": a.evals_per_epoch,
        "patience": a.patience, "effective_batch": a.effective_batch,
        "micro_tokens": int(j["overrides"].get("micro_tokens", a.micro_tokens)),
        "gen_batch": int(j["overrides"].get("gen_batch", a.gen_batch)), "dev_gen": a.dev_gen,
        "num_beams": int(j.get("num_beams", a.num_beams)), "no_repeat_ngram": a.no_repeat_ngram,
        "min_new_frac": a.min_new_frac,
        "bertscore": not a.no_bertscore, "n_virtual": a.n_virtual, "seed": a.seed,
        "budget_s": budget_s, "cache_path": data["cache_path"], "cache_key": data["cache_key"],
        "data_md5": data["data_md5"], "n_train": data["n"]["train"], "n_dev": data["n"]["dev"],
        "n_test": data["n"]["test"], "skip_job_checks": bool(
            a.skip_job_checks or j["overrides"].get("skip_job_checks")),
        "is_teacher": bool(j.get("is_teacher")), "fixed_plan": j.get("fixed_plan"),
        "gpu_slot": slot, "attempt": j["attempt"], "started_at": now_str(),
        "train_files": "train.json", "leak_guard": "passed", "ckpt_every_min": 30,
        "min_train_fraction": 0.1, "kd_alpha": a.kd_alpha, "kd_T": a.kd_T,
        "lorma_r": 16, "lorma_alpha": 32, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
        "adalora_init_r": 16, "adalora_target_r": 8, "adalora_alpha": 32,
        "adapter_bottleneck": 64, "partial_top_k": 6, "layer_decay": 0.85,
        "module_md5": module_md5s(),
    }
    if j.get("student"):
        cfg["layer_ids"] = student_layers(j["model"], p1[j["model"]])
        tdir = os.path.join(exp_dir, j["depends"][0])
        cfg["teacher_dir"] = tdir
        if not (os.path.exists(os.path.join(tdir, "DONE.json")) and
                os.path.exists(os.path.join(tdir, "checkpoints", "best.pt"))):
            # teacher failed: train the pruned student on the gold summaries only (CSV says so)
            cfg["technique"] = "student_hard"
            cfg["teacher_failed"] = True
    return cfg


def copy_from_inputs(a, exp_dir: str, job: str) -> Optional[str]:
    """--resume auto: a finished (or resumable) copy of this job in an attached input."""
    folder = EXPERIMENTS[a.exp]["folder"]
    from cjpe_common import iter_out_dirs
    for od in iter_out_dirs(None, default_input_roots()):
        src = os.path.join(od, folder, job)
        if os.path.exists(os.path.join(src, "DONE.json")) or os.path.exists(
                os.path.join(src, "checkpoints", "last.pt")):
            dst = os.path.join(exp_dir, job)
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            return src
    return None


def archive_attempt(run_dir: str, attempt: int) -> None:
    lg = os.path.join(run_dir, "logs")
    for f in ("console.log", "job_checks.json", "plan.json", "train_log.jsonl",
              "progress.json"):
        p = os.path.join(lg, f)
        if os.path.exists(p):
            os.replace(p, os.path.join(lg, f"attempt{attempt}_{f}"))
    for f in ("checkpoints/last.pt", "checkpoints/best.pt", "metrics.json", "DONE.json"):
        p = os.path.join(run_dir, f)
        if os.path.exists(p):
            os.remove(p)


def next_attempt(a, j: Dict, code: int, reducible: bool) -> Optional[str]:
    """The retry ladder (rule R5). Returns a description, or None when giving up."""
    if code == 2:
        return None
    ladder = []
    if code in (4, 5):
        ladder.append(("lr_mult", 0.3, "learning rate x0.3"))
    if code == 7 or code == 1:
        ladder.append(("micro_tokens", max(1024, a.micro_tokens // 2), "micro-batch halved"))
        ladder.append(("gen_batch", max(1, a.gen_batch // 4), "generation batch / 4"))
    if code == 6:
        ladder.append(("skip_job_checks", True, "self-checks skipped"))
    used = j.setdefault("ladder_used", [])
    for k, v, desc in ladder:
        if desc not in used:
            used.append(desc)
            j["overrides"][k] = v
            return desc
    return None


def schedule(a, jobs: List[Dict], p1, data, exp_dir, attn, bpe_dir, t_end, out_root) -> None:
    slots = max(1, n_gpus())
    log(f"   [queue] {len(jobs)} jobs on {slots} slot(s) "
        f"({'GPU' if n_gpus() else 'CPU'}); notebook budget ends at "
        f"{time.strftime('%H:%M', time.localtime(t_end))}")
    by = {j["job"]: j for j in jobs}
    running: Dict[int, Dict] = {}
    last_refresh = 0.0
    reserve = 60 * a.reserve_min
    while True:
        # ---- launch ---------------------------------------------------------------------
        for slot in range(slots):
            while slot not in running:
                ready = [j for j in jobs if j["state"] == "pending" and all(
                    by[d]["state"] in ("done", "failed", "skipped") for d in j["depends"]
                    if d in by)]
                if not ready:
                    break
                j = ready[0]
                remaining = t_end - time.time() - reserve
                n_pending = sum(1 for x in jobs if x["state"] == "pending")
                if j.get("is_teacher") and any(x.get("student") for x in jobs):
                    budget = a.teacher_frac * remaining
                else:
                    budget = remaining * slots / max(slots, n_pending + len(running))
                    budget = min(budget, remaining)
                run_dir = os.path.join(exp_dir, j["job"])
                if budget < 8 * 60:
                    j["state"] = "skipped"
                    os.makedirs(os.path.join(run_dir, "logs"), exist_ok=True)
                    cfg = job_config(a, j, p1, data, exp_dir, slot, budget, attn, bpe_dir, by)
                    write_json(os.path.join(run_dir, "logs", "run_config.json"), cfg)
                    write_json(os.path.join(run_dir, "status.json"),
                               {"state": "skipped_time", "time": now_str(), "exit_code": 2,
                                "reason": f"only {budget / 60:.0f} min left in the notebook budget"})
                    log(f"   [queue] SKIP {j['job']}: only {budget / 60:.0f} min left")
                    continue
                j["attempt"] += 1
                cfg = job_config(a, j, p1, data, exp_dir, slot, budget, attn, bpe_dir, by)
                os.makedirs(os.path.join(run_dir, "logs"), exist_ok=True)
                write_json(os.path.join(run_dir, "logs", "run_config.json"), cfg)
                st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
                write_json(os.path.join(run_dir, "status.json"),
                           {**st, "state": "preflight", "time": now_str(), "reason": "",
                            "attempt": j["attempt"]})
                mode = "train"
                env = dict(os.environ, PYTHONPATH=CODE_DIR, PYTHONUNBUFFERED="1")
                if n_gpus():
                    env["CUDA_VISIBLE_DEVICES"] = str(slot)
                cmd = [sys.executable, os.path.join(CODE_DIR, "summ_train.py"), "--run-dir", run_dir,
                       "--mode", mode]
                log(f"   [queue] START {j['job']} ({cfg['technique']}, attempt {j['attempt']}) on "
                    f"slot {slot}, budget {budget / 60:.0f} min")
                j["proc"] = subprocess.Popen(cmd, env=env, cwd=CODE_DIR)
                j["state"] = "running"
                j["slot"] = slot
                running[slot] = j
        if not running:
            break
        # ---- poll -----------------------------------------------------------------------
        time.sleep(5)
        for slot, j in list(running.items()):
            code = j["proc"].poll()
            if code is None:
                continue
            del running[slot]
            run_dir = os.path.join(exp_dir, j["job"])
            st = read_json(os.path.join(run_dir, "status.json"), {}) or {}
            if code == 0 and st.get("state") in ("done", "degenerate", "smoke_ok"):
                j["state"] = "done"
                log(f"   [queue] END {j['job']}: {st.get('state')} ({st.get('reason', '')})")
                continue
            reason = f"exit {code} ({JOB_EXIT.get(code, '?')}): {st.get('reason', '')}"
            log(f"   [queue] FAIL {j['job']}: {reason[:300]}")
            reducible = False
            remaining = t_end - time.time() - reserve
            desc = None
            if j["attempt"] <= a.max_retries and remaining > 15 * 60:
                desc = next_attempt(a, j, code, reducible)
            if desc:
                archive_attempt(run_dir, j["attempt"])
                write_json(os.path.join(run_dir, "status.json"),
                           {"state": "pending", "time": now_str(),
                            "reason": f"retry after {reason[:300]} -> {desc}"})
                j["state"] = "pending"
                log(f"   [queue] RETRY {j['job']} with {desc}")
            else:
                j["state"] = "failed"
                write_json(os.path.join(run_dir, "status.json"),
                           {**st, "state": "skipped_time" if code == 2 else "failed",
                            "exit_code": code, "time": now_str(), "reason": reason[:2000]})
        if time.time() - last_refresh > 60:
            try:
                write_index(a, out_root, exp_dir, jobs)
            except Exception as e:
                log(f"   [index] refresh failed: {e}")
            last_refresh = time.time()


# =============================================================================================
# main
# =============================================================================================
def finalize(a, out_root, exp_dir, jobs, extra=None) -> int:
    from aggregate import aggregate, error_analysis, ensemble
    from select_best import plan_uploads, select_exp
    rows = []
    try:
        from aggregate import write_exp_csvs
        rows = write_exp_csvs(exp_dir, [planned_row(a, j) for j in jobs])
        if a.exp in COUNTED or a.exp == "E7":
            select_exp(exp_dir, rows, log)
    except Exception as e:
        log(f"   [final] BEST selection failed: {e}\n{traceback.format_exc()[-1500:]}")
    try:
        allrows = aggregate(out_root, default_input_roots(), log)
        log(f"   [final] ALL_RESULTS.csv: {len(allrows)} rows")
        plan = plan_uploads(allrows, os.path.join(out_root, "UPLOAD_PLAN.csv"), log)
        if a.exp == "E8":
            ensemble(allrows, out_root, log)
            error_analysis(allrows, out_root, None, log)
    except Exception as e:
        log(f"   [final] aggregation failed: {e}\n{traceback.format_exc()[-1500:]}")
    write_index(a, out_root, exp_dir, jobs, extra)
    done = [r for r in rows if r.get("status") == "DONE"]
    write_json(os.path.join(exp_dir, "summary.json"),
               {"experiment": a.exp, "finished": now_str(), "rows": rows,
                "output_gb": dir_size_gb(out_root)})
    log(f"   [final] {len(done)}/{len(rows)} jobs DONE; output {dir_size_gb(out_root):.2f} GB")
    return 0 if (done or a.exp == "E8") else 1


def main() -> int:
    global LOG
    a = build_parser().parse_args()
    t_start = time.time()
    t_end = t_start + 3600 * a.time_budget_h
    out_root = a.out_root
    exp_dir = os.path.join(out_root, EXPERIMENTS[a.exp]["folder"])
    os.makedirs(exp_dir, exist_ok=True)
    LOG = os.path.join(exp_dir, "orchestrator.log")
    log(f"=== Phase 6 {a.exp}: {EXPERIMENTS[a.exp]['title']} | {now_str()} | budget "
        f"{a.time_budget_h} h | {n_gpus()} GPU(s) | free disk {free_gb(out_root):.0f} GB ===")
    if a.exp == "E8":
        jobs: List[Dict] = []
        write_index(a, out_root, exp_dir, jobs)
        return finalize(a, out_root, exp_dir, jobs)

    # ---- 1. Phase-1 models + job list + PENDING rows ----------------------------------------
    p1 = find_phase1_models(default_input_roots())
    for m in MODELS:
        log(f"   [phase1] {m} {MODELS[m]['arch']:<16} -> {p1.get(m, 'NOT FOUND')}")
    jobs = make_jobs(a, p1)
    need = sorted({j["model"] for j in jobs})
    missing = [m for m in need if m not in p1]
    write_index(a, out_root, exp_dir, jobs)
    if missing:
        msg = (f"Phase-1 weights missing for {missing} ({[MODELS[m]['arch'] for m in missing]}). "
               "Attach the four Phase-1 datasets (final/model.pt + config.json + bpe_model.json).")
        log("!" * 90 + f"\n   {msg}\n" + "!" * 90)
        jobs = [j for j in jobs if j["model"] not in missing]
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg, "time": now_str()})
        if not jobs:
            write_index(a, out_root, exp_dir, [], [f"**BLOCKED:** {msg}"])
            return 2
    # bpe_model.json: next to a Phase-1 model.pt, or anywhere in any attached dataset
    # (e.g. a separate "bpe_model" dataset).
    from ner_common import find_files
    bpe_dir = None
    for d in list(p1.values()):
        if os.path.exists(os.path.join(d, "bpe_model.json")):
            bpe_dir = d
            break
    if bpe_dir is None:
        hits = find_files("bpe_model.json", default_input_roots(), 50)
        hits = [h for h in hits if os.path.isfile(h)]
        if hits:
            bpe_dir = os.path.dirname(hits[0])
            log(f"   [bpe] using {hits[0]} ({len(hits)} found)")
    if bpe_dir is None:
        msg = ("bpe_model.json not found in any attached dataset (searched "
               f"{default_input_roots()})")
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg})
        write_index(a, out_root, exp_dir, jobs, [f"**BLOCKED:** {msg}"])
        return 2

    attn = "causal"
    try:
        data = prepare_data(a, out_root, bpe_dir)
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        log("!" * 90 + f"\n   DATA STEP FAILED: {msg}\n" + "!" * 90)
        log(traceback.format_exc()[-2000:])
        write_json(os.path.join(out_root, "BLOCKED.json"), {"reason": msg, "time": now_str()})
        for j in jobs:
            rd = os.path.join(exp_dir, j["job"])
            os.makedirs(os.path.join(rd, "logs"), exist_ok=True)
            write_json(os.path.join(rd, "status.json"), {"state": "failed", "reason": msg,
                                                         "time": now_str()})
        write_index(a, out_root, exp_dir, jobs, [f"**BLOCKED (data):** {msg}"])
        return 2

    # ---- 4. resume / queue ---------------------------------------------------------------
    for j in jobs:
        rd = os.path.join(exp_dir, j["job"])
        if a.resume == "auto":
            if os.path.exists(os.path.join(rd, "DONE.json")):
                j["state"] = "done"
                log(f"   [resume] {j['job']} already DONE -> skipped")
                continue
            src = None if os.path.exists(os.path.join(rd, "checkpoints", "last.pt")) else \
                copy_from_inputs(a, exp_dir, j["job"])
            if src:
                if os.path.exists(os.path.join(rd, "DONE.json")):
                    j["state"] = "done"
                    log(f"   [resume] {j['job']} finished in an earlier session ({src}) -> copied")
                else:
                    log(f"   [resume] {j['job']} resumes from {src}")
        os.makedirs(os.path.join(rd, "logs"), exist_ok=True)
        if j["state"] == "pending" and not os.path.exists(os.path.join(rd, "status.json")):
            write_json(os.path.join(rd, "status.json"), {"state": "pending", "time": now_str()})
    write_index(a, out_root, exp_dir, jobs)
    try:
        schedule(a, jobs, p1, data, exp_dir, attn, bpe_dir, t_end, out_root)
    except Exception as e:
        log(f"   [queue] scheduler error: {e}\n{traceback.format_exc()[-2000:]}")
        for j in jobs:
            if j.get("proc") is not None and j["proc"].poll() is None:
                j["proc"].terminate()
    return finalize(a, out_root, exp_dir, jobs)


if __name__ == "__main__":
    try:
        code = main()
    except SystemExit:
        raise
    except Exception as e:
        print(f"run_experiment.py crashed: {e}\n{traceback.format_exc()}", flush=True)
        code = 1
    sys.exit(code)


In [ ]:
%%writefile /kaggle/working/phase6/selftest.py
"""
selftest.py -- Phase-6 CPU tests with tiny random-initialised models of the four Phase-1
architectures and a tiny BPE trained on synthetic legal-ish text (no real weights or data).

    python selftest.py [--out SELFTEST.json]

  1 metrics     ROUGE-1/2/L vs hand values, bit-parallel LCS == DP LCS, token-exact sentence
                spans, word caps, submission validator (good + 5 kinds of bad zips)
  2 data        loader (list / dict-keyed / wrapped), leak guard drops a planted test copy,
                deterministic held-out dev split disjoint from train, token cache
  3 layout      headtail + dancer: every sequence <= ctx, EOS start, separator, targets = the
                aligned summary sentences (all sentences used exactly once), labels only on targets
  4 model       for all 4 architectures x 9 techniques: trainable set, identity at init
                (LoRMA / LoRA / AdaLoRA exact), SelectHead logits == full logits at the targets,
                gradients reach every trainable tensor, KV-cache greedy == no-cache greedy (incl.
                prefix / prompt virtual tokens), left-padded batch == one-by-one decoding,
                beam search runs, NF4 dequantisation error small
  5 overfit     8 sequences: summary-token loss falls below 25% of its start (subprocess)
  6 resume      train -> kill (137) at step 3 -> resume at step 3 -> DONE + valid submission
"""
import argparse
import copy
import json
import os
import random
import subprocess
import sys
import tempfile
import time
import traceback
from typing import Dict

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)

from cjpe_common import MODELS, write_json  # noqa: E402

TINY = {
    "M1": ("GPT2StyleLM", {"vocab_size": 600, "n_positions": 1024, "n_embd": 32, "n_layer": 4,
                           "n_head": 4, "mlp_ratio": 4}),
    "M2": ("LlamaStyleLM", {"vocab_size": 600, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": None}),
    "M3": ("LlamaStyleLM", {"vocab_size": 600, "max_position_embeddings": 1024,
                            "hidden_size": 32, "intermediate_size": 64, "num_hidden_layers": 4,
                            "num_attention_heads": 4, "num_key_value_heads": 2,
                            "sliding_window": 8}),
    "M4": ("BloomStyleLM", {"vocab_size": 600, "hidden_size": 32, "n_layer": 4, "n_head": 4,
                            "mlp_ratio": 4}),
}
WORDS = ("the court held that appellant respondent section act order petition dismissed "
         "allowed appeal judgment high supreme evidence witness learned counsel state "
         "accused conviction sentence bail facts issue reasons therefore accordingly").split()


# =============================================================================================
# fixtures
# =============================================================================================
def synth_text(rng: random.Random, n_words: int, label) -> str:
    """label: list of statute ids (Phase 5); an int is treated as [int]."""
    labs = label if isinstance(label, (list, tuple)) else [int(label)]
    words = []
    for k in range(n_words):
        w = rng.choice(WORDS)
        if rng.random() < 0.08:
            w = w.upper()
        words.append(w)
        if rng.random() < 0.1:
            words.append(rng.choice([",", ".", ";", "(1)", "12/2003", "\n", "  "]))
    cue = {1: "murder penal", 2: "cheating fraud", 3: "dowry cruelty", 4: "bail custody",
           5: "property land", 6: "contract breach"}
    tail = "".join(f" the accused was charged under section {x} for {cue.get(x, 'offence')} ."
                   for x in labs)
    return " ".join(words) + "\n\nORDER\n" + tail * 3


def write_tokenizer(d: str, seed: int = 0) -> str:
    from bpe_tokenizer import BPETokenizer, count_pretokens, train_bpe
    rng = random.Random(seed)
    texts = [synth_text(rng, 200, [i % 6 + 1]) for i in range(40)]
    merges, _ = train_bpe(count_pretokens(texts, log=lambda *a: None), 3 + 256 + 300,
                          log=lambda *a: None)
    tok = BPETokenizer(merges)
    tok.save_pretrained(d)
    return d


def write_phase1(root: str, mid: str, seed: int = 0, tok_dir: str = None) -> str:
    from phase1_models import MODEL_CLASSES
    cls_name, conf = TINY[mid]
    d = os.path.join(root, f"p1_{mid}", "final")
    os.makedirs(d, exist_ok=True)
    torch.manual_seed(seed)
    cls, ccls = MODEL_CLASSES[cls_name]
    torch.save(cls(ccls(**conf)).state_dict(), os.path.join(d, "model.pt"))
    with open(os.path.join(d, "config.json"), "w") as f:
        json.dump({"model_class": cls_name, "arch": MODELS[mid]["arch"], "config": conf}, f)
    if tok_dir:
        import shutil
        for fn in os.listdir(tok_dir):
            shutil.copy2(os.path.join(tok_dir, fn), os.path.join(d, fn))
    return d


def write_dataset(root: str, n_train=40, n_test=8, seed=0, shape="list", leak: int = 0,
                  long_summ=False) -> str:
    """Phase_6_Summ/{Train/train.json, Test/test.json}; `leak` test documents planted in train."""
    rng = random.Random(seed)
    base = os.path.join(root, "Phase_6_Summ")
    os.makedirs(os.path.join(base, "Train"), exist_ok=True)
    os.makedirs(os.path.join(base, "Test"), exist_ok=True)
    splits = {}
    for s, n in (("train", n_train), ("test", n_test)):
        recs = []
        for i in range(n):
            lab = sorted(rng.sample(range(1, 7), rng.choice([1, 2])))
            doc = synth_text(rng, rng.choice([60, 200, 600]), lab)
            k = 6 if long_summ else 2
            summ = " ".join(f"The court held that section {x} applies and the appeal was "
                            f"{'allowed' if x % 2 else 'dismissed'}." for x in lab * k)
            r = {"id": f"{s}_{i:04d}", "document": doc}
            if s == "train":
                r["summary"] = summ
            recs.append(r)
        splits[s] = recs
    for k in range(leak):
        t = dict(splits["test"][k])
        t["id"] = f"train_leak_{k}"
        t["summary"] = "The court held that the appeal was dismissed."
        splits["train"].append(t)
    for s, rel in (("train", "Train/train.json"), ("test", "Test/test.json")):
        obj = splits[s]
        if shape == "dict":
            obj = {r["id"]: {k: v for k, v in r.items() if k != "id"} for r in obj}
        elif shape == "wrapped":
            obj = {"data": obj}
        with open(os.path.join(base, rel), "w") as f:
            json.dump(obj, f)
    return base


def _data(root, tok_dir, name, **kw):
    """write a dataset, load it through the real loader, build the cache -> (cache, path)."""
    from summ_data import build_cache, load_all, load_cache
    base = write_dataset(os.path.join(root, name), **kw)
    os.environ["PHASE6_INPUT_ROOTS"] = os.path.dirname(base)
    data, _sch, info = load_all([os.path.dirname(base)], 0.2, 50, log=lambda *a: None)
    p = build_cache(data, tok_dir, os.path.join(root, name + "_cache"), name, 1,
                    log=lambda *a: None)
    return load_cache(p), p, data, info


# =============================================================================================
def t1_metrics(tok_dir: str) -> Dict:
    from gen_ml import cap_words, lcs_len, rouge_l, rouge_n, sentence_spans
    from bpe_tokenizer import BPETokenizer
    r = rouge_n("the cat sat on the mat", "the cat lay on the mat", 1)
    assert abs(r["f"] - 5 / 6) < 1e-9, r
    r2 = rouge_n("the cat sat on the mat", "the cat lay on the mat", 2)
    assert abs(r2["f"] - 3 / 5) < 1e-9, r2
    assert abs(rouge_l("a b c d e", "a x c y e")["f"] - 3 / 5) < 1e-9
    rng = random.Random(3)

    def dp(a, b):
        M = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
        for i in range(len(a)):
            for j in range(len(b)):
                M[i + 1][j + 1] = M[i][j] + 1 if a[i] == b[j] else max(M[i][j + 1], M[i + 1][j])
        return M[-1][-1]
    for _ in range(200):
        a = [rng.choice("abcde") for _ in range(rng.randint(0, 40))]
        b = [rng.choice("abcde") for _ in range(rng.randint(0, 40))]
        assert lcs_len(a, b) == dp(a, b), (a, b)
    tok = BPETokenizer.from_pretrained(tok_dir)
    for k in range(30):
        t = synth_text(rng, 80, [k % 6 + 1]) + " Held: appeal allowed! Costs? None; order:  done."
        sp = sentence_spans(t)
        assert "".join(sp) == t
        assert sum((tok.encode(x) for x in sp), []) == tok.encode(t), "span tokenisation"
    s = "One two three. Four five six. Seven eight nine."
    assert cap_words(s, 6) == "One two three. Four five six." and cap_words(s, 2) == "One two three."
    # validator
    import zipfile
    from validate_submission import validate
    d = tempfile.mkdtemp()

    def mk(name, obj, entry="prediction.json"):
        p = os.path.join(d, name)
        with zipfile.ZipFile(p, "w") as z:
            z.writestr(entry, json.dumps(obj))
        return p
    good = [{"id": "a", "summary": "x y"}, {"id": "b", "summary": "z"}]
    assert validate(mk("g.zip", good), ["a", "b"])["ok"]
    bad = [mk("b1.zip", good[:1]), mk("b2.zip", good + [good[0]]),
           mk("b3.zip", [{"id": "a", "summary": ""}, good[1]]), mk("b4.zip", good, "x/prediction.json"),
           mk("b5.zip", {"a": "x", "b": "y"})]
    assert not any(validate(p, ["a", "b"])["ok"] for p in bad)
    return {"rouge_lcs_spans_validator": "ok"}


def t2_data(root: str, tok_dir: str) -> Dict:
    from summ_data import load_all, split_dev
    out = {}
    for shape in ("list", "dict", "wrapped"):
        base = write_dataset(os.path.join(root, f"ds_{shape}"), 30, 6, shape=shape, leak=2)
        data, sch, info = load_all([os.path.dirname(base)], 0.2, 50, log=lambda *a: None)
        assert info["leak_guard"]["dropped_from_train"] == 2, info["leak_guard"]
        assert sch["train"]["summary_key"] == "summary" and sch["train"]["text_key"] == "document"
        tr, dv, te = ({d["id"] for d in data[s]} for s in ("train", "dev", "test"))
        assert not (tr & dv) and not (tr & te) and len(dv) >= 1 and len(te) == 6
        out[shape] = {"train": len(tr), "dev": len(dv)}
    a = split_dev(data["train"] + data["dev"], 0.2, 50)
    b = split_dev(list(reversed(data["train"] + data["dev"])), 0.2, 50)
    assert [d["id"] for d in a[1]] and sorted(d["id"] for d in a[1]) == sorted(d["id"] for d in b[1])
    cache, _p, _d, _i = _data(root, tok_dir, "ds_cache", n_train=20, n_test=4)
    r = cache["train"][0]
    assert sum(len(s) for s in r["sents"]) > 0 and len(r["doc"]) == r["n_tokens"]
    return out


def t3_layout(root: str, tok_dir: str) -> Dict:
    from bpe_tokenizer import BPETokenizer
    from summ_data import Layout, summary_stats
    from summ_model import collate_train
    tok = BPETokenizer.from_pretrained(tok_dir)
    cache, _p, _d, _i = _data(root, tok_dir, "ds_layout", n_train=24, n_test=4, long_summ=True)
    st = summary_stats(cache)
    res = {}
    for rep, extra in (("headtail", {}), ("head", {}), ("dancer", {"chunk_tokens": 120}),
                       ("auto", {"auto_long": 10, "chunk_tokens": 120})):
        lay = Layout(tok, {"doc_repr": rep, **extra}, st, 0)
        n = 0
        for rec in cache["train"]:
            ex = lay.examples(rec)
            allt = []
            for p, t in ex:
                assert p[0] == 0 and len(p) + len(t) <= lay.ctx
                allt.append(t[:-1] if len(t) and t[-1] == 0 else t)
                n += 1
            full = np.concatenate([np.asarray(s) for s in rec["sents"]])
            if lay.repr == "dancer" and all(len(t) < lay.S for _, t in ex):
                got = sorted(np.concatenate(allt).tolist())
                assert got == sorted(full.tolist()), "dancer must use every summary token once"
        b = collate_train(ex[:2])
        for i, (p, t) in enumerate(ex[:2]):
            assert int((b["labels"][i] != -100).sum()) == len(t)
        res[rep] = {"repr": lay.repr, "seqs": n}
    assert res["auto"]["repr"] == "dancer" and res["dancer"]["seqs"] > res["headtail"]["seqs"]
    return res


def t4_model(root: str) -> Dict:
    from summ_model import (NF4Linear, apply_summ_technique, build_summ_model, collate_prompts,
                            decode_batch, generate_many)
    import contextlib
    techs = ["lorma", "lora", "adalora", "adapters", "pfeiffer", "qlora", "prefix", "prompt",
             "full_ft"]
    res = {}
    rng = np.random.default_rng(0)
    for mid in ("M1", "M2", "M3", "M4"):
        p1 = write_phase1(root, mid, seed=3)
        for tech in techs:
            torch.manual_seed(0)
            m, _ = build_summ_model(p1)
            base = copy.deepcopy(m).eval()
            cfg = {"lr": 1e-3, "n_virtual": 4}
            groups, hook = apply_summ_technique(m, tech, cfg, log=lambda *a: None,
                                                sample_ids=list(range(5, 40)))
            m.eval()
            ids = torch.from_numpy(rng.integers(3, 500, (2, 30))).long()
            mask = torch.ones_like(ids)
            mask[1, 25:] = 0
            lab = ids.clone()
            lab[:, :12] = -100
            lab[mask == 0] = -100
            with torch.no_grad():
                a, tg = m.lm_logits_at_targets(ids, mask, lab)
                full = m(ids, mask).logits
            sel = torch.zeros_like(lab, dtype=torch.bool)
            sel[:, :-1] = lab[:, 1:] != -100
            assert torch.allclose(a, full[sel], atol=1e-5), "SelectHead"
            if tech in ("lorma", "lora", "adalora"):
                with torch.no_grad():
                    b0, _ = base.lm_logits_at_targets(ids, mask, lab)
                assert float((a - b0).abs().max()) < 1e-4, f"{mid}/{tech} not identity at init"
            if tech == "qlora":
                lin = [x for x in m.modules() if isinstance(x, NF4Linear)][0]
                assert lin.q_packed.dtype == torch.uint8
            m.train()
            lt, tg = m.lm_logits_at_targets(ids, mask, lab)
            torch.nn.functional.cross_entropy(lt, tg).backward()
            tr = [(n, p) for n, p in m.named_parameters() if p.requires_grad]
            assert tr and sum(int(p.grad is not None and p.grad.abs().sum() > 0) for _, p in tr) \
                >= (0.3 if tech == "adalora" else 0.5) * len(tr), f"{mid}/{tech}: gradients"
            m.eval()
            # KV-cache greedy == no-cache greedy
            pr = rng.integers(3, 500, 9).astype(np.int64)
            x, xm = collate_prompts([pr], "cpu")
            nul = contextlib.nullcontext
            with torch.no_grad():
                g = decode_batch(m, x, xm, 6, 6, 1, 0)[0]
                seq = list(pr)
                ref = []
                for _ in range(6):
                    t = torch.tensor([seq])
                    lo = m(t, torch.ones_like(t)).logits[0, -1].clone()
                    lo[[0, 1]] = float("-inf")
                    nx = int(lo.argmax())
                    ref.append(nx)
                    seq.append(nx)
            assert g == ref, f"{mid}/{tech}: cached {g} != uncached {ref}"
            if tech in ("lorma", "prefix", "prompt"):
                p2 = rng.integers(3, 500, 5).astype(np.int64)
                both = generate_many(m, [pr, p2], "cpu", nul, 5, 5, 1, 0, 2)
                one = [generate_many(m, [q], "cpu", nul, 5, 5, 1, 0, 1)[0] for q in (pr, p2)]
                assert both == one, f"{mid}/{tech}: left padding changes the output"
                bm = generate_many(m, [pr, p2], "cpu", nul, 5, 0, 3, 3, 2)
                assert len(bm) == 2
            res[f"{mid}/{tech}"] = len(tr)
    W = torch.randn(64, 128)
    lin = torch.nn.Linear(128, 64)
    lin.weight.data.copy_(W)
    q = NF4Linear(lin)
    rel = float((q.weight - W).norm() / W.norm())
    assert rel < 0.15, rel
    res["nf4_rel_error"] = rel
    return res


def _job_cfg(root, tok_dir, cache_path, tech, run, **kw):
    p1 = write_phase1(root, "M2", tok_dir=tok_dir)
    cfg = {"experiment": "T", "job": os.path.basename(run), "model": "M2", "phase1_dir": p1,
           "bpe_dir": tok_dir, "technique": tech, "doc_repr": "dancer", "chunk_tokens": 120,
           "lr": 3e-3, "head_lr": 3e-3, "effective_batch": 8, "micro_tokens": 4096,
           "gen_batch": 8, "dev_gen": 8, "seed": 1, "budget_s": 900, "cache_path": cache_path,
           "bertscore": False, "skip_job_checks": False, "n_virtual": 4, "epochs_max": 2}
    cfg.update(kw)
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    return cfg


def t5_overfit(root: str, tok_dir: str) -> Dict:
    _c, path, _d, _i = _data(root, tok_dir, "ds_job", n_train=30, n_test=6, long_summ=True)
    run = os.path.join(root, "run_overfit")
    _job_cfg(root, tok_dir, path, "full_ft", run)
    env = dict(os.environ, PYTHONPATH=HERE, CUDA_VISIBLE_DEVICES="")
    p = subprocess.run([sys.executable, os.path.join(HERE, "summ_train.py"), "--run-dir", run,
                        "--mode", "overfit"], env=env, capture_output=True, text=True, timeout=900)
    f = os.path.join(run, "logs", "overfit.json")
    assert os.path.exists(f), (p.stdout + p.stderr)[-2500:]
    ov = json.load(open(f))
    assert ov["passed"], ov
    return ov


def t6_resume(root: str, tok_dir: str) -> Dict:
    from validate_submission import validate
    path = json.load(open(os.path.join(root, "run_overfit", "logs", "run_config.json")))["cache_path"]
    run = os.path.join(root, "run_resume")
    cfg = _job_cfg(root, tok_dir, path, "lorma", run, kill_at_step=3, lr=1e-3)
    env = dict(os.environ, PYTHONPATH=HERE, CUDA_VISIBLE_DEVICES="")
    cmd = [sys.executable, os.path.join(HERE, "summ_train.py"), "--run-dir", run]
    p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
    assert p.returncode == 137, (p.returncode, (p.stdout + p.stderr)[-2000:])
    assert os.path.exists(os.path.join(run, "checkpoints", "last.pt"))
    cfg["kill_at_step"] = -1
    write_json(os.path.join(run, "logs", "run_config.json"), cfg)
    p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=900)
    log = open(os.path.join(run, "logs", "console.log")).read()
    assert p.returncode == 0, log[-2500:]
    assert json.load(open(os.path.join(run, "logs", "resume.json")))["resumed_at_step"] == 3
    assert os.path.exists(os.path.join(run, "DONE.json"))
    tids = [r["id"] for r in __import__("summ_data").load_cache(path)["test"]]
    v = validate(os.path.join(run, "submission", "submission.zip"), tids)
    assert v["ok"], v
    met = json.load(open(os.path.join(run, "metrics.json")))
    assert met["dev_rouge1"] is not None and met["score_metric"] == "rouge1_rougeL_mean"
    for f in ("dev/dev_predictions.jsonl", "dev/length_curve.csv", "dev/dev_report.json",
              "dev/length_buckets.csv"):
        assert os.path.exists(os.path.join(run, f)), f
    return {"resumed_at_step": 3, "dev_rouge1": met["dev_rouge1"], "test_words": v["mean_words"]}


def run_all(log=print) -> Dict:
    res = {"tests": {}, "failed": [], "started": time.strftime("%Y-%m-%d %H:%M:%S")}
    torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
    with tempfile.TemporaryDirectory() as root:
        tok_dir = write_tokenizer(os.path.join(root, "tok"))
        tests = [("1_metrics", lambda: t1_metrics(tok_dir)),
                 ("2_data_leak_split_cache", lambda: t2_data(root, tok_dir)),
                 ("3_layout", lambda: t3_layout(root, tok_dir)),
                 ("4_model_techniques_decoding", lambda: t4_model(root)),
                 ("5_overfit", lambda: t5_overfit(root, tok_dir)),
                 ("6_kill_resume", lambda: t6_resume(root, tok_dir))]
        only = os.environ.get("SELFTEST_ONLY")
        for name, fn in tests:
            if only and not any(name.startswith(o) for o in only.split(",")):
                continue
            t0 = time.time()
            try:
                res["tests"][name] = {"result": fn(), "seconds": round(time.time() - t0, 1)}
                log(f"   PASS {name} ({time.time() - t0:.1f}s)")
            except Exception as e:
                res["failed"].append(name)
                res["tests"][name] = {"error": str(e)[:2000],
                                      "trace": traceback.format_exc()[-3000:]}
                log(f"   FAIL {name}: {str(e)[:500]}\n{traceback.format_exc()[-1500:]}")
    res["ok"] = not res["failed"]
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    res = run_all()
    if a.out:
        write_json(a.out, res)
    print(json.dumps({"ok": res["ok"], "failed": res["failed"]}))
    sys.exit(0 if res["ok"] else 1)


if __name__ == "__main__":
    main()


In [ ]:
# every module must be byte-identical to the generated version (same code in all notebooks)
try:
    bad = [rel for rel, h in EXPECTED_MD5.items()
           if hashlib.md5(open(os.path.join(CODE, rel), 'rb').read()).hexdigest() != h]
    if bad:
        print('!' * 80 + f'\n!!! modules differ from the generated version: {bad}\n' + '!' * 80)
    else:
        print(f'all {len(EXPECTED_MD5)} modules verified')
except Exception as e:
    print('module check failed:', e)


In [ ]:
# what is attached (never raises)
try:
    sys.path.insert(0, CODE)
    from cjpe_common import default_input_roots, find_files, find_phase1_models
    from cjpe_data import get_hf_token
    print('Phase-1 weights:', json.dumps(find_phase1_models(default_input_roots()), indent=1))
    for n in ('train.json', 'test.json'):
        print(n, '->', find_files(n, default_input_roots())[:4])
    t = get_hf_token()
    print('HF secret found:', 'yes (length %d)' % len(t) if t else 'no')
    print('earlier phase6_out inputs:', glob.glob('/kaggle/input/**/phase6_out', recursive=True)[:6])
except Exception as e:
    print('discovery failed:', e); traceback.print_exc()


In [ ]:
# the run (streams its log; a non-zero exit code is PRINTED, never raised -- rule R1)
# to split the four models over two accounts: add e.g. '--models M4,M2' here and
# '--models M3,M1' on the other account (then attach both outputs to E8).
ARGS = "--exp E2 --models M4,M2,M3,M1 --gpu-mode jobs --resume auto --time-budget-h 2.0"
RC = None
try:
    p = subprocess.Popen([sys.executable, '-u', os.path.join(CODE, 'run_experiment.py')] + ARGS.split(),
                         cwd=CODE, env=dict(os.environ, PYTHONPATH=CODE, PYTHONUNBUFFERED='1'),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    RC = p.wait()
except Exception as e:
    print('launch failed:', e); traceback.print_exc()
print(f'\n===== run_experiment.py exit code: {RC} '
      '(0 ok, 1 nothing finished, 2 data / weights missing) =====')


In [ ]:
# R7: final verification + aggregation (never raises)
FOLDER = "E2_lora"
try:
    sys.path.insert(0, CODE)
    from cjpe_common import read_csv
    from validate_submission import validate
    test_ids = json.load(open(f'{OUT}/test_ids.json')) if os.path.exists(f'{OUT}/test_ids.json') else None
    rows = read_csv(f'{OUT}/{FOLDER}/results_{FOLDER}.csv')
    print(f"{'job':<34}{'status':<13}{'BERTSc':>8}{'R-1':>7}{'R-L':>7}{'frac':>6}{'min':>7}  check")
    n_ok = 0
    for r in rows:
        z = f"{OUT}/{FOLDER}/{r['job']}/submission/submission.zip"
        verdict = ''
        if r['status'] == 'DONE':
            v = validate(z, test_ids) if (test_ids and os.path.exists(z)) else {'ok': False, 'errors': ['zip missing']}
            files_ok = all(os.path.exists(f"{OUT}/{FOLDER}/{r['job']}/{f}") and
                           sum(1 for _ in open(f"{OUT}/{FOLDER}/{r['job']}/{f}")) > 1
                           for f in ('dev/dev_predictions.jsonl', 'dev/length_curve.csv', 'dev/dev_report.json'))
            verdict = 'PASS' if (v['ok'] and files_ok and r.get('dev_score')) else f"FAIL {v.get('errors')}"
            n_ok += verdict == 'PASS'
        g = lambda k: f'{float(r[k]):.4f}' if r.get(k) else ''
        print(f"{r['job']:<34}{r['status']:<13}{g('dev_bertscore_f1'):>8}{g('dev_rouge1')[:6]:>7}{g('dev_rougeL')[:6]:>7}"
              f"{(r.get('train_fraction') or '')[:4]:>6}{(r.get('train_minutes') or '')[:5]:>7}  {verdict} {r.get('fail_reason', '')[:80]}")
    best = f'{OUT}/{FOLDER}/BEST/submission.zip'
    if os.path.exists(best):
        print('\nBEST:', json.dumps({k: v for k, v in json.load(open(f'{OUT}/{FOLDER}/BEST/SELECTION.json')).items() if k != 'ranking'}))
        print('UPLOAD THIS:', best)
    elif FOLDER != 'E8_report':
        print('!' * 80 + '\n!!! NO MODEL FINISHED in this notebook. Read the fail_reason column above and\n'
              f'!!! {OUT}/INDEX.md / BLOCKED.json; the logs are in {OUT}/{FOLDER}/<job>/logs/console.log\n' + '!' * 80)
    for f in ('INDEX.md', 'UPLOAD_PLAN.csv', 'BLOCKED.json'):
        if os.path.exists(f'{OUT}/{f}'):
            print(f'\n----- {f} -----'); print(open(f'{OUT}/{f}').read()[:6000])
    tot = 0
    for dp, dn, fn in sorted(os.walk(OUT)):
        depth = dp.count(os.sep) - OUT.count(os.sep)
        size = sum(os.path.getsize(os.path.join(dp, f)) for f in fn)
        tot += size
        if depth <= 2:
            print('  ' * depth + os.path.basename(dp) + f'/  ({size / 1e6:.1f} MB in files here)')
    print(f'total output {tot / 1e9:.2f} GB')
except Exception as e:
    print('final check failed:', e); traceback.print_exc()
print('\nREMINDER: Save Version -> Save & Run All (Commit) keeps /kaggle/working/phase6_out as this '
      'notebook\'s output. Download BEST/submission.zip and results_*.csv from the Output tab.')


Upload `phase6_out/E2_lora/BEST/submission.zip` to Codabench (or any `<job>/submission/submission.zip`). After E1-E6 are done, run `E8_report` with all outputs attached: it writes `UPLOAD_PLAN.csv` (six distinct uploads, all four Phase-1 models).